# MPG-FER Cumulative Ablation - Configuration A0

Automated standalone execution for A0.
Evaluates Train/PublicTest during training and selects best checkpoint via EMA TTA Accuracy. Evaluates PrivateTest exactly once in canonical FP32 after checkpoint freeze.


In [ ]:
import os
from pathlib import Path

CUMULATIVE_MODE = 'A0'
RUN_ID = 'mpg-fer-cumabl7-opus-a0'
OUTPUT_DIR = Path("/kaggle/working/mpg_fer_cumulative_ablation7") / RUN_ID
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
print({"CUMULATIVE_MODE": CUMULATIVE_MODE, "RUN_ID": RUN_ID, "OUTPUT_DIR": str(OUTPUT_DIR)})


In [ ]:
import base64
import hashlib
import sys

EMBEDDED_SOURCES = {'mpg_fer_cumulative_ablation7/__init__.py': 'IiIiQ3VtdWxhdGl2ZSBhYmxhdGlvbiBsYWRkZXIgKDcgY29uZmlndXJhdGlvbnMpIGZvciBNUEctRkVSLiIiIgoKZnJvbSAubW9kZWwgaW1wb3J0ICgKICAgIENVTVVMQVRJVkVfQUJMQVRJT05fT1JERVIsCiAgICBDdW11bGF0aXZlQWJsYXRpb25Nb2RlLAogICAgQ3VtdWxhdGl2ZUFibGF0aW9uTVBHRkVSLAogICAgQ3VtdWxhdGl2ZUFibGF0aW9uU3BlYywKICAgIGN1bXVsYXRpdmVfcmVnaXN0cnlfZG9jdW1lbnQsCikKCl9fYWxsX18gPSBbCiAgICAiQ1VNVUxBVElWRV9BQkxBVElPTl9PUkRFUiIsCiAgICAiQ3VtdWxhdGl2ZUFibGF0aW9uTW9kZSIsCiAgICAiQ3VtdWxhdGl2ZUFibGF0aW9uTVBHRkVSIiwKICAgICJDdW11bGF0aXZlQWJsYXRpb25TcGVjIiwKICAgICJjdW11bGF0aXZlX3JlZ2lzdHJ5X2RvY3VtZW50IiwKXQo=', 'mpg_fer_cumulative_ablation7/model.py': '...', 'mpg_fer_cumulative_ablation7/protocol.py': 'IiIiUHJlcmVnaXN0ZXJlZCBjdW11bGF0aXZlIGFibGF0aW9uIHByb3RvY29sLCBkYXRhc2V0IGNoZWNrcywgYW5kIHBhaXJ3aXNlIGRpZmYgdmFsaWRhdGlvbi4iIiIKCmZyb20gX19mdXR1cmVfXyBpbXBvcnQgYW5ub3RhdGlvbnMKCmltcG9ydCBjc3YKZnJvbSBkYXRhY2xhc3NlcyBpbXBvcnQgYXNkaWN0LCBkYXRhY2xhc3MKaW1wb3J0IGhhc2hsaWIKaW1wb3J0IGpzb24KZnJvbSBwYXRobGliIGltcG9ydCBQYXRoCmZyb20gdHlwaW5nIGltcG9ydCBBbnkKCmZyb20gLm1vZGVsIGltcG9ydCBDVU1VTEFUSVZFX0FCTEFUSU9OX09SREVSLCBDVU1VTEFUSVZFX1JFR0lTVFJZLCBDdW11bGF0aXZlQWJsYXRpb25Nb2RlLCBDdW11bGF0aXZlQWJsYXRpb25TcGVjCgpDQU5PTklDQUxfREFUQVNFVF9IQVNIRVM6IGRpY3Rbc3RyLCBzdHJdID0gewogICAgInRyYWluLmNzdiI6ICJkZWI4MmM0YjRlMDFiOTA3NzZhNzE4YzM0OTM0NjY2YjBiZGRlNjY5NmNhMWQwMTQ5ZjhmZTgwN2E4ZmY0YmE4IiwKICAgICJ2YWwuY3N2IjogIjQxMjAzNmQwNzdjNmVjMjAzMDQ3YjI5MzVhYjE0YmM4NThkODEzNmVlMjZlOGRiM2UyMzAyM2YxZmM5ZGVlMDgiLAogICAgInRlc3QuY3N2IjogImJlMzg1MzQ0YjkzNjA2Yzc1Y2I2NWNhYzNkMzliMTNjZGVkYjViOTE2MjE2NDBlZTA4ZjczYjlhNTUxZmJkOWQiLAp9CgpDQU5PTklDQUxfREFUQVNFVF9ST1dTOiBkaWN0W3N0ciwgaW50XSA9IHsKICAgICJ0cmFpbi5jc3YiOiAyODcwOSwKICAgICJ2YWwuY3N2IjogMzU4OSwKICAgICJ0ZXN0LmNzdiI6IDM1ODksCn0KCkVYUEVDVEVEX0E2X01FVFJJQ1MgPSB7CiAgICAicmF3X2FjY3VyYWN5IjogMC42ODc2NTY3Mjg4OTM4NDIzLAogICAgInJhd19tYWNyb19mMSI6IDAuNjczNDgxODkzMzM0NTQwNywKICAgICJ0dGFfYWNjdXJhY3kiOiAwLjcwNjYwMzUxMDcyNzIyMjAsCiAgICAidHRhX21hY3JvX2YxIjogMC42OTgxNTgzMTUwNjMyNTc3LAogICAgImNoZWNrcG9pbnRfc2hhMjU2IjogIjIzZGJlOWIxNDUzZmRjN2U1ZGNhODFjYTJlOWJkMjZmMzYxZjViMWZlM2Q3ZmZlODAzYzY1NTQ2YjIyZDE2MmUiLAogICAgInNlbGVjdGVkX2Vwb2NoIjogNTcsCn0KCgpkZWYgc2hhMjU2X2ZpbGUocGF0aDogUGF0aCB8IHN0cikgLT4gc3RyOgogICAgaCA9IGhhc2hsaWIuc2hhMjU2KCkKICAgIHdpdGggb3BlbihwYXRoLCAicmIiKSBhcyBmOgogICAgICAgIHdoaWxlIGNodW5rIDo9IGYucmVhZCgxMDI0ICogMTAyNCk6CiAgICAgICAgICAgIGgudXBkYXRlKGNodW5rKQogICAgcmV0dXJuIGguaGV4ZGlnZXN0KCkKCgpkZWYgY291bnRfY3N2X3Jvd3MocGF0aDogUGF0aCB8IHN0cikgLT4gaW50OgogICAgd2l0aCBvcGVuKHBhdGgsICJyIiwgZW5jb2Rpbmc9InV0Zi04IikgYXMgZjoKICAgICAgICByZWFkZXIgPSBjc3YucmVhZGVyKGYpCiAgICAgICAgaGVhZGVyID0gbmV4dChyZWFkZXIsIE5vbmUpCiAgICAgICAgcmV0dXJuIHN1bSgxIGZvciBfIGluIHJlYWRlcikKCgpkZWYgdmFsaWRhdGVfc3BsaXRfaWRlbnRpdHkobmFtZTogc3RyLCBwYXRoOiBQYXRoIHwgc3RyKSAtPiBkaWN0W3N0ciwgQW55XToKICAgICIiIkZhaWwgY2xvc2VkIG9uIGRhdGFzZXQgaWRlbnRpdHkgdXNpbmcgYm90aCBTSEEyNTYgYW5kIHJvdyBjb3VudC4iIiIKICAgIHAgPSBQYXRoKHBhdGgpCiAgICBpZiBub3QgcC5pc19maWxlKCk6CiAgICAgICAgcmFpc2UgRmlsZU5vdEZvdW5kRXJyb3IoZiJSZXF1aXJlZCBkYXRhc2V0IGZpbGUgbWlzc2luZzoge3B9IikKICAgIGV4cGVjdGVkX2hhc2ggPSBDQU5PTklDQUxfREFUQVNFVF9IQVNIRVNbbmFtZV0KICAgIGV4cGVjdGVkX3Jvd3MgPSBDQU5PTklDQUxfREFUQVNFVF9ST1dTW25hbWVdCiAgICBhY3R1YWxfaGFzaCA9IHNoYTI1Nl9maWxlKHApCiAgICBhY3R1YWxfcm93cyA9IGNvdW50X2Nzdl9yb3dzKHApCiAgICBpZiBhY3R1YWxfaGFzaCAhPSBleHBlY3RlZF9oYXNoOgogICAgICAgIHJhaXNlIFZhbHVlRXJyb3IoCiAgICAgICAgICAgIGYiRGF0YXNldCBTSEEyNTYgbWlzbWF0Y2ggZm9yIHtuYW1lfTogZXhwZWN0ZWQge2V4cGVjdGVkX2hhc2h9LCBnb3Qge2FjdHVhbF9oYXNofSIKICAgICAgICApCiAgICBpZiBhY3R1YWxfcm93cyAhPSBleHBlY3RlZF9yb3dzOgogICAgICAgIHJhaXNlIFZhbHVlRXJyb3IoCiAgICAgICAgICAgIGYiRGF0YXNldCByb3cgY291bnQgbWlzbWF0Y2ggZm9yIHtuYW1lfTogZXhwZWN0ZWQge2V4cGVjdGVkX3Jvd3N9LCBnb3Qge2FjdHVhbF9yb3dzfSIKICAgICAgICApCiAgICByZXR1cm4gewogICAgICAgICJmaWxlIjogbmFtZSwKICAgICAgICAicGF0aCI6IHN0cihwLnJlc29sdmUoKSksCiAgICAgICAgInJvd3MiOiBhY3R1YWxfcm93cywKICAgICAgICAic2hhMjU2IjogYWN0dWFsX2hhc2gsCiAgICAgICAgInN0YXR1cyI6ICJQQVNTIiwKICAgIH0KCgpkZWYgdmFsaWRhdGVfYWxsX3NwbGl0cyh0cmFpbl9wYXRoOiBQYXRoLCB2YWxfcGF0aDogUGF0aCwgdGVzdF9wYXRoOiBQYXRoKSAtPiBkaWN0W3N0ciwgQW55XToKICAgIHRyYWluX3JlcyA9IHZhbGlkYXRlX3NwbGl0X2lkZW50aXR5KCJ0cmFpbi5jc3YiLCB0cmFpbl9wYXRoKQogICAgdmFsX3JlcyA9IHZhbGlkYXRlX3NwbGl0X2lkZW50aXR5KCJ2YWwuY3N2IiwgdmFsX3BhdGgpCiAgICB0ZXN0X3JlcyA9IHZhbGlkYXRlX3NwbGl0X2lkZW50aXR5KCJ0ZXN0LmNzdiIsIHRlc3RfcGF0aCkKICAgIHJldHVybiB7CiAgICAgICAgInRyYWluIjogdHJhaW5fcmVzLAogICAgICAgICJ2YWwiOiB2YWxfcmVzLAogICAgICAgICJ0ZXN0IjogdGVzdF9yZXMsCiAgICAgICAgImFsbF9wYXNzZWQiOiBUcnVlLAogICAgfQoKCmRlZiBjb21wdXRlX3BhaXJ3aXNlX2NvbmZpZ3VyYXRpb25fZGlmZnMoKSAtPiBsaXN0W2RpY3Rbc3RyLCBBbnldXToKICAgICIiIkdlbmVyYXRlIGFuZCB2YWxpZGF0ZSBwYWlyd2lzZSBjb25maWd1cmF0aW9uIGRpZmY6IEEwLT5BMSwgQTEtPkEyLCAuLi4sIEE1LT5BNi4KICAgIAogICAgRWFjaCB0cmFuc2l0aW9uIG11c3QgY29udGFpbiBleGFjdGx5IHRoZSBpbnRlbmRlZCBhcmNoaXRlY3R1cmFsIGFkZGl0aW9uLgogICAgIiIiCiAgICBkaWZmcyA9IFtdCiAgICBvcmRlciA9IGxpc3QoQ1VNVUxBVElWRV9BQkxBVElPTl9PUkRFUikKICAgIGZvciBpIGluIHJhbmdlKGxlbihvcmRlcikgLSAxKToKICAgICAgICBwcmV2X21vZGUgPSBvcmRlcltpXQogICAgICAgIG5leHRfbW9kZSA9IG9yZGVyW2kgKyAxXQogICAgICAgIHByZXZfc3BlYyA9IENVTVVMQVRJVkVfUkVHSVNUUllbcHJldl9tb2RlXQogICAgICAgIG5leHRfc3BlYyA9IENVTVVMQVRJVkVfUkVHSVNUUllbbmV4dF9tb2RlXQoKICAgICAgICBwcmV2X2RpY3QgPSBhc2RpY3QocHJldl9zcGVjKQogICAgICAgIG5leHRfZGljdCA9IGFzZGljdChuZXh0X3NwZWMpCgogICAgICAgIGNoYW5nZWRfa2V5cyA9IHt9CiAgICAgICAgZm9yIGsgaW4gcHJldl9kaWN0OgogICAgICAgICAgICBpZiBrIGluICgiaW50ZXJuYWxfaWQiLCAicGFwZXJfbmFtZSIsICJ2aWV0bmFtZXNlX25hbWUiLCAiY3VtdWxhdGl2ZV9kZXNjcmlwdGlvbiIsICJub3RlcyIpOgogICAgICAgICAgICAgICAgY29udGludWUKICAgICAgICAgICAgaWYgcHJldl9kaWN0W2tdICE9IG5leHRfZGljdFtrXToKICAgICAgICAgICAgICAgIGNoYW5nZWRfa2V5c1trXSA9IHsiZnJvbSI6IHByZXZfZGljdFtrXSwgInRvIjogbmV4dF9kaWN0W2tdfQoKICAgICAgICAjIENoZWNrIGludGVuZGVkIGFyY2hpdGVjdHVyYWwgY2hhbmdlIHBlciBzdGVwCiAgICAgICAgaW50ZW5kZWRfY2hhbmdlID0gTm9uZQogICAgICAgIGlmIHByZXZfbW9kZSA9PSBDdW11bGF0aXZlQWJsYXRpb25Nb2RlLkEwIGFuZCBuZXh0X21vZGUgPT0gQ3VtdWxhdGl2ZUFibGF0aW9uTW9kZS5BMToKICAgICAgICAgICAgaW50ZW5kZWRfY2hhbmdlID0gIkFkZCBTcGF0aWFsIE1vdGlmIENvbXBvc2VyICgxMngxMikgKyBGaXhlZCBwb29saW5nICsgZW5hYmxlIG1vdGlmIGxvc3NlcyIKICAgICAgICAgICAgZXhwZWN0ZWRfa2V5cyA9IHsiY29tcG9zZXIiLCAiY29tcG9zZXJfc2NhbGVzIiwgIm1vdGlmX3Bvb2xpbmciLCAiYXBwbGljYWJsZV9sb3NzZXMifQogICAgICAgIGVsaWYgcHJldl9tb2RlID09IEN1bXVsYXRpdmVBYmxhdGlvbk1vZGUuQTEgYW5kIG5leHRfbW9kZSA9PSBDdW11bGF0aXZlQWJsYXRpb25Nb2RlLkEyOgogICAgICAgICAgICBpbnRlbmRlZF9jaGFuZ2UgPSAiRW5hYmxlIE1vdGlmIEdOTiAoZGVuc2UgcmVsYXRpb25zKSIKICAgICAgICAgICAgZXhwZWN0ZWRfa2V5cyA9IHsibW90aWZfZ25uIiwgInJlbGF0aW9uX21vZGUiLCAibW90aWZfdG9wa19zY2hlZHVsZSJ9CiAgICAgICAgZWxpZiBwcmV2X21vZGUgPT0gQ3VtdWxhdGl2ZUFibGF0aW9uTW9kZS5BMiBhbmQgbmV4dF9tb2RlID09IEN1bXVsYXRpdmVBYmxhdGlvbk1vZGUuQTM6CiAgICAgICAgICAgIGludGVuZGVkX2NoYW5nZSA9ICJFbmFibGUgR2VvbWV0cnkgQmlhcyBpbiBNb3RpZiBHTk4iCiAgICAgICAgICAgIGV4cGVjdGVkX2tleXMgPSB7Imdlb21ldHJ5X2JpYXMifQogICAgICAgIGVsaWYgcHJldl9tb2RlID09IEN1bXVsYXRpdmVBYmxhdGlvbk1vZGUuQTMgYW5kIG5leHRfbW9kZSA9PSBDdW11bGF0aXZlQWJsYXRpb25Nb2RlLkE0OgogICAgICAgICAgICBpbnRlbmRlZF9jaGFuZ2UgPSAiRW5hYmxlIE11bHRpLXNjYWxlIENvbXBvc2l0aW9uICg4LCAxMiwgMTYpIgogICAgICAgICAgICBleHBlY3RlZF9rZXlzID0geyJjb21wb3NlciIsICJjb21wb3Nlcl9zY2FsZXMifQogICAgICAgIGVsaWYgcHJldl9tb2RlID09IEN1bXVsYXRpdmVBYmxhdGlvbk1vZGUuQTQgYW5kIG5leHRfbW9kZSA9PSBDdW11bGF0aXZlQWJsYXRpb25Nb2RlLkE1OgogICAgICAgICAgICBpbnRlbmRlZF9jaGFuZ2UgPSAiRW5hYmxlIER5bmFtaWMgVG9wLUsgUmVsYXRpb25zIFs4LCAxNiwgMTYsIDE2LCAyNF0iCiAgICAgICAgICAgIGV4cGVjdGVkX2tleXMgPSB7InJlbGF0aW9uX21vZGUiLCAibW90aWZfdG9wa19zY2hlZHVsZSJ9CiAgICAgICAgZWxpZiBwcmV2X21vZGUgPT0gQ3VtdWxhdGl2ZUFibGF0aW9uTW9kZS5BNSBhbmQgbmV4dF9tb2RlID09IEN1bXVsYXRpdmVBYmxhdGlvbk1vZGUuQTY6CiAgICAgICAgICAgIGludGVuZGVkX2NoYW5nZSA9ICJSZXBsYWNlIEZpeGVkIFBvb2xpbmcgd2l0aCBMZWFybmFibGUgQXR0ZW50aW9uIFBvb2xpbmcgKEZ1bGwgTW9kZWwpIgogICAgICAgICAgICBleHBlY3RlZF9rZXlzID0geyJtb3RpZl9wb29saW5nIn0KICAgICAgICBlbHNlOgogICAgICAgICAgICByYWlzZSBWYWx1ZUVycm9yKGYiVW5leHBlY3RlZCBzdGVwOiB7cHJldl9tb2RlfSAtPiB7bmV4dF9tb2RlfSIpCgogICAgICAgIGFjdHVhbF9rZXlzID0gc2V0KGNoYW5nZWRfa2V5cy5rZXlzKCkpCiAgICAgICAgaWYgYWN0dWFsX2tleXMgIT0gZXhwZWN0ZWRfa2V5czoKICAgICAgICAgICAgcmFpc2UgVmFsdWVFcnJvcigKICAgICAgICAgICAgICAgIGYiUGFpcndpc2UgZGlmZiB2aW9sYXRpb24ge3ByZXZfbW9kZS52YWx1ZX0tPntuZXh0X21vZGUudmFsdWV9OiAiCiAgICAgICAgICAgICAgICBmImV4cGVjdGVkIGNoYW5nZSBpbiB7ZXhwZWN0ZWRfa2V5c30sIGJ1dCBnb3Qge2FjdHVhbF9rZXlzfSB3aXRoIGRpZmZzOiB7Y2hhbmdlZF9rZXlzfSIKICAgICAgICAgICAgKQoKICAgICAgICBkaWZmcy5hcHBlbmQoewogICAgICAgICAgICAidHJhbnNpdGlvbiI6IGYie3ByZXZfbW9kZS52YWx1ZX0tPntuZXh0X21vZGUudmFsdWV9IiwKICAgICAgICAgICAgImZyb21fY29uZmlnIjogcHJldl9zcGVjLnBhcGVyX25hbWUsCiAgICAgICAgICAgICJ0b19jb25maWciOiBuZXh0X3NwZWMucGFwZXJfbmFtZSwKICAgICAgICAgICAgImludGVuZGVkX2NoYW5nZSI6IGludGVuZGVkX2NoYW5nZSwKICAgICAgICAgICAgImNoYW5nZWRfZmllbGRzIjogY2hhbmdlZF9rZXlzLAogICAgICAgICAgICAic3RhdHVzIjogIlZBTElEIiwKICAgICAgICB9KQogICAgcmV0dXJuIGRpZmZzCgoKZGVmIGJ1aWxkX2N1bXVsYXRpdmVfYWJsYXRpb25fbWF0cml4KCkgLT4gZGljdFtzdHIsIEFueV06CiAgICByb3dzID0gW10KICAgIGZvciBtb2RlIGluIENVTVVMQVRJVkVfQUJMQVRJT05fT1JERVI6CiAgICAgICAgc3BlYyA9IENVTVVMQVRJVkVfUkVHSVNUUllbbW9kZV0KICAgICAgICByb3dzLmFwcGVuZCh7CiAgICAgICAgICAgICJjb25maWd1cmF0aW9uIjogc3BlYy5pbnRlcm5hbF9pZCwKICAgICAgICAgICAgImRpc3BsYXlfbmFtZSI6IHNwZWMucGFwZXJfbmFtZSwKICAgICAgICAgICAgInZpZXRuYW1lc2VfbmFtZSI6IHNwZWMudmlldG5hbWVzZV9uYW1lLAogICAgICAgICAgICAicGl4ZWxfZ25uIjogc3BlYy5waXhlbF9nbm4sCiAgICAgICAgICAgICJjb21wb3NlciI6IHNwZWMuY29tcG9zZXIgb3IgIk5vbmUiLAogICAgICAgICAgICAiY29tcG9zZXJfc2NhbGVzIjogc3BlYy5jb21wb3Nlcl9zY2FsZXMsCiAgICAgICAgICAgICJtb3RpZl9nbm4iOiBzcGVjLm1vdGlmX2dubiwKICAgICAgICAgICAgImdlb21ldHJ5X2JpYXMiOiBzcGVjLmdlb21ldHJ5X2JpYXMsCiAgICAgICAgICAgICJyZWxhdGlvbl9tb2RlIjogc3BlYy5yZWxhdGlvbl9tb2RlLAogICAgICAgICAgICAibW90aWZfdG9wa19zY2hlZHVsZSI6IHNwZWMubW90aWZfdG9wa19zY2hlZHVsZSwKICAgICAgICAgICAgIm1vdGlmX3Bvb2xpbmciOiBzcGVjLm1vdGlmX3Bvb2xpbmcsCiAgICAgICAgICAgICJkaXJlY3RfcGl4ZWxfZnVzaW9uIjogc3BlYy5kaXJlY3RfcGl4ZWxfZnVzaW9uLAogICAgICAgICAgICAiY2xhc3NpZmllciI6IHNwZWMuY2xhc3NpZmllciwKICAgICAgICAgICAgImFwcGxpY2FibGVfbG9zc2VzIjogc3BlYy5hcHBsaWNhYmxlX2xvc3NlcywKICAgICAgICAgICAgIm5vdGVzIjogc3BlYy5ub3RlcywKICAgICAgICB9KQogICAgcGFpcndpc2VfZGlmZnMgPSBjb21wdXRlX3BhaXJ3aXNlX2NvbmZpZ3VyYXRpb25fZGlmZnMoKQogICAgcmV0dXJuIHsKICAgICAgICAic2NoZW1hX3ZlcnNpb24iOiAxLAogICAgICAgICJtZXRob2QiOiAiTVBHLUZFUiIsCiAgICAgICAgIm5lc3RlZF9sYWRkZXJfcHJvcGVydHkiOiAiQTAg4oqCIEExIOKKgiBBMiDiioIgQTMg4oqCIEE0IOKKgiBBNSDiioIgQTYiLAogICAgICAgICJyb3dzIjogcm93cywKICAgICAgICAicGFpcndpc2VfdHJhbnNpdGlvbnMiOiBwYWlyd2lzZV9kaWZmcywKICAgIH0K', 'mpg_fer_cumulative_ablation7/train.py': '...', 'mpg_fer_table_vi/__init__.py': 'IiIiQ2Fub25pY2FsIHNldmVuLXZhcmlhbnQgTVBHLUZFUiBUYWJsZSBWSSBmcmFtZXdvcmsgKElzc3VlICMxMDEpLiIiIgoKZnJvbSAubW9kZWwgaW1wb3J0IEFCTEFUSU9OX1JFR0lTVFJZLCBBYmxhdGlvbk1vZGUsIEFibGF0aW9uTVBHRkVSCmZyb20gLnByb3RvY29sIGltcG9ydCBBYmxhdGlvbkNvbmZpZwoKX19hbGxfXyA9IFsKICAgICJBQkxBVElPTl9SRUdJU1RSWSIsCiAgICAiQWJsYXRpb25Nb2RlIiwKICAgICJBYmxhdGlvbk1QR0ZFUiIsCiAgICAiQWJsYXRpb25Db25maWciLApdCg==', 'mpg_fer_table_vi/model.py': '...', 'mpg_fer_table_vi/protocol.py': '...', 'mpg_fer_table_vi/train.py': '...', 'mpg_fer_v2_3/__init__.py': 'IiIiUm9vdCBwYWNrYWdlIGZvciB0aGUgSXNzdWUgIzk3IE1QRy1GRVIgdjIuMyBpbXBsZW1lbnRhdGlvbi4iIiIKCmZyb20gX19mdXR1cmVfXyBpbXBvcnQgYW5ub3RhdGlvbnMKCmZyb20gLmNvbmZpZyBpbXBvcnQgTVBHQ29uZmlnCmZyb20gLmZlYXR1cmVzIGltcG9ydCBQaXhlbEZlYXR1cmVFeHRyYWN0b3IKZnJvbSAuZ3JhcGggaW1wb3J0IFBpeGVsR3JhcGhUb3BvbG9neQpmcm9tIC5tb3RpZiBpbXBvcnQgU3BhdGlhbE1vdGlmQ29tcG9zZXIKZnJvbSAubW9kZWwgaW1wb3J0IE1QR0ZFUgpmcm9tIC5lbWEgaW1wb3J0IE1vZGVsRU1BCgpfX2FsbF9fID0gWwogICAgIk1QR0NvbmZpZyIsCiAgICAiUGl4ZWxGZWF0dXJlRXh0cmFjdG9yIiwKICAgICJQaXhlbEdyYXBoVG9wb2xvZ3kiLAogICAgIlNwYXRpYWxNb3RpZkNvbXBvc2VyIiwKICAgICJNUEdGRVIiLAogICAgIk1vZGVsRU1BIiwKXQo=', 'mpg_fer_v2_3/checkpoint.py': 'IiIiQXRvbWljLCBoYXNoLXZlcmlmaWVkLCBmdWxsLXN0YXRlIHJlc3VtZSBidW5kbGVzIGZvciBNUEctRkVSIHYyLjIuIiIiCgpmcm9tIF9fZnV0dXJlX18gaW1wb3J0IGFubm90YXRpb25zCgpmcm9tIGRhdGFjbGFzc2VzIGltcG9ydCBhc2RpY3QKZnJvbSBkYXRldGltZSBpbXBvcnQgZGF0ZXRpbWUsIHRpbWV6b25lCmltcG9ydCBoYXNobGliCmltcG9ydCBqc29uCmltcG9ydCBvcwpmcm9tIHBhdGhsaWIgaW1wb3J0IFBhdGgKaW1wb3J0IHJhbmRvbQppbXBvcnQgc2h1dGlsCmZyb20gdHlwaW5nIGltcG9ydCBBbnkKCmltcG9ydCBudW1weSBhcyBucAppbXBvcnQgdG9yY2gKCmZyb20gLmNvbmZpZyBpbXBvcnQgTVBHQ29uZmlnCgoKZGVmIHNoYTI1Nl9maWxlKHBhdGg6IHN0ciB8IFBhdGgpIC0+IHN0cjoKICAgIGRpZ2VzdCA9IGhhc2hsaWIuc2hhMjU2KCkKICAgIHdpdGggUGF0aChwYXRoKS5vcGVuKCJyYiIpIGFzIGhhbmRsZToKICAgICAgICBmb3IgY2h1bmsgaW4gaXRlcihsYW1iZGE6IGhhbmRsZS5yZWFkKDEwMjQgKiAxMDI0KSwgYiIiKToKICAgICAgICAgICAgZGlnZXN0LnVwZGF0ZShjaHVuaykKICAgIHJldHVybiBkaWdlc3QuaGV4ZGlnZXN0KCkKCgpkZWYgY2Fub25pY2FsX2NvbmZpZyhjb25maWc6IE1QR0NvbmZpZywgaW5jbHVkZV9ydW50aW1lX3NhZmU6IGJvb2wgPSBGYWxzZSkgLT4gZGljdDoKICAgIGRhdGEgPSBhc2RpY3QoY29uZmlnKQogICAgc2FmZSA9IHNldChjb25maWcucnVudGltZV9zYWZlX3Jlc3VtZV9maWVsZHMpIHwgeyJydW50aW1lX3NhZmVfcmVzdW1lX2ZpZWxkcyJ9CiAgICByZXR1cm4gZGF0YSBpZiBpbmNsdWRlX3J1bnRpbWVfc2FmZSBlbHNlIHtrOiB2IGZvciBrLCB2IGluIGRhdGEuaXRlbXMoKSBpZiBrIG5vdCBpbiBzYWZlfQoKCmRlZiBjb25maWdfaGFzaChjb25maWc6IE1QR0NvbmZpZykgLT4gc3RyOgogICAgcGF5bG9hZCA9IGpzb24uZHVtcHMoY2Fub25pY2FsX2NvbmZpZyhjb25maWcpLCBzb3J0X2tleXM9VHJ1ZSwgc2VwYXJhdG9ycz0oIiwiLCAiOiIpKQogICAgcmV0dXJuIGhhc2hsaWIuc2hhMjU2KHBheWxvYWQuZW5jb2RlKCJ1dGYtOCIpKS5oZXhkaWdlc3QoKQoKCmRlZiBjYXB0dXJlX3JuZ19zdGF0ZShsb2FkZXJfZ2VuZXJhdG9yOiB0b3JjaC5HZW5lcmF0b3IpIC0+IGRpY3Q6CiAgICByZXR1cm4gewogICAgICAgICJweXRob24iOiByYW5kb20uZ2V0c3RhdGUoKSwKICAgICAgICAibnVtcHkiOiBucC5yYW5kb20uZ2V0X3N0YXRlKCksCiAgICAgICAgInRvcmNoX2NwdSI6IHRvcmNoLmdldF9ybmdfc3RhdGUoKSwKICAgICAgICAidG9yY2hfY3VkYSI6IHRvcmNoLmN1ZGEuZ2V0X3JuZ19zdGF0ZV9hbGwoKSBpZiB0b3JjaC5jdWRhLmlzX2F2YWlsYWJsZSgpIGVsc2UgW10sCiAgICAgICAgImRhdGFsb2FkZXJfZ2VuZXJhdG9yIjogbG9hZGVyX2dlbmVyYXRvci5nZXRfc3RhdGUoKSwKICAgIH0KCgpkZWYgcmVzdG9yZV9ybmdfc3RhdGUoc3RhdGU6IGRpY3QsIGxvYWRlcl9nZW5lcmF0b3I6IHRvcmNoLkdlbmVyYXRvcikgLT4gTm9uZToKICAgIHJhbmRvbS5zZXRzdGF0ZShzdGF0ZVsicHl0aG9uIl0pCiAgICBucC5yYW5kb20uc2V0X3N0YXRlKHN0YXRlWyJudW1weSJdKQogICAgdG9yY2guc2V0X3JuZ19zdGF0ZShzdGF0ZVsidG9yY2hfY3B1Il0pCiAgICBpZiB0b3JjaC5jdWRhLmlzX2F2YWlsYWJsZSgpIGFuZCBzdGF0ZVsidG9yY2hfY3VkYSJdOgogICAgICAgIHRvcmNoLmN1ZGEuc2V0X3JuZ19zdGF0ZV9hbGwoc3RhdGVbInRvcmNoX2N1ZGEiXSkKICAgIGxvYWRlcl9nZW5lcmF0b3Iuc2V0X3N0YXRlKHN0YXRlWyJkYXRhbG9hZGVyX2dlbmVyYXRvciJdKQoKCmRlZiBidWlsZF9yZXN1bWVfYnVuZGxlKAogICAgKiwgY29uZmlnOiBNUEdDb25maWcsIHJ1bl9pZDogc3RyLCBzb3VyY2VfaGFzaDogc3RyLCBjb21wbGV0ZWRfZXBvY2g6IGludCwKICAgIGdsb2JhbF9vcHRpbWl6ZXJfc3RlcDogaW50LCBtb2RlbDogdG9yY2gubm4uTW9kdWxlLCBlbWE6IEFueSwKICAgIG9wdGltaXplcjogdG9yY2gub3B0aW0uT3B0aW1pemVyLCBzY2hlZHVsZXI6IEFueSwgc2NhbGVyOiBBbnksCiAgICBiZXN0X2NvbXBhcmF0b3Jfc3RhdGU6IGRpY3QgfCBOb25lLCBiZXN0X2Vwb2NoOiBpbnQgfCBOb25lLAogICAgYmVzdF9tZXRyaWNzOiBkaWN0IHwgTm9uZSwgZWFybHlfc3RvcF9jb3VudGVyOiBpbnQsIGhpc3Rvcnk6IGxpc3RbZGljdF0sCiAgICBsb2FkZXJfZ2VuZXJhdG9yOiB0b3JjaC5HZW5lcmF0b3IsIGNvbnNpc3RlbmN5X3N0YXRlOiBkaWN0LAogICAgc2FtcGxlcl9zdGF0ZTogZGljdCB8IE5vbmUgPSBOb25lLAogICAgYXVnbWVudGF0aW9uX3N0YXRlOiBkaWN0IHwgTm9uZSA9IE5vbmUsCikgLT4gZGljdDoKICAgIHRlbXBlcmF0dXJlX3N0YXRlID0gTm9uZQogICAgaWYgaGFzYXR0cihtb2RlbCwgIm1vdGlmX2NvbXBvc2VyIik6CiAgICAgICAgdGVtcGVyYXR1cmVfc3RhdGUgPSB7CiAgICAgICAgICAgICJlcG9jaCI6IGNvbXBsZXRlZF9lcG9jaCwKICAgICAgICAgICAgImN1cnJlbnRfdGF1IjogZmxvYXQobW9kZWwubW90aWZfY29tcG9zZXIudGVtcGVyYXR1cmUpLAogICAgICAgIH0KICAgIHJldHVybiB7CiAgICAgICAgInJlc3VtZV9zY2hlbWFfdmVyc2lvbiI6IGNvbmZpZy5yZXN1bWVfc2NoZW1hX3ZlcnNpb24sCiAgICAgICAgInJ1bl9pZCI6IHJ1bl9pZCwKICAgICAgICAic291cmNlX2hhc2giOiBzb3VyY2VfaGFzaCwKICAgICAgICAiY29uZmlnX2hhc2giOiBjb25maWdfaGFzaChjb25maWcpLAogICAgICAgICJzY2llbnRpZmljX2NvbmZpZyI6IGNhbm9uaWNhbF9jb25maWcoY29uZmlnKSwKICAgICAgICAiY29tcGxldGVkX2Vwb2NoIjogY29tcGxldGVkX2Vwb2NoLAogICAgICAgICJuZXh0X2Vwb2NoIjogY29tcGxldGVkX2Vwb2NoICsgMSwKICAgICAgICAiZ2xvYmFsX29wdGltaXplcl9zdGVwIjogZ2xvYmFsX29wdGltaXplcl9zdGVwLAogICAgICAgICJtb2RlbF9zdGF0ZV9kaWN0IjogbW9kZWwuc3RhdGVfZGljdCgpLAogICAgICAgICJlbWFfc3RhdGVfZGljdCI6IGVtYS5zdGF0ZV9kaWN0KCksCiAgICAgICAgIm9wdGltaXplcl9zdGF0ZV9kaWN0Ijogb3B0aW1pemVyLnN0YXRlX2RpY3QoKSwKICAgICAgICAic2NoZWR1bGVyX3N0YXRlX2RpY3QiOiBzY2hlZHVsZXIuc3RhdGVfZGljdCgpLAogICAgICAgICJncmFkX3NjYWxlcl9zdGF0ZV9kaWN0Ijogc2NhbGVyLnN0YXRlX2RpY3QoKSBpZiBzY2FsZXIgaXMgbm90IE5vbmUgZWxzZSB7fSwKICAgICAgICAiYmVzdF9jb21wYXJhdG9yX3N0YXRlIjogYmVzdF9jb21wYXJhdG9yX3N0YXRlLAogICAgICAgICJiZXN0X2Vwb2NoIjogYmVzdF9lcG9jaCwKICAgICAgICAiYmVzdF9tZXRyaWNzIjogYmVzdF9tZXRyaWNzLAogICAgICAgICJlYXJseV9zdG9wX2NvdW50ZXIiOiBlYXJseV9zdG9wX2NvdW50ZXIsCiAgICAgICAgImVhcmx5X3N0b3BfbW9uaXRvcl9zdGF0ZSI6IHsKICAgICAgICAgICAgInN0YXJ0X2Vwb2NoIjogY29uZmlnLmVhcmx5X3N0b3BfbW9uaXRvcl9zdGFydF9lcG9jaCwKICAgICAgICAgICAgImFjdGl2ZSI6IGNvbXBsZXRlZF9lcG9jaCA+PSBjb25maWcuZWFybHlfc3RvcF9tb25pdG9yX3N0YXJ0X2Vwb2NoLAogICAgICAgICAgICAicGF0aWVuY2UiOiBlYXJseV9zdG9wX2NvdW50ZXIsCiAgICAgICAgfSwKICAgICAgICAidGVtcGVyYXR1cmVfc3RhdGUiOiB0ZW1wZXJhdHVyZV9zdGF0ZSwKICAgICAgICAiaGlzdG9yeSI6IGhpc3RvcnksCiAgICAgICAgInJuZ19zdGF0ZSI6IGNhcHR1cmVfcm5nX3N0YXRlKGxvYWRlcl9nZW5lcmF0b3IpLAogICAgICAgICJjb25zaXN0ZW5jeV9zdGF0ZSI6IGNvbnNpc3RlbmN5X3N0YXRlLAogICAgICAgICJzYW1wbGVyX3N0YXRlIjogc2FtcGxlcl9zdGF0ZSwKICAgICAgICAiYXVnbWVudGF0aW9uX3N0YXRlIjogYXVnbWVudGF0aW9uX3N0YXRlLAogICAgfQoKCmRlZiBhdG9taWNfc2F2ZV9yZXN1bWUoYnVuZGxlOiBkaWN0LCBvdXRwdXRfZGlyOiBzdHIgfCBQYXRoLCBzdGF0dXM6IHN0ciA9ICJUUkFJTklORyIpIC0+IHR1cGxlW1BhdGgsIHN0cl06CiAgICBvdXRwdXQgPSBQYXRoKG91dHB1dF9kaXIpCiAgICBvdXRwdXQubWtkaXIocGFyZW50cz1UcnVlLCBleGlzdF9vaz1UcnVlKQogICAgdGFyZ2V0ID0gb3V0cHV0IC8gInJlc3VtZV9sYXRlc3QucHQiCiAgICB0ZW1wb3JhcnkgPSBvdXRwdXQgLyAicmVzdW1lX2xhdGVzdC50bXAiCiAgICB3aXRoIHRlbXBvcmFyeS5vcGVuKCJ3YiIpIGFzIGhhbmRsZToKICAgICAgICB0b3JjaC5zYXZlKGJ1bmRsZSwgaGFuZGxlKQogICAgICAgIGhhbmRsZS5mbHVzaCgpCiAgICAgICAgb3MuZnN5bmMoaGFuZGxlLmZpbGVubygpKQogICAgb3MucmVwbGFjZSh0ZW1wb3JhcnksIHRhcmdldCkKICAgIGRpZ2VzdCA9IHNoYTI1Nl9maWxlKHRhcmdldCkKICAgIG1ldGFkYXRhID0gewogICAgICAgICJydW5faWQiOiBidW5kbGVbInJ1bl9pZCJdLAogICAgICAgICJlcG9jaCI6IGJ1bmRsZVsiY29tcGxldGVkX2Vwb2NoIl0sCiAgICAgICAgIm5leHRfZXBvY2giOiBidW5kbGVbIm5leHRfZXBvY2giXSwKICAgICAgICAiY2hlY2twb2ludF9zaGEyNTYiOiBkaWdlc3QsCiAgICAgICAgInNvdXJjZV9oYXNoIjogYnVuZGxlWyJzb3VyY2VfaGFzaCJdLAogICAgICAgICJjb25maWdfaGFzaCI6IGJ1bmRsZVsiY29uZmlnX2hhc2giXSwKICAgICAgICAidGltZXN0YW1wIjogZGF0ZXRpbWUubm93KHRpbWV6b25lLnV0YykuaXNvZm9ybWF0KCksCiAgICAgICAgInN0YXR1cyI6IHN0YXR1cywKICAgIH0KICAgIG1ldGFfdG1wID0gb3V0cHV0IC8gInJlc3VtZV9sYXRlc3QuanNvbi50bXAiCiAgICBtZXRhX3RhcmdldCA9IG91dHB1dCAvICJyZXN1bWVfbGF0ZXN0Lmpzb24iCiAgICB3aXRoIG1ldGFfdG1wLm9wZW4oInciLCBlbmNvZGluZz0idXRmLTgiKSBhcyBoYW5kbGU6CiAgICAgICAganNvbi5kdW1wKG1ldGFkYXRhLCBoYW5kbGUsIGluZGVudD0yKQogICAgICAgIGhhbmRsZS53cml0ZSgiXG4iKQogICAgICAgIGhhbmRsZS5mbHVzaCgpCiAgICAgICAgb3MuZnN5bmMoaGFuZGxlLmZpbGVubygpKQogICAgb3MucmVwbGFjZShtZXRhX3RtcCwgbWV0YV90YXJnZXQpCiAgICByZXR1cm4gdGFyZ2V0LCBkaWdlc3QKCgpkZWYgX2F0b21pY19qc29uKHBhdGg6IFBhdGgsIHBheWxvYWQ6IGRpY3QpIC0+IE5vbmU6CiAgICB0ZW1wb3JhcnkgPSBwYXRoLndpdGhfc3VmZml4KHBhdGguc3VmZml4ICsgIi50bXAiKQogICAgd2l0aCB0ZW1wb3Jhcnkub3BlbigidyIsIGVuY29kaW5nPSJ1dGYtOCIpIGFzIGhhbmRsZToKICAgICAgICBqc29uLmR1bXAocGF5bG9hZCwgaGFuZGxlLCBpbmRlbnQ9MikKICAgICAgICBoYW5kbGUud3JpdGUoIlxuIikKICAgICAgICBoYW5kbGUuZmx1c2goKQogICAgICAgIG9zLmZzeW5jKGhhbmRsZS5maWxlbm8oKSkKICAgIG9zLnJlcGxhY2UodGVtcG9yYXJ5LCBwYXRoKQoKCmRlZiBzYXZlX3BlcmlvZGljX3NuYXBzaG90KGxhdGVzdDogUGF0aCwgZXBvY2g6IGludCwgaW50ZXJ2YWw6IGludCwga2VlcDogaW50KSAtPiBQYXRoIHwgTm9uZToKICAgIGlmIGludGVydmFsIDw9IDAgb3IgZXBvY2ggJSBpbnRlcnZhbDoKICAgICAgICByZXR1cm4gTm9uZQogICAgc25hcHNob3QgPSBsYXRlc3Qud2l0aF9uYW1lKGYicmVzdW1lX2Vwb2NoX3tlcG9jaDowM2R9LnB0IikKICAgIGlmIG5vdCBzbmFwc2hvdC5leGlzdHMoKToKICAgICAgICB0ZW1wb3JhcnkgPSBzbmFwc2hvdC53aXRoX3N1ZmZpeCgiLnRtcCIpCiAgICAgICAgd2l0aCBsYXRlc3Qub3BlbigicmIiKSBhcyBzb3VyY2UsIHRlbXBvcmFyeS5vcGVuKCJ3YiIpIGFzIGRlc3RpbmF0aW9uOgogICAgICAgICAgICBzaHV0aWwuY29weWZpbGVvYmooc291cmNlLCBkZXN0aW5hdGlvbiwgbGVuZ3RoPTEwMjQgKiAxMDI0KQogICAgICAgICAgICBkZXN0aW5hdGlvbi5mbHVzaCgpCiAgICAgICAgICAgIG9zLmZzeW5jKGRlc3RpbmF0aW9uLmZpbGVubygpKQogICAgICAgIG9zLnJlcGxhY2UodGVtcG9yYXJ5LCBzbmFwc2hvdCkKICAgICAgICBidW5kbGUgPSB0b3JjaC5sb2FkKHNuYXBzaG90LCBtYXBfbG9jYXRpb249ImNwdSIsIHdlaWdodHNfb25seT1GYWxzZSkKICAgICAgICBfYXRvbWljX2pzb24oCiAgICAgICAgICAgIHNuYXBzaG90LndpdGhfc3VmZml4KCIuanNvbiIpLAogICAgICAgICAgICB7CiAgICAgICAgICAgICAgICAiY2hlY2twb2ludF9zaGEyNTYiOiBzaGEyNTZfZmlsZShzbmFwc2hvdCksCiAgICAgICAgICAgICAgICAicmVzdW1lX3NjaGVtYV92ZXJzaW9uIjogYnVuZGxlWyJyZXN1bWVfc2NoZW1hX3ZlcnNpb24iXSwKICAgICAgICAgICAgICAgICJydW5faWQiOiBidW5kbGVbInJ1bl9pZCJdLAogICAgICAgICAgICAgICAgInNvdXJjZV9oYXNoIjogYnVuZGxlWyJzb3VyY2VfaGFzaCJdLAogICAgICAgICAgICAgICAgImNvbmZpZ19oYXNoIjogYnVuZGxlWyJjb25maWdfaGFzaCJdLAogICAgICAgICAgICAgICAgImNvbXBsZXRlZF9lcG9jaCI6IGJ1bmRsZVsiY29tcGxldGVkX2Vwb2NoIl0sCiAgICAgICAgICAgICAgICAibmV4dF9lcG9jaCI6IGJ1bmRsZVsibmV4dF9lcG9jaCJdLAogICAgICAgICAgICAgICAgInN0YXR1cyI6ICJJTU1VVEFCTEVfRkFMTEJBQ0siLAogICAgICAgICAgICB9LAogICAgICAgICkKICAgIHNuYXBzaG90cyA9IHNvcnRlZChsYXRlc3QucGFyZW50Lmdsb2IoInJlc3VtZV9lcG9jaF8qLnB0IikpCiAgICBmb3Igb2xkIGluIHNuYXBzaG90c1s6LW1heChrZWVwLCAyKV06CiAgICAgICAgb2xkLnVubGluaygpCiAgICAgICAgb2xkLndpdGhfc3VmZml4KCIuanNvbiIpLnVubGluayhtaXNzaW5nX29rPVRydWUpCiAgICByZXR1cm4gc25hcHNob3QKCgpkZWYgZmluZF9sYXRlc3RfdmFsaWRfc25hcHNob3QoCiAgICBkaXJlY3Rvcnk6IHN0ciB8IFBhdGgsCiAgICBleHBlY3RlZF9pZGVudGl0eTogZGljdCB8IE5vbmUgPSBOb25lLAopIC0+IHR1cGxlW1BhdGgsIHN0cl0gfCBOb25lOgogICAgIiIiUmVwb3J0IHRoZSBuZXdlc3QgaW5kZXBlbmRlbnRseSBoYXNoLXZhbGlkIGltbXV0YWJsZSBmYWxsYmFjay4iIiIKICAgIHJvb3QgPSBQYXRoKGRpcmVjdG9yeSkKICAgIGZvciBtZXRhZGF0YV9wYXRoIGluIHNvcnRlZChyb290Lmdsb2IoInJlc3VtZV9lcG9jaF8qLmpzb24iKSwgcmV2ZXJzZT1UcnVlKToKICAgICAgICB0cnk6CiAgICAgICAgICAgIG1ldGFkYXRhID0ganNvbi5sb2FkcyhtZXRhZGF0YV9wYXRoLnJlYWRfdGV4dChlbmNvZGluZz0idXRmLTgiKSkKICAgICAgICAgICAgaWYgZXhwZWN0ZWRfaWRlbnRpdHkgaXMgbm90IE5vbmUgYW5kIGFueSgKICAgICAgICAgICAgICAgIGV4cGVjdGVkX2lkZW50aXR5LmdldChrZXkpIGlzIG5vdCBOb25lCiAgICAgICAgICAgICAgICBhbmQgbWV0YWRhdGEuZ2V0KGtleSkgIT0gZXhwZWN0ZWRfaWRlbnRpdHlba2V5XQogICAgICAgICAgICAgICAgZm9yIGtleSBpbiAoCiAgICAgICAgICAgICAgICAgICAgInJlc3VtZV9zY2hlbWFfdmVyc2lvbiIsICJydW5faWQiLCAic291cmNlX2hhc2giLCAiY29uZmlnX2hhc2giCiAgICAgICAgICAgICAgICApCiAgICAgICAgICAgICk6CiAgICAgICAgICAgICAgICBjb250aW51ZQogICAgICAgICAgICBjaGVja3BvaW50ID0gbWV0YWRhdGFfcGF0aC53aXRoX3N1ZmZpeCgiLnB0IikKICAgICAgICAgICAgZXhwZWN0ZWQgPSBtZXRhZGF0YVsiY2hlY2twb2ludF9zaGEyNTYiXQogICAgICAgICAgICBpZiBjaGVja3BvaW50LmlzX2ZpbGUoKSBhbmQgc2hhMjU2X2ZpbGUoY2hlY2twb2ludCkgPT0gZXhwZWN0ZWQ6CiAgICAgICAgICAgICAgICByZXR1cm4gY2hlY2twb2ludCwgZXhwZWN0ZWQKICAgICAgICBleGNlcHQgKE9TRXJyb3IsIEtleUVycm9yLCBWYWx1ZUVycm9yLCBqc29uLkpTT05EZWNvZGVFcnJvcik6CiAgICAgICAgICAgIGNvbnRpbnVlCiAgICByZXR1cm4gTm9uZQoKCmRlZiBsb2FkX3Jlc3VtZV9idW5kbGUoCiAgICBwYXRoOiBzdHIgfCBQYXRoLCAqLCBleHBlY3RlZF9zaGEyNTY6IHN0ciB8IE5vbmUsIGNvbmZpZzogTVBHQ29uZmlnLAogICAgcnVuX2lkOiBzdHIgfCBOb25lLCBzb3VyY2VfaGFzaDogc3RyLAopIC0+IGRpY3Q6CiAgICBjaGVja3BvaW50ID0gUGF0aChwYXRoKQogICAgaWYgZXhwZWN0ZWRfc2hhMjU2IGFuZCBzaGEyNTZfZmlsZShjaGVja3BvaW50KSAhPSBleHBlY3RlZF9zaGEyNTY6CiAgICAgICAgcmFpc2UgUnVudGltZUVycm9yKCJSZXN1bWUgU0hBLTI1NiBtaXNtYXRjaCIpCiAgICBidW5kbGUgPSB0b3JjaC5sb2FkKGNoZWNrcG9pbnQsIG1hcF9sb2NhdGlvbj0iY3B1Iiwgd2VpZ2h0c19vbmx5PUZhbHNlKQogICAgaWYgYnVuZGxlLmdldCgicmVzdW1lX3NjaGVtYV92ZXJzaW9uIikgIT0gY29uZmlnLnJlc3VtZV9zY2hlbWFfdmVyc2lvbjoKICAgICAgICByYWlzZSBSdW50aW1lRXJyb3IoIlJlc3VtZSBzY2hlbWEgbWlzbWF0Y2giKQogICAgaWYgcnVuX2lkIGlzIG5vdCBOb25lIGFuZCBidW5kbGUuZ2V0KCJydW5faWQiKSAhPSBydW5faWQ6CiAgICAgICAgcmFpc2UgUnVudGltZUVycm9yKCJSZXN1bWUgcnVuX2lkIG1pc21hdGNoIikKICAgIGlmIGJ1bmRsZS5nZXQoInNvdXJjZV9oYXNoIikgIT0gc291cmNlX2hhc2g6CiAgICAgICAgcmFpc2UgUnVudGltZUVycm9yKCJSZXN1bWUgc291cmNlIGhhc2ggbWlzbWF0Y2giKQogICAgaWYgYnVuZGxlLmdldCgiY29uZmlnX2hhc2giKSAhPSBjb25maWdfaGFzaChjb25maWcpOgogICAgICAgIHJhaXNlIFJ1bnRpbWVFcnJvcigiUmVzdW1lIHNjaWVudGlmaWMvYXJjaGl0ZWN0dXJlIGNvbmZpZyBtaXNtYXRjaCIpCiAgICBtb25pdG9yID0gYnVuZGxlLmdldCgiZWFybHlfc3RvcF9tb25pdG9yX3N0YXRlIikKICAgIGlmIG1vbml0b3IgIT0gewogICAgICAgICJzdGFydF9lcG9jaCI6IGNvbmZpZy5lYXJseV9zdG9wX21vbml0b3Jfc3RhcnRfZXBvY2gsCiAgICAgICAgImFjdGl2ZSI6IGJ1bmRsZVsiY29tcGxldGVkX2Vwb2NoIl0gPj0gY29uZmlnLmVhcmx5X3N0b3BfbW9uaXRvcl9zdGFydF9lcG9jaCwKICAgICAgICAicGF0aWVuY2UiOiBidW5kbGVbImVhcmx5X3N0b3BfY291bnRlciJdLAogICAgfToKICAgICAgICByYWlzZSBSdW50aW1lRXJyb3IoIlJlc3VtZSBlYXJseS1zdG9wIG1vbml0b3Igc3RhdGUgbWlzbWF0Y2giKQogICAgdGVtcGVyYXR1cmUgPSBidW5kbGUuZ2V0KCJ0ZW1wZXJhdHVyZV9zdGF0ZSIpCiAgICBpZiB0ZW1wZXJhdHVyZSBpcyBub3QgTm9uZToKICAgICAgICBzdGF0ZV90YXUgPSBidW5kbGVbIm1vZGVsX3N0YXRlX2RpY3QiXS5nZXQoIm1vdGlmX2NvbXBvc2VyLmN1cnJlbnRfdGF1IikKICAgICAgICBpZiAoCiAgICAgICAgICAgIHRlbXBlcmF0dXJlLmdldCgiZXBvY2giKSAhPSBidW5kbGVbImNvbXBsZXRlZF9lcG9jaCJdCiAgICAgICAgICAgIG9yIHN0YXRlX3RhdSBpcyBOb25lCiAgICAgICAgICAgIG9yIG5vdCB0b3JjaC5pc2Nsb3NlKAogICAgICAgICAgICAgICAgdG9yY2guYXNfdGVuc29yKHN0YXRlX3RhdSkuY3B1KCksCiAgICAgICAgICAgICAgICB0b3JjaC50ZW5zb3IoZmxvYXQodGVtcGVyYXR1cmVbImN1cnJlbnRfdGF1Il0pKSwKICAgICAgICAgICAgICAgIHJ0b2w9MC4wLAogICAgICAgICAgICAgICAgYXRvbD0xZS03LAogICAgICAgICAgICApCiAgICAgICAgKToKICAgICAgICAgICAgcmFpc2UgUnVudGltZUVycm9yKCJSZXN1bWUgc2NoZWR1bGVkIHRlbXBlcmF0dXJlIHN0YXRlIG1pc21hdGNoIikKICAgIHJldHVybiBidW5kbGUKCgpkZWYgcmVzdG9yZV90cmFpbmluZ19zdGF0ZSgKICAgIGJ1bmRsZTogZGljdCwgKiwgbW9kZWw6IHRvcmNoLm5uLk1vZHVsZSwgZW1hOiBBbnksCiAgICBvcHRpbWl6ZXI6IHRvcmNoLm9wdGltLk9wdGltaXplciwgc2NoZWR1bGVyOiBBbnksIHNjYWxlcjogQW55LAogICAgbG9hZGVyX2dlbmVyYXRvcjogdG9yY2guR2VuZXJhdG9yLAogICAgc2FtcGxlcjogQW55IHwgTm9uZSA9IE5vbmUsCiAgICBkYXRhc2V0OiBBbnkgfCBOb25lID0gTm9uZSwKKSAtPiBOb25lOgogICAgbW9kZWwubG9hZF9zdGF0ZV9kaWN0KGJ1bmRsZVsibW9kZWxfc3RhdGVfZGljdCJdLCBzdHJpY3Q9VHJ1ZSkKICAgIGVtYS5sb2FkX3N0YXRlX2RpY3QoYnVuZGxlWyJlbWFfc3RhdGVfZGljdCJdKQogICAgb3B0aW1pemVyLmxvYWRfc3RhdGVfZGljdChidW5kbGVbIm9wdGltaXplcl9zdGF0ZV9kaWN0Il0pCiAgICBzY2hlZHVsZXIubG9hZF9zdGF0ZV9kaWN0KGJ1bmRsZVsic2NoZWR1bGVyX3N0YXRlX2RpY3QiXSkKICAgIGlmIHNjYWxlciBpcyBub3QgTm9uZSBhbmQgYnVuZGxlWyJncmFkX3NjYWxlcl9zdGF0ZV9kaWN0Il06CiAgICAgICAgc2NhbGVyLmxvYWRfc3RhdGVfZGljdChidW5kbGVbImdyYWRfc2NhbGVyX3N0YXRlX2RpY3QiXSkKICAgIHJlc3RvcmVfcm5nX3N0YXRlKGJ1bmRsZVsicm5nX3N0YXRlIl0sIGxvYWRlcl9nZW5lcmF0b3IpCiAgICBpZiBzYW1wbGVyIGlzIG5vdCBOb25lIGFuZCBidW5kbGUuZ2V0KCJzYW1wbGVyX3N0YXRlIikgaXMgbm90IE5vbmU6CiAgICAgICAgc2FtcGxlci5sb2FkX3N0YXRlX2RpY3QoYnVuZGxlWyJzYW1wbGVyX3N0YXRlIl0pCiAgICBpZiBkYXRhc2V0IGlzIG5vdCBOb25lIGFuZCBidW5kbGUuZ2V0KCJhdWdtZW50YXRpb25fc3RhdGUiKSBpcyBub3QgTm9uZToKICAgICAgICBkYXRhc2V0LmxvYWRfc3RhdGVfZGljdChidW5kbGVbImF1Z21lbnRhdGlvbl9zdGF0ZSJdKQo=', 'mpg_fer_v2_3/config.py': 'IiIiQ29uZmlndXJhdGlvbiBjb250cmFjdCBmb3IgTVBHLUZFUiB2Mi4zIChJc3N1ZSAjOTcpLiIiIgoKZnJvbSBfX2Z1dHVyZV9fIGltcG9ydCBhbm5vdGF0aW9ucwoKZnJvbSBkYXRhY2xhc3NlcyBpbXBvcnQgZGF0YWNsYXNzLCBmaWVsZAoKCkBkYXRhY2xhc3MKY2xhc3MgTVBHQ29uZmlnOgogICAgc2VlZDogaW50ID0gNDIKICAgIGRldmljZTogc3RyID0gImN1ZGEiCgogICAgaW1nX3NpemU6IGludCA9IDQ4CiAgICBudW1fcGl4ZWxzOiBpbnQgPSAyMzA0CiAgICBudW1fbmVpZ2hib3JzOiBpbnQgPSA4CiAgICByYXdfcGl4ZWxfZGltOiBpbnQgPSAzMgogICAgZF9waXhlbDogaW50ID0gOTYKICAgIG51bV9waXhlbF9nbm5fbGF5ZXJzOiBpbnQgPSA0CiAgICBudW1fcGl4ZWxfaGVhZHM6IGludCA9IDQKICAgIHBpeGVsX2Ryb3BvdXQ6IGZsb2F0ID0gMC4xMAogICAgcGl4ZWxfZHJvcF9wYXRoX21heDogZmxvYXQgPSAwLjAzCiAgICBwaXhlbF9lZGdlX2RpbTogaW50ID0gNQogICAgZF9waXhlbF9yZWFkb3V0OiBpbnQgPSAxMjgKCiAgICBudW1fbW90aWZzOiBpbnQgPSA0OAogICAgbW90aWZfd2luZG93X3NpemVzOiB0dXBsZVtpbnQsIGludCwgaW50XSA9ICg4LCAxMiwgMTYpCiAgICBtb3RpZl93aW5kb3dfc2l6ZTogaW50ID0gMTIKICAgIG1vdGlmX3N0cmlkZTogaW50ID0gNgogICAgbnVtX29jY3VycmVuY2VzOiBpbnQgPSA0OQogICAgZF93aGF0OiBpbnQgPSA5NgogICAgZF90eXBlOiBpbnQgPSAzMgogICAgZF93aGVyZTogaW50ID0gNQogICAgZF9tb3RpZl9vY2N1cnJlbmNlX3JhdzogaW50ID0gMTMzCiAgICBkX21vdGlmOiBpbnQgPSAxOTIKICAgIHRhdV9zdGFydDogZmxvYXQgPSAwLjcwCiAgICB0YXVfZmluYWw6IGZsb2F0ID0gMC4zMAogICAgdGF1X2FubmVhbF9lbmRfZXBvY2g6IGludCA9IDM1CgogICAgbnVtX21vdGlmX2xheWVyczogaW50ID0gNQogICAgbnVtX21vdGlmX2hlYWRzOiBpbnQgPSA2CiAgICBtb3RpZl9nZW9tX2RpbTogaW50ID0gNgogICAgbW90aWZfZHJvcG91dDogZmxvYXQgPSAwLjEwCiAgICBtb3RpZl9kcm9wX3BhdGhfbWF4OiBmbG9hdCA9IDAuMDUKICAgIGRfbW90aWZfcmVhZG91dDogaW50ID0gMzg0CiAgICAjIER5bmFtaWMgc3BhcnNlIG1vdGlmIHJvdXRpbmcgc2NoZWR1bGU6IEsgZWRnZXMgcGVyIHF1ZXJ5IG5vZGUgZm9yIGxheWVycyAxLi41CiAgICBtb3RpZl90b3BrX3NjaGVkdWxlOiB0dXBsZVtpbnQsIC4uLl0gPSAoOCwgMTYsIDE2LCAxNiwgMjQpCiAgICAjIElzc3VlICM5NyBzaW5nbGUgZGVsdGE6IHByZXNlcnZlIGVhcmx5IHJlcHJlc2VudGF0aW9ucyB3aXRoIGhhbGYtc3RlcAogICAgIyByZXNpZHVhbCB1cGRhdGVzIGluIEwxLUwyOyBhbGwgbGF0ZXIgbW90aWYgYmxvY2tzIHJlbWFpbiB2Mi4yLWVxdWl2YWxlbnQuCiAgICBtb3RpZl9yZXNpZHVhbF9zY2FsZV9zY2hlZHVsZTogdHVwbGVbZmxvYXQsIC4uLl0gPSAoMC41LCAwLjUsIDEuMCwgMS4wLCAxLjApCgogICAgZF9jbGFzc2lmaWVyX2luOiBpbnQgPSA1MTIKICAgIGRfY2xhc3NpZmllcl9oaWRkZW46IGludCA9IDI1NgogICAgbnVtX2NsYXNzZXM6IGludCA9IDcKICAgIGNsYXNzaWZpZXJfZHJvcG91dDogZmxvYXQgPSAwLjI1CiAgICBzdXBjb25fZGltOiBpbnQgPSAxMjgKICAgIHN1cGNvbl90ZW1wZXJhdHVyZTogZmxvYXQgPSAwLjEwCgogICAgYXV4X3BpeGVsX3dlaWdodDogZmxvYXQgPSAwLjA1CiAgICBhdXhfbW90aWZfd2VpZ2h0OiBmbG9hdCA9IDAuMjAKICAgIGxhbWJkYV9kaXY6IGZsb2F0ID0gMC4wMQogICAgbGFtYmRhX21pOiBmbG9hdCA9IDAuMDI1CiAgICBtaV9iZXRhOiBmbG9hdCA9IDEuMAogICAgY29uc2lzdGVuY3lfcHJvYmFiaWxpdHk6IGZsb2F0ID0gMC41MAogICAgbGFtYmRhX2NvbnNpc3RlbmN5OiBmbG9hdCA9IDAuMTUKICAgIGxhbWJkYV9zdXBjb246IGZsb2F0ID0gMC4wNQoKICAgIGVtYV9kZWNheTogZmxvYXQgPSAwLjk5OQogICAgYmF0Y2hfc2l6ZTogaW50ID0gMTYKICAgIGdyYWRpZW50X2FjY3VtdWxhdGlvbl9zdGVwczogaW50ID0gMgogICAgbGVhcm5pbmdfcmF0ZTogZmxvYXQgPSAzZS00CiAgICB3ZWlnaHRfZGVjYXk6IGZsb2F0ID0gNWUtNAogICAgbWF4X2Vwb2NoczogaW50ID0gMTIwCiAgICBtaW5fZXBvY2hzOiBpbnQgPSA1MAogICAgd2FybXVwX2Vwb2NoczogaW50ID0gNQogICAgbHJfZGVjYXlfZW5kX2Vwb2NoOiBpbnQgPSA4NQogICAgbWluX2xlYXJuaW5nX3JhdGU6IGZsb2F0ID0gMWUtNgogICAgZWFybHlfc3RvcF9tb25pdG9yX3N0YXJ0X2Vwb2NoOiBpbnQgPSA4NQogICAgZWFybHlfc3RvcF9wYXRpZW5jZTogaW50ID0gMTUKICAgIGdyYWRfY2xpcDogZmxvYXQgPSAxLjAKICAgIGxhYmVsX3Ntb290aGluZzogZmxvYXQgPSAwLjA1CiAgICB1c2VfYW1wOiBib29sID0gVHJ1ZQogICAgbnVtX3dvcmtlcnM6IGludCA9IDIKCiAgICByZXN1bWVfc2NoZW1hX3ZlcnNpb246IGludCA9IDMKICAgIHJlc3VtZV9zbmFwc2hvdF9pbnRlcnZhbDogaW50ID0gMTAKICAgIHJlc3VtZV9zbmFwc2hvdHNfdG9fa2VlcDogaW50ID0gMgogICAgc2VnbWVudF9zb2Z0X2xpbWl0X2hvdXJzOiBmbG9hdCA9IDEwLjUKICAgIHNlZ21lbnRfc2FmZXR5X21hcmdpbl9taW51dGVzOiBmbG9hdCA9IDE1LjAKICAgIHNlZ21lbnRfbnVtYmVyOiBpbnQgPSAxCiAgICBydW5faWQ6IHN0ciB8IE5vbmUgPSBOb25lCiAgICBvdXRwdXRfZGlyOiBzdHIgfCBOb25lID0gTm9uZQogICAgcmVzdW1lX3BhdGg6IHN0ciB8IE5vbmUgPSBOb25lCgogICAgbWljcm9fb3ZlcmZpdF9zYW1wbGVzOiBpbnQgPSAxNgogICAgbWljcm9fb3ZlcmZpdF90YXJnZXQ6IGZsb2F0ID0gMC44NzUKICAgIG1pY3JvX292ZXJmaXRfbWF4X3N0ZXBzOiBpbnQgPSA4MAogICAgbWljcm9fb3ZlcmZpdF9sZWFybmluZ19yYXRlOiBmbG9hdCA9IDFlLTMKCiAgICBydW50aW1lX3NhZmVfcmVzdW1lX2ZpZWxkczogdHVwbGVbc3RyLCAuLi5dID0gZmllbGQoCiAgICAgICAgZGVmYXVsdD0oCiAgICAgICAgICAgICJudW1fd29ya2VycyIsICJvdXRwdXRfZGlyIiwgInJlc3VtZV9wYXRoIiwgInNlZ21lbnRfbnVtYmVyIiwKICAgICAgICAgICAgInNlZ21lbnRfc29mdF9saW1pdF9ob3VycyIsICJzZWdtZW50X3NhZmV0eV9tYXJnaW5fbWludXRlcyIsICJydW5faWQiLAogICAgICAgICksCiAgICAgICAgcmVwcj1GYWxzZSwKICAgICkKCiAgICBkZWYgX19wb3N0X2luaXRfXyhzZWxmKSAtPiBOb25lOgogICAgICAgIGlmIGxlbihzZWxmLm1vdGlmX3RvcGtfc2NoZWR1bGUpICE9IHNlbGYubnVtX21vdGlmX2xheWVyczoKICAgICAgICAgICAgcmFpc2UgVmFsdWVFcnJvcigKICAgICAgICAgICAgICAgICJtb3RpZl90b3BrX3NjaGVkdWxlIGxlbmd0aCBtdXN0IGVxdWFsIG51bV9tb3RpZl9sYXllcnMiCiAgICAgICAgICAgICkKICAgICAgICBtYXhpbXVtID0gc2VsZi5udW1fb2NjdXJyZW5jZXMgLSAxCiAgICAgICAgaWYgYW55KAogICAgICAgICAgICBpc2luc3RhbmNlKHZhbHVlLCBib29sKQogICAgICAgICAgICBvciBub3QgaXNpbnN0YW5jZSh2YWx1ZSwgaW50KQogICAgICAgICAgICBvciBub3QgMSA8PSB2YWx1ZSA8PSBtYXhpbXVtCiAgICAgICAgICAgIGZvciB2YWx1ZSBpbiBzZWxmLm1vdGlmX3RvcGtfc2NoZWR1bGUKICAgICAgICApOgogICAgICAgICAgICByYWlzZSBWYWx1ZUVycm9yKAogICAgICAgICAgICAgICAgZiJtb3RpZl90b3BrX3NjaGVkdWxlIHZhbHVlcyBtdXN0IGJlIGludGVnZXJzIGluIFsxLCB7bWF4aW11bX1dIgogICAgICAgICAgICApCiAgICAgICAgaWYgbGVuKHNlbGYubW90aWZfcmVzaWR1YWxfc2NhbGVfc2NoZWR1bGUpICE9IHNlbGYubnVtX21vdGlmX2xheWVyczoKICAgICAgICAgICAgcmFpc2UgVmFsdWVFcnJvcigKICAgICAgICAgICAgICAgICJtb3RpZl9yZXNpZHVhbF9zY2FsZV9zY2hlZHVsZSBsZW5ndGggbXVzdCBlcXVhbCBudW1fbW90aWZfbGF5ZXJzIgogICAgICAgICAgICApCiAgICAgICAgaWYgYW55KAogICAgICAgICAgICBpc2luc3RhbmNlKHZhbHVlLCBib29sKQogICAgICAgICAgICBvciBub3QgaXNpbnN0YW5jZSh2YWx1ZSwgKGludCwgZmxvYXQpKQogICAgICAgICAgICBvciBub3QgMC4wIDwgZmxvYXQodmFsdWUpIDw9IDEuMAogICAgICAgICAgICBmb3IgdmFsdWUgaW4gc2VsZi5tb3RpZl9yZXNpZHVhbF9zY2FsZV9zY2hlZHVsZQogICAgICAgICk6CiAgICAgICAgICAgIHJhaXNlIFZhbHVlRXJyb3IoCiAgICAgICAgICAgICAgICAibW90aWZfcmVzaWR1YWxfc2NhbGVfc2NoZWR1bGUgdmFsdWVzIG11c3QgYmUgbnVtZXJpYyBpbiAoMCwgMV0iCiAgICAgICAgICAgICkK', 'mpg_fer_v2_3/data.py': 'IiIiRkVSMjAxMyBsb2FkaW5nIHdpdGggc3BsaXQgaXNvbGF0aW9uIGFuZCByZXN1bWUtc3RhYmxlIGF1Z21lbnRhdGlvbi4iIiIKCmZyb20gX19mdXR1cmVfXyBpbXBvcnQgYW5ub3RhdGlvbnMKCmltcG9ydCBjc3YKZnJvbSBwYXRobGliIGltcG9ydCBQYXRoCmZyb20gdHlwaW5nIGltcG9ydCBUdXBsZQoKaW1wb3J0IG51bXB5IGFzIG5wCmltcG9ydCB0b3JjaApmcm9tIHRvcmNoLnV0aWxzLmRhdGEgaW1wb3J0IERhdGFMb2FkZXIsIERhdGFzZXQsIFNhbXBsZXIKaW1wb3J0IHRvcmNodmlzaW9uLnRyYW5zZm9ybXMuZnVuY3Rpb25hbCBhcyBURgoKCkVYUEVDVEVEX1RSQUlOX1JPV1MgPSAyODcwOQpFWFBFQ1RFRF9QVUJMSUNfUk9XUyA9IDM1ODkKRVhQRUNURURfUFJJVkFURV9ST1dTID0gMzU4OQpST0xFX1RPX0JBU0VOQU1FID0gewogICAgInRyYWluIjogInRyYWluLmNzdiIsCiAgICAidmFsIjogInZhbC5jc3YiLAogICAgInRlc3QiOiAidGVzdC5jc3YiLAp9ClJPTEVfVE9fRVhQRUNURURfUk9XUyA9IHsKICAgICJ0cmFpbiI6IEVYUEVDVEVEX1RSQUlOX1JPV1MsCiAgICAidmFsIjogRVhQRUNURURfUFVCTElDX1JPV1MsCiAgICAidGVzdCI6IEVYUEVDVEVEX1BSSVZBVEVfUk9XUywKfQoKCmRlZiBhdWdtZW50YXRpb25fc2VlZChiYXNlX3NlZWQ6IGludCwgZXBvY2g6IGludCwgc2FtcGxlX2luZGV4OiBpbnQpIC0+IGludDoKICAgICIiIlN0YWJsZSBzYW1wbGUtbG9jYWwgc2VlZCwgaW5kZXBlbmRlbnQgb2Ygd29ya2VyIHByb2Nlc3MgUk5HIGhpc3RvcnkuIiIiCiAgICByZXR1cm4gaW50KGJhc2Vfc2VlZCkgKyBpbnQoZXBvY2gpICogMV8wMDBfMDAzICsgaW50KHNhbXBsZV9pbmRleCkgKiA5N180MDkKCgpjbGFzcyBFcG9jaFJhbmRvbVNhbXBsZXIoU2FtcGxlcltpbnRdKToKICAgICIiIkRlcml2ZSBlYWNoIGVwb2NoJ3MgcGVybXV0YXRpb24gc29sZWx5IGZyb20gYmFzZSBzZWVkIGFuZCBlcG9jaC4iIiIKCiAgICBkZWYgX19pbml0X18oc2VsZiwgZGF0YV9zb3VyY2U6IERhdGFzZXQsIHNlZWQ6IGludCA9IDQyKSAtPiBOb25lOgogICAgICAgIHNlbGYuZGF0YV9zb3VyY2UgPSBkYXRhX3NvdXJjZQogICAgICAgIHNlbGYuc2VlZCA9IGludChzZWVkKQogICAgICAgIHNlbGYuZXBvY2ggPSAwCgogICAgZGVmIHNldF9lcG9jaChzZWxmLCBlcG9jaDogaW50KSAtPiBOb25lOgogICAgICAgIHNlbGYuZXBvY2ggPSBpbnQoZXBvY2gpCgogICAgZGVmIF9faXRlcl9fKHNlbGYpOgogICAgICAgIGdlbmVyYXRvciA9IHRvcmNoLkdlbmVyYXRvcigpLm1hbnVhbF9zZWVkKAogICAgICAgICAgICBzZWxmLnNlZWQgKyBzZWxmLmVwb2NoICogMl8wMDBfMDMzCiAgICAgICAgKQogICAgICAgIHJldHVybiBpdGVyKHRvcmNoLnJhbmRwZXJtKGxlbihzZWxmLmRhdGFfc291cmNlKSwgZ2VuZXJhdG9yPWdlbmVyYXRvcikudG9saXN0KCkpCgogICAgZGVmIF9fbGVuX18oc2VsZikgLT4gaW50OgogICAgICAgIHJldHVybiBsZW4oc2VsZi5kYXRhX3NvdXJjZSkKCiAgICBkZWYgc3RhdGVfZGljdChzZWxmKSAtPiBkaWN0OgogICAgICAgIHJldHVybiB7InNjaGVtZSI6ICJiYXNlX3NlZWRfZXBvY2hfdjEiLCAic2VlZCI6IHNlbGYuc2VlZCwgImVwb2NoIjogc2VsZi5lcG9jaH0KCiAgICBkZWYgbG9hZF9zdGF0ZV9kaWN0KHNlbGYsIHN0YXRlOiBkaWN0KSAtPiBOb25lOgogICAgICAgIGlmIHN0YXRlLmdldCgic2NoZW1lIikgIT0gImJhc2Vfc2VlZF9lcG9jaF92MSIgb3IgaW50KHN0YXRlWyJzZWVkIl0pICE9IHNlbGYuc2VlZDoKICAgICAgICAgICAgcmFpc2UgUnVudGltZUVycm9yKCJUcmFpbmluZyBzYW1wbGVyIHN0YXRlIGlzIGluY29tcGF0aWJsZSIpCiAgICAgICAgc2VsZi5lcG9jaCA9IGludChzdGF0ZVsiZXBvY2giXSkKCgpkZWYgdmFsaWRhdGVfc3BsaXRfcGF0aChjc3ZfcGF0aDogc3RyIHwgUGF0aCwgcm9sZTogc3RyKSAtPiBQYXRoOgogICAgIiIiVmFsaWRhdGUgYSBzcGxpdCBieSBleHBsaWNpdCByb2xlIGFuZCBiYXNlbmFtZSwgbmV2ZXIgYnkgcm93IGNvdW50IGFsb25lLiIiIgogICAgbm9ybWFsaXplZF9yb2xlID0gcm9sZS5sb3dlcigpCiAgICBpZiBub3JtYWxpemVkX3JvbGUgbm90IGluIFJPTEVfVE9fQkFTRU5BTUU6CiAgICAgICAgcmFpc2UgVmFsdWVFcnJvcihmIlVua25vd24gRkVSMjAxMyBzcGxpdCByb2xlOiB7cm9sZSFyfSIpCiAgICBwYXRoID0gUGF0aChjc3ZfcGF0aCkKICAgIGV4cGVjdGVkID0gUk9MRV9UT19CQVNFTkFNRVtub3JtYWxpemVkX3JvbGVdCiAgICBpZiBwYXRoLm5hbWUubG93ZXIoKSAhPSBleHBlY3RlZDoKICAgICAgICByYWlzZSBWYWx1ZUVycm9yKAogICAgICAgICAgICBmIlNwbGl0IHJvbGUge25vcm1hbGl6ZWRfcm9sZSFyfSByZXF1aXJlcyBiYXNlbmFtZSB7ZXhwZWN0ZWQhcn07ICIKICAgICAgICAgICAgZiJyZWZ1c2luZyBzdXNwaWNpb3VzIHBhdGgge3BhdGh9IgogICAgICAgICkKICAgIGlmIG5vdCBwYXRoLmlzX2ZpbGUoKToKICAgICAgICByYWlzZSBGaWxlTm90Rm91bmRFcnJvcihmIkZFUjIwMTMge25vcm1hbGl6ZWRfcm9sZX0gQ1NWIG5vdCBmb3VuZDoge3BhdGh9IikKICAgIHJldHVybiBwYXRoCgoKZGVmIHZhbGlkYXRlX3NwbGl0X3BhdGhzKAogICAgdHJhaW5fY3N2OiBzdHIgfCBQYXRoLAogICAgdmFsX2Nzdjogc3RyIHwgUGF0aCwKICAgIHRlc3RfY3N2OiBzdHIgfCBQYXRoIHwgTm9uZSA9IE5vbmUsCikgLT4gdHVwbGVbUGF0aCwgUGF0aCwgUGF0aCB8IE5vbmVdOgogICAgIiIiRmFpbCBjbG9zZWQgb24gcm9sZSBzd2FwcywgYWxpYXNlcywgYW5kIGR1cGxpY2F0ZSBzcGxpdCBwYXRocy4iIiIKICAgIHRyYWluX3BhdGggPSB2YWxpZGF0ZV9zcGxpdF9wYXRoKHRyYWluX2NzdiwgInRyYWluIikKICAgIHZhbF9wYXRoID0gdmFsaWRhdGVfc3BsaXRfcGF0aCh2YWxfY3N2LCAidmFsIikKICAgIHRlc3RfcGF0aCA9IHZhbGlkYXRlX3NwbGl0X3BhdGgodGVzdF9jc3YsICJ0ZXN0IikgaWYgdGVzdF9jc3YgaXMgbm90IE5vbmUgZWxzZSBOb25lCiAgICByZXNvbHZlZCA9IFtwLnJlc29sdmUoKSBmb3IgcCBpbiAodHJhaW5fcGF0aCwgdmFsX3BhdGgsIHRlc3RfcGF0aCkgaWYgcCBpcyBub3QgTm9uZV0KICAgIGlmIGxlbihzZXQocmVzb2x2ZWQpKSAhPSBsZW4ocmVzb2x2ZWQpOgogICAgICAgIHJhaXNlIFZhbHVlRXJyb3IoIlRyYWluLCBQdWJsaWNUZXN0LCBhbmQgUHJpdmF0ZVRlc3QgcGF0aHMgbXVzdCBiZSBkaXN0aW5jdCIpCiAgICByZXR1cm4gdHJhaW5fcGF0aCwgdmFsX3BhdGgsIHRlc3RfcGF0aAoKCmRlZiBpbnNwZWN0X3NwbGl0X2ZpbGUoCiAgICBjc3ZfcGF0aDogc3RyIHwgUGF0aCwKICAgIHJvbGU6IHN0ciwKICAgIHZhbGlkYXRlX2NvbnRlbnQ6IGJvb2wsCikgLT4gZGljdDoKICAgICIiIlZhbGlkYXRlIHNwbGl0IG1ldGFkYXRhIGFuZCwgd2hlbiBhdXRob3JpemVkLCBldmVyeSBsYWJlbGVkIGltYWdlIHJvdy4iIiIKICAgIHBhdGggPSB2YWxpZGF0ZV9zcGxpdF9wYXRoKGNzdl9wYXRoLCByb2xlKQogICAgbm9ybWFsaXplZF9yb2xlID0gcm9sZS5sb3dlcigpCiAgICByb3dzID0gMAogICAgd2l0aCBwYXRoLm9wZW4oInIiLCBlbmNvZGluZz0idXRmLTgiLCBuZXdsaW5lPSIiKSBhcyBoYW5kbGU6CiAgICAgICAgcmVhZGVyID0gY3N2LnJlYWRlcihoYW5kbGUpCiAgICAgICAgdHJ5OgogICAgICAgICAgICBoZWFkZXIgPSBbY29sdW1uLnN0cmlwKCkubG93ZXIoKSBmb3IgY29sdW1uIGluIG5leHQocmVhZGVyKV0KICAgICAgICBleGNlcHQgU3RvcEl0ZXJhdGlvbiBhcyBleGM6CiAgICAgICAgICAgIHJhaXNlIFZhbHVlRXJyb3IoZiJFbXB0eSBGRVIyMDEzIENTVjoge3BhdGh9IikgZnJvbSBleGMKICAgICAgICBpZiAicGl4ZWxzIiBub3QgaW4gaGVhZGVyIG9yICJlbW90aW9uIiBub3QgaW4gaGVhZGVyOgogICAgICAgICAgICByYWlzZSBWYWx1ZUVycm9yKGYiQ1NWIHJlcXVpcmVzIGVtb3Rpb24gYW5kIHBpeGVscyBjb2x1bW5zOiB7cGF0aH0iKQogICAgICAgIHBpeGVsX2luZGV4ID0gaGVhZGVyLmluZGV4KCJwaXhlbHMiKQogICAgICAgIGVtb3Rpb25faW5kZXggPSBoZWFkZXIuaW5kZXgoImVtb3Rpb24iKQogICAgICAgIGZvciByb3cgaW4gcmVhZGVyOgogICAgICAgICAgICBpZiBub3Qgcm93OgogICAgICAgICAgICAgICAgY29udGludWUKICAgICAgICAgICAgcm93cyArPSAxCiAgICAgICAgICAgIGlmIHZhbGlkYXRlX2NvbnRlbnQ6CiAgICAgICAgICAgICAgICBwaXhlbHMgPSBucC5mcm9tc3RyaW5nKHJvd1twaXhlbF9pbmRleF0sIHNlcD0iICIsIGR0eXBlPW5wLmludDE2KQogICAgICAgICAgICAgICAgaWYgbGVuKHBpeGVscykgIT0gMjMwNDoKICAgICAgICAgICAgICAgICAgICByYWlzZSBWYWx1ZUVycm9yKAogICAgICAgICAgICAgICAgICAgICAgICBmIk1hbGZvcm1lZCBwaXhlbCByb3cgd2l0aCB7bGVuKHBpeGVscyl9IHBpeGVscyBpbiB7cGF0aH0iCiAgICAgICAgICAgICAgICAgICAgKQogICAgICAgICAgICAgICAgaWYgbnAuYW55KChwaXhlbHMgPCAwKSB8IChwaXhlbHMgPiAyNTUpKToKICAgICAgICAgICAgICAgICAgICByYWlzZSBWYWx1ZUVycm9yKGYiUGl4ZWwgb3V0c2lkZSBbMCwgMjU1XSBpbiB7cGF0aH0iKQogICAgICAgICAgICAgICAgbGFiZWwgPSBpbnQocm93W2Vtb3Rpb25faW5kZXhdKQogICAgICAgICAgICAgICAgaWYgbm90IDAgPD0gbGFiZWwgPD0gNjoKICAgICAgICAgICAgICAgICAgICByYWlzZSBWYWx1ZUVycm9yKGYiSW52YWxpZCBsYWJlbCB7bGFiZWx9IGluIHtwYXRofSIpCiAgICBleHBlY3RlZF9yb3dzID0gUk9MRV9UT19FWFBFQ1RFRF9ST1dTW25vcm1hbGl6ZWRfcm9sZV0KICAgIGlmIHJvd3MgIT0gZXhwZWN0ZWRfcm93czoKICAgICAgICByYWlzZSBWYWx1ZUVycm9yKAogICAgICAgICAgICBmIntub3JtYWxpemVkX3JvbGV9IHJlcXVpcmVzIGV4YWN0bHkge2V4cGVjdGVkX3Jvd3N9IHJvd3MsIGdvdCB7cm93c30iCiAgICAgICAgKQogICAgcmV0dXJuIHsKICAgICAgICAicm9sZSI6IG5vcm1hbGl6ZWRfcm9sZSwKICAgICAgICAicGF0aCI6IHN0cihwYXRoLnJlc29sdmUoKSksCiAgICAgICAgInJvd3MiOiByb3dzLAogICAgICAgICJleHBlY3RlZF9yb3dzIjogZXhwZWN0ZWRfcm93cywKICAgICAgICAiY29udGVudF92YWxpZGF0ZWQiOiB2YWxpZGF0ZV9jb250ZW50LAogICAgfQoKCmRlZiB2YWxpZGF0ZV9kYXRhc2V0X2dhdGUoCiAgICB0cmFpbl9jc3Y6IHN0ciB8IFBhdGgsCiAgICB2YWxfY3N2OiBzdHIgfCBQYXRoLAogICAgdGVzdF9jc3Y6IHN0ciB8IFBhdGgsCikgLT4gZGljdFtzdHIsIGRpY3RdOgogICAgIiIiUGhhc2UtQSBnYXRlOiBpbnNwZWN0IFRyYWluL1B1YmxpYyBjb250ZW50LCBQcml2YXRlIG1ldGFkYXRhIG9ubHkuIiIiCiAgICB0cmFpbl9wYXRoLCB2YWxfcGF0aCwgdGVzdF9wYXRoID0gdmFsaWRhdGVfc3BsaXRfcGF0aHMoCiAgICAgICAgdHJhaW5fY3N2LCB2YWxfY3N2LCB0ZXN0X2NzdgogICAgKQogICAgYXNzZXJ0IHRlc3RfcGF0aCBpcyBub3QgTm9uZQogICAgcmV0dXJuIHsKICAgICAgICAidHJhaW4iOiBpbnNwZWN0X3NwbGl0X2ZpbGUodHJhaW5fcGF0aCwgInRyYWluIiwgdmFsaWRhdGVfY29udGVudD1UcnVlKSwKICAgICAgICAicHVibGljIjogaW5zcGVjdF9zcGxpdF9maWxlKHZhbF9wYXRoLCAidmFsIiwgdmFsaWRhdGVfY29udGVudD1UcnVlKSwKICAgICAgICAjIFByaXZhdGVUZXN0IGNvbnRlbnQgc3RheXMgbG9ja2VkIHVudGlsIHRoZSBmcm96ZW4tY2hlY2twb2ludCBib3VuZGFyeS4KICAgICAgICAicHJpdmF0ZSI6IGluc3BlY3Rfc3BsaXRfZmlsZSh0ZXN0X3BhdGgsICJ0ZXN0IiwgdmFsaWRhdGVfY29udGVudD1GYWxzZSksCiAgICB9CgoKY2xhc3MgRkVSMjAxM0RhdGFzZXQoRGF0YXNldCk6CiAgICAiIiJGRVIyMDEzIENTViBEYXRhc2V0IHJldHVybmluZyBub3JtYWxpemVkIFsxLCA0OCwgNDhdIGltYWdlcyBhbmQgbGFiZWxzLiIiIgoKICAgIGRlZiBfX2luaXRfXygKICAgICAgICBzZWxmLAogICAgICAgIGNzdl9wYXRoOiBzdHIgfCBQYXRoLAogICAgICAgIHNwbGl0OiBzdHIgPSAidHJhaW4iLAogICAgICAgIGF1Z21lbnQ6IGJvb2wgPSBGYWxzZSwKICAgICAgICBzZWVkOiBpbnQgPSA0MiwKICAgICkgLT4gTm9uZToKICAgICAgICBzdXBlcigpLl9faW5pdF9fKCkKICAgICAgICBhbGlhc2VzID0geyJwdWJsaWMiOiAidmFsIiwgInByaXZhdGUiOiAidGVzdCJ9CiAgICAgICAgc2VsZi5zcGxpdCA9IGFsaWFzZXMuZ2V0KHNwbGl0Lmxvd2VyKCksIHNwbGl0Lmxvd2VyKCkpCiAgICAgICAgaWYgc2VsZi5zcGxpdCBub3QgaW4gUk9MRV9UT19CQVNFTkFNRToKICAgICAgICAgICAgcmFpc2UgVmFsdWVFcnJvcihmIlVua25vd24gRkVSMjAxMyBzcGxpdCByb2xlOiB7c3BsaXQhcn0iKQogICAgICAgIHNlbGYuYXVnbWVudCA9IGF1Z21lbnQgYW5kIChzZWxmLnNwbGl0ID09ICJ0cmFpbiIpCiAgICAgICAgc2VsZi5zZWVkID0gaW50KHNlZWQpCiAgICAgICAgc2VsZi5lcG9jaCA9IDAKCiAgICAgICAgcCA9IHZhbGlkYXRlX3NwbGl0X3BhdGgoY3N2X3BhdGgsIHNlbGYuc3BsaXQpCgogICAgICAgIGltYWdlc19saXN0ID0gW10KICAgICAgICBsYWJlbHNfbGlzdCA9IFtdCgogICAgICAgIHdpdGggb3BlbihwLCAiciIsIGVuY29kaW5nPSJ1dGYtOCIsIG5ld2xpbmU9IiIpIGFzIGY6CiAgICAgICAgICAgIHJlYWRlciA9IGNzdi5yZWFkZXIoZikKICAgICAgICAgICAgaGVhZGVyID0gW2Muc3RyaXAoKS5sb3dlcigpIGZvciBjIGluIG5leHQocmVhZGVyKV0KICAgICAgICAgICAgaWYgInBpeGVscyIgbm90IGluIGhlYWRlcjoKICAgICAgICAgICAgICAgIHJhaXNlIFZhbHVlRXJyb3IoZiJDU1YgbWlzc2luZyAncGl4ZWxzJyBjb2x1bW46IHtwfSIpCiAgICAgICAgICAgIHBpeF9pZHggPSBoZWFkZXIuaW5kZXgoInBpeGVscyIpCiAgICAgICAgICAgIGlmICJlbW90aW9uIiBub3QgaW4gaGVhZGVyOgogICAgICAgICAgICAgICAgcmFpc2UgVmFsdWVFcnJvcihmIkNTViBtaXNzaW5nICdlbW90aW9uJyBjb2x1bW46IHtwfSIpCiAgICAgICAgICAgIGVtb19pZHggPSBoZWFkZXIuaW5kZXgoImVtb3Rpb24iKQoKICAgICAgICAgICAgZm9yIHJvdyBpbiByZWFkZXI6CiAgICAgICAgICAgICAgICBpZiBub3Qgcm93OgogICAgICAgICAgICAgICAgICAgIGNvbnRpbnVlCiAgICAgICAgICAgICAgICBwaXggPSBucC5mcm9tc3RyaW5nKHJvd1twaXhfaWR4XSwgc2VwPSIgIiwgZHR5cGU9bnAuaW50MTYpCiAgICAgICAgICAgICAgICBpZiBsZW4ocGl4KSAhPSAyMzA0OgogICAgICAgICAgICAgICAgICAgIHJhaXNlIFZhbHVlRXJyb3IoZiJNYWxmb3JtZWQgcGl4ZWwgcm93IHdpdGgge2xlbihwaXgpfSBwaXhlbHMgaW4ge3B9IikKICAgICAgICAgICAgICAgIGlmIG5wLmFueSgocGl4IDwgMCkgfCAocGl4ID4gMjU1KSk6CiAgICAgICAgICAgICAgICAgICAgcmFpc2UgVmFsdWVFcnJvcihmIlBpeGVsIG91dHNpZGUgWzAsIDI1NV0gaW4ge3B9IikKICAgICAgICAgICAgICAgIGltYWdlc19saXN0LmFwcGVuZChwaXguYXN0eXBlKG5wLnVpbnQ4KS5yZXNoYXBlKDQ4LCA0OCkpCgogICAgICAgICAgICAgICAgbGFiZWwgPSBpbnQocm93W2Vtb19pZHhdKQogICAgICAgICAgICAgICAgaWYgbm90IDAgPD0gbGFiZWwgPD0gNjoKICAgICAgICAgICAgICAgICAgICByYWlzZSBWYWx1ZUVycm9yKGYiSW52YWxpZCBsYWJlbCB7bGFiZWx9IGluIHtwfSIpCiAgICAgICAgICAgICAgICBsYWJlbHNfbGlzdC5hcHBlbmQobGFiZWwpCgogICAgICAgIGlmIG5vdCBpbWFnZXNfbGlzdDoKICAgICAgICAgICAgcmFpc2UgVmFsdWVFcnJvcihmIkZFUjIwMTMgQ1NWIGNvbnRhaW5zIG5vIGRhdGEgcm93czoge3B9IikKICAgICAgICBzZWxmLmltYWdlcyA9IG5wLnN0YWNrKGltYWdlc19saXN0LCBheGlzPTApICAjIFtOLCA0OCwgNDhdIHVpbnQ4CiAgICAgICAgc2VsZi5sYWJlbHMgPSBucC5hcnJheShsYWJlbHNfbGlzdCwgZHR5cGU9bnAuaW50NjQpCgogICAgICAgIGlmIHNlbGYuc3BsaXQgPT0gInRyYWluIiBhbmQgbGVuKHNlbGYuaW1hZ2VzKSAhPSBFWFBFQ1RFRF9UUkFJTl9ST1dTOgogICAgICAgICAgICB3YXJuaW5nc19tc2cgPSBmIkV4cGVjdGVkIHtFWFBFQ1RFRF9UUkFJTl9ST1dTfSByb3dzIGZvciB0cmFpbiwgZ290IHtsZW4oc2VsZi5pbWFnZXMpfSIKICAgICAgICAgICAgcHJpbnQoZiJbV2FybmluZ10ge3dhcm5pbmdzX21zZ30iKQogICAgICAgIGVsaWYgc2VsZi5zcGxpdCBpbiAoInZhbCIsICJwdWJsaWMiKSBhbmQgbGVuKHNlbGYuaW1hZ2VzKSAhPSBFWFBFQ1RFRF9QVUJMSUNfUk9XUzoKICAgICAgICAgICAgcHJpbnQoZiJbV2FybmluZ10gRXhwZWN0ZWQge0VYUEVDVEVEX1BVQkxJQ19ST1dTfSByb3dzIGZvciB2YWwsIGdvdCB7bGVuKHNlbGYuaW1hZ2VzKX0iKQogICAgICAgIGVsaWYgc2VsZi5zcGxpdCBpbiAoInRlc3QiLCAicHJpdmF0ZSIpIGFuZCBsZW4oc2VsZi5pbWFnZXMpICE9IEVYUEVDVEVEX1BSSVZBVEVfUk9XUzoKICAgICAgICAgICAgcHJpbnQoZiJbV2FybmluZ10gRXhwZWN0ZWQge0VYUEVDVEVEX1BSSVZBVEVfUk9XU30gcm93cyBmb3IgdGVzdCwgZ290IHtsZW4oc2VsZi5pbWFnZXMpfSIpCgogICAgZGVmIF9fbGVuX18oc2VsZikgLT4gaW50OgogICAgICAgIHJldHVybiBsZW4oc2VsZi5pbWFnZXMpCgogICAgZGVmIHNldF9lcG9jaChzZWxmLCBlcG9jaDogaW50KSAtPiBOb25lOgogICAgICAgICIiIlNlbGVjdCBkZXRlcm1pbmlzdGljLCBzYW1wbGUtbG9jYWwgYXVnbWVudGF0aW9uIGZvciBhbiBlcG9jaC4iIiIKICAgICAgICBzZWxmLmVwb2NoID0gaW50KGVwb2NoKQoKICAgIGRlZiBzdGF0ZV9kaWN0KHNlbGYpIC0+IGRpY3Q6CiAgICAgICAgcmV0dXJuIHsKICAgICAgICAgICAgInNjaGVtZSI6ICJzYW1wbGVfbG9jYWxfc2VlZF92MSIsCiAgICAgICAgICAgICJzZWVkIjogc2VsZi5zZWVkLAogICAgICAgICAgICAiZXBvY2giOiBzZWxmLmVwb2NoLAogICAgICAgIH0KCiAgICBkZWYgbG9hZF9zdGF0ZV9kaWN0KHNlbGYsIHN0YXRlOiBkaWN0KSAtPiBOb25lOgogICAgICAgIGlmIHN0YXRlLmdldCgic2NoZW1lIikgIT0gInNhbXBsZV9sb2NhbF9zZWVkX3YxIiBvciBpbnQoc3RhdGVbInNlZWQiXSkgIT0gc2VsZi5zZWVkOgogICAgICAgICAgICByYWlzZSBSdW50aW1lRXJyb3IoIlRyYWluaW5nIGF1Z21lbnRhdGlvbiBzdGF0ZSBpcyBpbmNvbXBhdGlibGUiKQogICAgICAgIHNlbGYuZXBvY2ggPSBpbnQoc3RhdGVbImVwb2NoIl0pCgogICAgZGVmIF9fZ2V0aXRlbV9fKHNlbGYsIGlkeDogaW50KSAtPiBUdXBsZVt0b3JjaC5UZW5zb3IsIGludF06CiAgICAgICAgaW1nX25wID0gc2VsZi5pbWFnZXNbaWR4XQogICAgICAgIGltZ190ZW5zb3IgPSB0b3JjaC5mcm9tX251bXB5KGltZ19ucCkuZmxvYXQoKS51bnNxdWVlemUoMCkgLyAyNTUuMCAgIyBbMSwgNDgsIDQ4XSBpbiBbMCwgMV0KICAgICAgICBsYWJlbCA9IGludChzZWxmLmxhYmVsc1tpZHhdKQoKICAgICAgICBpZiBzZWxmLmF1Z21lbnQ6CiAgICAgICAgICAgIGdlbmVyYXRvciA9IHRvcmNoLkdlbmVyYXRvcigpLm1hbnVhbF9zZWVkKAogICAgICAgICAgICAgICAgYXVnbWVudGF0aW9uX3NlZWQoc2VsZi5zZWVkLCBzZWxmLmVwb2NoLCBpbnQoaWR4KSkKICAgICAgICAgICAgKQogICAgICAgICAgICAjIENvbnNlcnZhdGl2ZSBGRVItc2FmZSBhdWdtZW50YXRpb25zOgogICAgICAgICAgICAjIDEuIFJhbmRvbSBob3Jpem9udGFsIGZsaXAgKHAgPSAwLjUpCiAgICAgICAgICAgIGlmIHRvcmNoLnJhbmQoMSwgZ2VuZXJhdG9yPWdlbmVyYXRvcikuaXRlbSgpID4gMC41OgogICAgICAgICAgICAgICAgaW1nX3RlbnNvciA9IFRGLmhmbGlwKGltZ190ZW5zb3IpCgogICAgICAgICAgICAjIDIuIFJhbmRvbSB0cmFuc2xhdGlvbiAoY3JvcC9zaGlmdCBieSB1cCB0byAyIHBpeGVscykKICAgICAgICAgICAgaWYgdG9yY2gucmFuZCgxLCBnZW5lcmF0b3I9Z2VuZXJhdG9yKS5pdGVtKCkgPiAwLjU6CiAgICAgICAgICAgICAgICBzaGlmdF95ID0gaW50KHRvcmNoLnJhbmRpbnQoLTIsIDMsICgxLCksIGdlbmVyYXRvcj1nZW5lcmF0b3IpLml0ZW0oKSkKICAgICAgICAgICAgICAgIHNoaWZ0X3ggPSBpbnQodG9yY2gucmFuZGludCgtMiwgMywgKDEsKSwgZ2VuZXJhdG9yPWdlbmVyYXRvcikuaXRlbSgpKQogICAgICAgICAgICAgICAgaW1nX3RlbnNvciA9IFRGLmFmZmluZSgKICAgICAgICAgICAgICAgICAgICBpbWdfdGVuc29yLAogICAgICAgICAgICAgICAgICAgIGFuZ2xlPTAuMCwKICAgICAgICAgICAgICAgICAgICB0cmFuc2xhdGU9W3NoaWZ0X3gsIHNoaWZ0X3ldLAogICAgICAgICAgICAgICAgICAgIHNjYWxlPTEuMCwKICAgICAgICAgICAgICAgICAgICBzaGVhcj1bMC4wLCAwLjBdLAogICAgICAgICAgICAgICAgICAgIGZpbGw9MC4wLAogICAgICAgICAgICAgICAgKQoKICAgICAgICAgICAgIyAzLiBWZXJ5IG1pbGQgaW50ZW5zaXR5IHNjYWxpbmc6IFswLjk1LCAxLjA1XQogICAgICAgICAgICBpZiB0b3JjaC5yYW5kKDEsIGdlbmVyYXRvcj1nZW5lcmF0b3IpLml0ZW0oKSA+IDAuNToKICAgICAgICAgICAgICAgIHNjYWxlID0gMC45NSArIDAuMTAgKiB0b3JjaC5yYW5kKDEsIGdlbmVyYXRvcj1nZW5lcmF0b3IpLml0ZW0oKQogICAgICAgICAgICAgICAgaW1nX3RlbnNvciA9IHRvcmNoLmNsYW1wKGltZ190ZW5zb3IgKiBzY2FsZSwgMC4wLCAxLjApCgogICAgICAgIHJldHVybiBpbWdfdGVuc29yLCBsYWJlbAoKCmRlZiBjcmVhdGVfdHJhaW5pbmdfZGF0YWxvYWRlcnMoCiAgICB0cmFpbl9jc3Y6IHN0ciB8IFBhdGgsCiAgICB2YWxfY3N2OiBzdHIgfCBQYXRoLAogICAgYmF0Y2hfc2l6ZTogaW50ID0gMTYsCiAgICBudW1fd29ya2VyczogaW50ID0gMiwKICAgIHNlZWQ6IGludCA9IDQyLAogICAgZ2VuZXJhdG9yOiB0b3JjaC5HZW5lcmF0b3IgfCBOb25lID0gTm9uZSwKKSAtPiBkaWN0W3N0ciwgRGF0YUxvYWRlcl06CiAgICAiIiJDcmVhdGUgb25seSBUcmFpbiBhbmQgUHVibGljVGVzdCBsb2FkZXJzOyBQcml2YXRlVGVzdCBzdGF5cyB1bm9wZW5lZC4iIiIKICAgIHRyYWluX3BhdGgsIHZhbF9wYXRoLCBfID0gdmFsaWRhdGVfc3BsaXRfcGF0aHModHJhaW5fY3N2LCB2YWxfY3N2KQogICAgdHJhaW5fZHMgPSBGRVIyMDEzRGF0YXNldCh0cmFpbl9wYXRoLCBzcGxpdD0idHJhaW4iLCBhdWdtZW50PVRydWUsIHNlZWQ9c2VlZCkKICAgIHZhbF9kcyA9IEZFUjIwMTNEYXRhc2V0KHZhbF9wYXRoLCBzcGxpdD0idmFsIiwgYXVnbWVudD1GYWxzZSkKICAgIGlmIGdlbmVyYXRvciBpcyBOb25lOgogICAgICAgIGdlbmVyYXRvciA9IHRvcmNoLkdlbmVyYXRvcigpLm1hbnVhbF9zZWVkKHNlZWQpCiAgICB0cmFpbl9zYW1wbGVyID0gRXBvY2hSYW5kb21TYW1wbGVyKHRyYWluX2RzLCBzZWVkPXNlZWQpCgogICAgbG9hZGVycyA9IHsKICAgICAgICAidHJhaW4iOiBEYXRhTG9hZGVyKAogICAgICAgICAgICB0cmFpbl9kcywKICAgICAgICAgICAgYmF0Y2hfc2l6ZT1iYXRjaF9zaXplLAogICAgICAgICAgICBzaHVmZmxlPUZhbHNlLAogICAgICAgICAgICBzYW1wbGVyPXRyYWluX3NhbXBsZXIsCiAgICAgICAgICAgIG51bV93b3JrZXJzPW51bV93b3JrZXJzLAogICAgICAgICAgICBwaW5fbWVtb3J5PVRydWUsCiAgICAgICAgICAgIGRyb3BfbGFzdD1GYWxzZSwKICAgICAgICAgICAgZ2VuZXJhdG9yPWdlbmVyYXRvciwKICAgICAgICApLAogICAgICAgICJ2YWwiOiBEYXRhTG9hZGVyKAogICAgICAgICAgICB2YWxfZHMsCiAgICAgICAgICAgIGJhdGNoX3NpemU9YmF0Y2hfc2l6ZSwKICAgICAgICAgICAgc2h1ZmZsZT1GYWxzZSwKICAgICAgICAgICAgbnVtX3dvcmtlcnM9bnVtX3dvcmtlcnMsCiAgICAgICAgICAgIHBpbl9tZW1vcnk9VHJ1ZSwKICAgICAgICAgICAgZHJvcF9sYXN0PUZhbHNlLAogICAgICAgICksCiAgICB9CgogICAgcmV0dXJuIGxvYWRlcnMKCgpkZWYgY3JlYXRlX3ByaXZhdGVfZGF0YWxvYWRlcigKICAgIHRlc3RfY3N2OiBzdHIgfCBQYXRoLAogICAgYmF0Y2hfc2l6ZTogaW50ID0gMTYsCiAgICBudW1fd29ya2VyczogaW50ID0gMiwKKSAtPiBEYXRhTG9hZGVyOgogICAgIiIiQ29uc3RydWN0IHRoZSBQcml2YXRlVGVzdCBsb2FkZXIgb25seSBhZnRlciBjaGVja3BvaW50IGZyZWV6ZS4iIiIKICAgIHRlc3RfcGF0aCA9IHZhbGlkYXRlX3NwbGl0X3BhdGgodGVzdF9jc3YsICJ0ZXN0IikKICAgIHRlc3RfZHMgPSBGRVIyMDEzRGF0YXNldCh0ZXN0X3BhdGgsIHNwbGl0PSJ0ZXN0IiwgYXVnbWVudD1GYWxzZSkKICAgIHJldHVybiBEYXRhTG9hZGVyKAogICAgICAgIHRlc3RfZHMsCiAgICAgICAgYmF0Y2hfc2l6ZT1iYXRjaF9zaXplLAogICAgICAgIHNodWZmbGU9RmFsc2UsCiAgICAgICAgbnVtX3dvcmtlcnM9bnVtX3dvcmtlcnMsCiAgICAgICAgcGluX21lbW9yeT1UcnVlLAogICAgICAgIGRyb3BfbGFzdD1GYWxzZSwKICAgICkK', 'mpg_fer_v2_3/ema.py': 'IiIiQ29tcGxldGUgbW9kZWwtc3RhdGUgZXhwb25lbnRpYWwgbW92aW5nIGF2ZXJhZ2UuIiIiCgpmcm9tIF9fZnV0dXJlX18gaW1wb3J0IGFubm90YXRpb25zCgppbXBvcnQgY29weQppbXBvcnQgdG9yY2gKaW1wb3J0IHRvcmNoLm5uIGFzIG5uCgoKY2xhc3MgTW9kZWxFTUE6CiAgICAiIiJUcmFjayBwYXJhbWV0ZXJzIGFuZCBidWZmZXJzIGFuZCBleHBvc2UgZGV0ZXJtaW5pc3RpYyBldmFsdWF0aW9uIHN0YXRlLiIiIgoKICAgIGRlZiBfX2luaXRfXyhzZWxmLCBtb2RlbDogbm4uTW9kdWxlLCBkZWNheTogZmxvYXQgPSAwLjk5OSkgLT4gTm9uZToKICAgICAgICBpZiBub3QgMC4wIDw9IGRlY2F5IDwgMS4wOgogICAgICAgICAgICByYWlzZSBWYWx1ZUVycm9yKCJFTUEgZGVjYXkgbXVzdCBiZSBpbiBbMCwgMSkiKQogICAgICAgIHNlbGYuZGVjYXkgPSBmbG9hdChkZWNheSkKICAgICAgICBzZWxmLm51bV91cGRhdGVzID0gMAogICAgICAgIHNlbGYubW9kdWxlID0gY29weS5kZWVwY29weShtb2RlbCkuZXZhbCgpCiAgICAgICAgc2VsZi5tb2R1bGUucmVxdWlyZXNfZ3JhZF8oRmFsc2UpCgogICAgQHRvcmNoLm5vX2dyYWQoKQogICAgZGVmIHVwZGF0ZShzZWxmLCBtb2RlbDogbm4uTW9kdWxlKSAtPiBOb25lOgogICAgICAgIHNvdXJjZSA9IG1vZGVsLnN0YXRlX2RpY3QoKQogICAgICAgIHRhcmdldCA9IHNlbGYubW9kdWxlLnN0YXRlX2RpY3QoKQogICAgICAgIHBhcmFtZXRlcl9uYW1lcyA9IHNldChkaWN0KG1vZGVsLm5hbWVkX3BhcmFtZXRlcnMoKSkpCiAgICAgICAgaWYgc291cmNlLmtleXMoKSAhPSB0YXJnZXQua2V5cygpOgogICAgICAgICAgICByYWlzZSBSdW50aW1lRXJyb3IoIkVNQS9tb2RlbCBzdGF0ZSBrZXlzIGRpZmZlciIpCiAgICAgICAgZm9yIG5hbWUsIGVtYV92YWx1ZSBpbiB0YXJnZXQuaXRlbXMoKToKICAgICAgICAgICAgdmFsdWUgPSBzb3VyY2VbbmFtZV0uZGV0YWNoKCkKICAgICAgICAgICAgaWYgbmFtZSBpbiBwYXJhbWV0ZXJfbmFtZXMgYW5kIHRvcmNoLmlzX2Zsb2F0aW5nX3BvaW50KGVtYV92YWx1ZSk6CiAgICAgICAgICAgICAgICBlbWFfdmFsdWUubXVsXyhzZWxmLmRlY2F5KS5hZGRfKHZhbHVlLCBhbHBoYT0xLjAgLSBzZWxmLmRlY2F5KQogICAgICAgICAgICBlbHNlOgogICAgICAgICAgICAgICAgIyBTY2hlZHVsZWQgc3RhdGUgc3VjaCBhcyBjdXJyZW50X3RhdSBpcyBhdXRob3JpdGF0aXZlIHN0YXRlLAogICAgICAgICAgICAgICAgIyBub3QgYSBxdWFudGl0eSB0aGF0IEVNQSBpcyBhbGxvd2VkIHRvIG51bWVyaWNhbGx5IGF2ZXJhZ2UuCiAgICAgICAgICAgICAgICBlbWFfdmFsdWUuY29weV8odmFsdWUpCiAgICAgICAgc2VsZi5udW1fdXBkYXRlcyArPSAxCgogICAgZGVmIHN0YXRlX2RpY3Qoc2VsZikgLT4gZGljdDoKICAgICAgICByZXR1cm4gewogICAgICAgICAgICAiZGVjYXkiOiBzZWxmLmRlY2F5LAogICAgICAgICAgICAibnVtX3VwZGF0ZXMiOiBzZWxmLm51bV91cGRhdGVzLAogICAgICAgICAgICAibW9kZWxfc3RhdGVfZGljdCI6IHNlbGYubW9kdWxlLnN0YXRlX2RpY3QoKSwKICAgICAgICB9CgogICAgZGVmIGxvYWRfc3RhdGVfZGljdChzZWxmLCBzdGF0ZTogZGljdCkgLT4gTm9uZToKICAgICAgICBzZWxmLmRlY2F5ID0gZmxvYXQoc3RhdGVbImRlY2F5Il0pCiAgICAgICAgc2VsZi5udW1fdXBkYXRlcyA9IGludChzdGF0ZVsibnVtX3VwZGF0ZXMiXSkKICAgICAgICBzZWxmLm1vZHVsZS5sb2FkX3N0YXRlX2RpY3Qoc3RhdGVbIm1vZGVsX3N0YXRlX2RpY3QiXSwgc3RyaWN0PVRydWUpCg==', 'mpg_fer_v2_3/evaluate.py': 'IiIiRXZhbHVhdGlvbiB1dGlsaXRpZXMgd2l0aCBzaW5nbGUtcGFzcyByYXcgYW5kIGhvcml6b250YWwtZmxpcCBUVEEgbWV0cmljcy4iIiIKCmZyb20gX19mdXR1cmVfXyBpbXBvcnQgYW5ub3RhdGlvbnMKCmltcG9ydCBudW1weSBhcyBucApmcm9tIHNrbGVhcm4ubWV0cmljcyBpbXBvcnQgY29uZnVzaW9uX21hdHJpeCwgZjFfc2NvcmUsIHByZWNpc2lvbl9zY29yZSwgcmVjYWxsX3Njb3JlCmltcG9ydCB0b3JjaAppbXBvcnQgdG9yY2gubm4gYXMgbm4KZnJvbSB0b3JjaC51dGlscy5kYXRhIGltcG9ydCBEYXRhTG9hZGVyCmltcG9ydCB0b3JjaHZpc2lvbi50cmFuc2Zvcm1zLmZ1bmN0aW9uYWwgYXMgVEYKCgpkZWYgX2NsYXNzaWZpY2F0aW9uX21ldHJpY3MoCiAgICB0YXJnZXRzOiBsaXN0W2ludF0sIHByZWRpY3Rpb25zOiBsaXN0W2ludF0sIHRvdGFsX2xvc3M6IGZsb2F0CikgLT4gZGljdDoKICAgIHlfdHJ1ZSA9IG5wLmFzYXJyYXkodGFyZ2V0cywgZHR5cGU9bnAuaW50NjQpCiAgICB5X3ByZWQgPSBucC5hc2FycmF5KHByZWRpY3Rpb25zLCBkdHlwZT1ucC5pbnQ2NCkKICAgIGlmIGxlbih5X3RydWUpID09IDA6CiAgICAgICAgcmFpc2UgVmFsdWVFcnJvcigiQ2Fubm90IGV2YWx1YXRlIGFuIGVtcHR5IGRhdGFsb2FkZXIiKQogICAgcmV0dXJuIHsKICAgICAgICAibG9zcyI6IGZsb2F0KHRvdGFsX2xvc3MgLyBsZW4oeV90cnVlKSksCiAgICAgICAgImFjY3VyYWN5IjogZmxvYXQobnAubWVhbih5X3RydWUgPT0geV9wcmVkKSksCiAgICAgICAgIm1hY3JvX2YxIjogZmxvYXQoCiAgICAgICAgICAgIGYxX3Njb3JlKHlfdHJ1ZSwgeV9wcmVkLCBhdmVyYWdlPSJtYWNybyIsIHplcm9fZGl2aXNpb249MCkKICAgICAgICApLAogICAgICAgICJwZXJfY2xhc3NfZjEiOiBbCiAgICAgICAgICAgIGZsb2F0KHgpCiAgICAgICAgICAgIGZvciB4IGluIGYxX3Njb3JlKAogICAgICAgICAgICAgICAgeV90cnVlLCB5X3ByZWQsIGF2ZXJhZ2U9Tm9uZSwgbGFiZWxzPW5wLmFyYW5nZSg3KSwgemVyb19kaXZpc2lvbj0wCiAgICAgICAgICAgICkudG9saXN0KCkKICAgICAgICBdLAogICAgICAgICJwcmVjaXNpb24iOiBbCiAgICAgICAgICAgIGZsb2F0KHgpCiAgICAgICAgICAgIGZvciB4IGluIHByZWNpc2lvbl9zY29yZSgKICAgICAgICAgICAgICAgIHlfdHJ1ZSwgeV9wcmVkLCBhdmVyYWdlPU5vbmUsIGxhYmVscz1ucC5hcmFuZ2UoNyksIHplcm9fZGl2aXNpb249MAogICAgICAgICAgICApLnRvbGlzdCgpCiAgICAgICAgXSwKICAgICAgICAicmVjYWxsIjogWwogICAgICAgICAgICBmbG9hdCh4KQogICAgICAgICAgICBmb3IgeCBpbiByZWNhbGxfc2NvcmUoCiAgICAgICAgICAgICAgICB5X3RydWUsIHlfcHJlZCwgYXZlcmFnZT1Ob25lLCBsYWJlbHM9bnAuYXJhbmdlKDcpLCB6ZXJvX2RpdmlzaW9uPTAKICAgICAgICAgICAgKS50b2xpc3QoKQogICAgICAgIF0sCiAgICAgICAgImNvbmZ1c2lvbl9tYXRyaXgiOiBjb25mdXNpb25fbWF0cml4KAogICAgICAgICAgICB5X3RydWUsIHlfcHJlZCwgbGFiZWxzPW5wLmFyYW5nZSg3KQogICAgICAgICkudG9saXN0KCksCiAgICAgICAgInN1cHBvcnQiOiBbaW50KCh5X3RydWUgPT0gbGFiZWwpLnN1bSgpKSBmb3IgbGFiZWwgaW4gcmFuZ2UoNyldLAogICAgfQoKCkB0b3JjaC5ub19ncmFkKCkKZGVmIGV2YWx1YXRlX3Jhd19hbmRfdHRhKAogICAgbW9kZWw6IG5uLk1vZHVsZSwKICAgIGRhdGFsb2FkZXI6IERhdGFMb2FkZXIsCiAgICBkZXZpY2U6IHN0ciB8IHRvcmNoLmRldmljZSwKICAgIHVzZV9hbXA6IGJvb2wgPSBUcnVlLAopIC0+IGRpY3Rbc3RyLCBkaWN0XToKICAgICIiIkNvbXB1dGUgcmF3IGFuZCBsb2dpdC1hdmVyYWdlZCBmbGlwLVRUQSBtZXRyaWNzIGluIG9uZSBsb2FkZXIgdHJhdmVyc2FsLgoKICAgIFRoZSBpbWFnZSBpcyBob3Jpem9udGFsbHkgZmxpcHBlZCBiZWZvcmUgdGhlIG1vZGVsIGV4dHJhY3RzIHJlbGF0aW9uYWwKICAgIGZlYXR1cmVzLiBQcml2YXRlVGVzdCBjYWxsZXJzIHRoZXJlZm9yZSBtYWtlIGV4YWN0bHkgb25lIHBoeXNpY2FsIHBhc3Mgb3ZlcgogICAgdGhlIGxvYWRlciB3aGlsZSBvYnRhaW5pbmcgYm90aCByZXBvcnRpbmcgdmlld3MgZnJvbSB0aGUgZnJvemVuIHdlaWdodHMuCiAgICAiIiIKICAgIG1vZGVsLmV2YWwoKQogICAgcmF3X3ByZWRzOiBsaXN0W2ludF0gPSBbXQogICAgdHRhX3ByZWRzOiBsaXN0W2ludF0gPSBbXQogICAgdGFyZ2V0c19hbGw6IGxpc3RbaW50XSA9IFtdCiAgICByYXdfbG9zc190b3RhbCA9IDAuMAogICAgdHRhX2xvc3NfdG90YWwgPSAwLjAKICAgIGNyaXRlcmlvbiA9IG5uLkNyb3NzRW50cm9weUxvc3MoKQogICAgYW1wX2VuYWJsZWQgPSB1c2VfYW1wIGFuZCB0b3JjaC5jdWRhLmlzX2F2YWlsYWJsZSgpCgogICAgZm9yIGltYWdlcywgdGFyZ2V0cyBpbiBkYXRhbG9hZGVyOgogICAgICAgIGltYWdlcyA9IGltYWdlcy50byhkZXZpY2UpCiAgICAgICAgdGFyZ2V0cyA9IHRhcmdldHMudG8oZGV2aWNlKQogICAgICAgIHdpdGggdG9yY2guYW1wLmF1dG9jYXN0KCJjdWRhIiwgZW5hYmxlZD1hbXBfZW5hYmxlZCk6CiAgICAgICAgICAgIHJhd19sb2dpdHMsIF8gPSBtb2RlbChpbWFnZXMpCiAgICAgICAgICAgIGZsaXBwZWRfbG9naXRzLCBfID0gbW9kZWwoVEYuaGZsaXAoaW1hZ2VzKSkKICAgICAgICAgICAgdHRhX2xvZ2l0cyA9IDAuNSAqIChyYXdfbG9naXRzICsgZmxpcHBlZF9sb2dpdHMpCiAgICAgICAgICAgIHJhd19sb3NzID0gY3JpdGVyaW9uKHJhd19sb2dpdHMsIHRhcmdldHMpCiAgICAgICAgICAgIHR0YV9sb3NzID0gY3JpdGVyaW9uKHR0YV9sb2dpdHMsIHRhcmdldHMpCgogICAgICAgIGJhdGNoX3NpemUgPSBsZW4odGFyZ2V0cykKICAgICAgICByYXdfbG9zc190b3RhbCArPSByYXdfbG9zcy5pdGVtKCkgKiBiYXRjaF9zaXplCiAgICAgICAgdHRhX2xvc3NfdG90YWwgKz0gdHRhX2xvc3MuaXRlbSgpICogYmF0Y2hfc2l6ZQogICAgICAgIHJhd19wcmVkcy5leHRlbmQodG9yY2guYXJnbWF4KHJhd19sb2dpdHMsIGRpbT0tMSkuY3B1KCkudG9saXN0KCkpCiAgICAgICAgdHRhX3ByZWRzLmV4dGVuZCh0b3JjaC5hcmdtYXgodHRhX2xvZ2l0cywgZGltPS0xKS5jcHUoKS50b2xpc3QoKSkKICAgICAgICB0YXJnZXRzX2FsbC5leHRlbmQodGFyZ2V0cy5jcHUoKS50b2xpc3QoKSkKCiAgICByZXR1cm4gewogICAgICAgICJyYXciOiBfY2xhc3NpZmljYXRpb25fbWV0cmljcyh0YXJnZXRzX2FsbCwgcmF3X3ByZWRzLCByYXdfbG9zc190b3RhbCksCiAgICAgICAgInR0YSI6IF9jbGFzc2lmaWNhdGlvbl9tZXRyaWNzKHRhcmdldHNfYWxsLCB0dGFfcHJlZHMsIHR0YV9sb3NzX3RvdGFsKSwKICAgIH0KCgpAdG9yY2gubm9fZ3JhZCgpCmRlZiBldmFsdWF0ZV9tb2RlbCgKICAgIG1vZGVsOiBubi5Nb2R1bGUsCiAgICBkYXRhbG9hZGVyOiBEYXRhTG9hZGVyLAogICAgZGV2aWNlOiBzdHIgfCB0b3JjaC5kZXZpY2UsCiAgICB1c2VfdHRhOiBib29sID0gRmFsc2UsCiAgICB1c2VfYW1wOiBib29sID0gVHJ1ZSwKKSAtPiBkaWN0OgogICAgIiIiQ29tcGF0aWJpbGl0eSB3cmFwcGVyIHNlbGVjdGluZyBvbmUgdmlldyBmcm9tIHRoZSBzaW5nbGUtcGFzcyBldmFsdWF0b3IuIiIiCiAgICBib3RoID0gZXZhbHVhdGVfcmF3X2FuZF90dGEobW9kZWwsIGRhdGFsb2FkZXIsIGRldmljZSwgdXNlX2FtcD11c2VfYW1wKQogICAgc2VsZWN0ZWQgPSBkaWN0KGJvdGhbInR0YSIgaWYgdXNlX3R0YSBlbHNlICJyYXciXSkKICAgIHNlbGVjdGVkWyJ1c2VfdHRhIl0gPSB1c2VfdHRhCiAgICByZXR1cm4gc2VsZWN0ZWQK', 'mpg_fer_v2_3/features.py': 'IiIiQ29udGludW91cyBIMS1pbnNwaXJlZCAzMkQgcmVsYXRpb25hbCBwaXhlbCBmZWF0dXJlIGV4dHJhY3Rpb24uIiIiCgpmcm9tIF9fZnV0dXJlX18gaW1wb3J0IGFubm90YXRpb25zCgppbXBvcnQgdG9yY2gKaW1wb3J0IHRvcmNoLm5uIGFzIG5uCmltcG9ydCB0b3JjaC5ubi5mdW5jdGlvbmFsIGFzIEYKCgpjbGFzcyBQaXhlbEZlYXR1cmVFeHRyYWN0b3Iobm4uTW9kdWxlKToKICAgICIiIkV4dHJhY3RzIGV4YWN0bHkgMzIgcmF3IGNvbnRpbnVvdXMgcmVsYXRpb25hbCBmZWF0dXJlcyBwZXIgcGl4ZWwgb24gYSA0OHg0OCBpbWFnZS4KCiAgICBGZWF0dXJlIGNvbXBvc2l0aW9uIHBlciBwaXhlbDoKICAgICAgLSAxICBub3JtYWxpemVkIGludGVuc2l0eSBJIGluIFswLCAxXQogICAgICAtIDIgIG5vcm1hbGl6ZWQgY29vcmRpbmF0ZXMgeCwgeSBpbiBbLTEsIDFdCiAgICAgIC0gMjQgc2lnbWEtbm9ybWFsaXplZCBjZW50ZXItcmVsYXRpdmUgZGlmZmVyZW5jZXMgZnJvbSA1eDUgbmVpZ2hib3Job29kCiAgICAgIC0gMSAgbG9nX3NpZ21hOiBsb2coc3FydCh2YXJfbG9jYWwgKyBlcHNeMikgKyBlcHMpCiAgICAgIC0gNCAgZ3gsIGd5LCBncmFkaWVudF9tYWduaXR1ZGUsIGxhcGxhY2lhbgogICAgICAtLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0KICAgICAgVG90YWw6IGV4YWN0bHkgMzIgZGltZW5zaW9ucy4KCiAgICBVc2VzIHJlZmxlY3Rpb24gcGFkZGluZyBmb3IgdGhlIDV4NSBzcGF0aWFsIHN1cHBvcnQgdG8gcHJlc2VydmUgYWxsIDIzMDQgcGl4ZWxzLgogICAgIiIiCgogICAgZGVmIF9faW5pdF9fKHNlbGYsIGltZ19zaXplOiBpbnQgPSA0OCwgZXBzOiBmbG9hdCA9IDFlLTUpIC0+IE5vbmU6CiAgICAgICAgc3VwZXIoKS5fX2luaXRfXygpCiAgICAgICAgc2VsZi5pbWdfc2l6ZSA9IGltZ19zaXplCiAgICAgICAgc2VsZi5udW1fcGl4ZWxzID0gaW1nX3NpemUgKiBpbWdfc2l6ZQogICAgICAgIHNlbGYuZXBzID0gZXBzCgogICAgICAgICMgUHJlY29tcHV0ZSBub3JtYWxpemVkIHNwYXRpYWwgY29vcmRpbmF0ZSBncmlkcyBpbiBbLTEsIDFdCiAgICAgICAgY29vcmRzID0gdG9yY2gubGluc3BhY2UoLTEuMCwgMS4wLCBpbWdfc2l6ZSkKICAgICAgICBncmlkX3ksIGdyaWRfeCA9IHRvcmNoLm1lc2hncmlkKGNvb3JkcywgY29vcmRzLCBpbmRleGluZz0iaWoiKQogICAgICAgICMgWzEsIDEsIDQ4LCA0OF0KICAgICAgICBzZWxmLnJlZ2lzdGVyX2J1ZmZlcigiZ3JpZF94IiwgZ3JpZF94LnVuc3F1ZWV6ZSgwKS51bnNxdWVlemUoMCkuY2xvbmUoKSkKICAgICAgICBzZWxmLnJlZ2lzdGVyX2J1ZmZlcigiZ3JpZF95IiwgZ3JpZF95LnVuc3F1ZWV6ZSgwKS51bnNxdWVlemUoMCkuY2xvbmUoKSkKCiAgICAgICAgIyBTb2JlbCBncmFkaWVudCBmaWx0ZXJzCiAgICAgICAgc29iZWxfeCA9IHRvcmNoLnRlbnNvcigKICAgICAgICAgICAgW1stMS4wLCAwLjAsIDEuMF0sIFstMi4wLCAwLjAsIDIuMF0sIFstMS4wLCAwLjAsIDEuMF1dLAogICAgICAgICAgICBkdHlwZT10b3JjaC5mbG9hdDMyLAogICAgICAgICkucmVzaGFwZSgxLCAxLCAzLCAzKSAvIDguMAogICAgICAgIHNvYmVsX3kgPSB0b3JjaC50ZW5zb3IoCiAgICAgICAgICAgIFtbLTEuMCwgLTIuMCwgLTEuMF0sIFswLjAsIDAuMCwgMC4wXSwgWzEuMCwgMi4wLCAxLjBdXSwKICAgICAgICAgICAgZHR5cGU9dG9yY2guZmxvYXQzMiwKICAgICAgICApLnJlc2hhcGUoMSwgMSwgMywgMykgLyA4LjAKICAgICAgICBsYXBsYWNpYW4gPSB0b3JjaC50ZW5zb3IoCiAgICAgICAgICAgIFtbMC4wLCAxLjAsIDAuMF0sIFsxLjAsIC00LjAsIDEuMF0sIFswLjAsIDEuMCwgMC4wXV0sCiAgICAgICAgICAgIGR0eXBlPXRvcmNoLmZsb2F0MzIsCiAgICAgICAgKS5yZXNoYXBlKDEsIDEsIDMsIDMpIC8gNC4wCgogICAgICAgIHNlbGYucmVnaXN0ZXJfYnVmZmVyKCJzb2JlbF94Iiwgc29iZWxfeCkKICAgICAgICBzZWxmLnJlZ2lzdGVyX2J1ZmZlcigic29iZWxfeSIsIHNvYmVsX3kpCiAgICAgICAgc2VsZi5yZWdpc3Rlcl9idWZmZXIoImxhcGxhY2lhbiIsIGxhcGxhY2lhbikKCiAgICAgICAgIyA1eDUgbG9jYWwgbWVhbiBmaWx0ZXIKICAgICAgICBvbmVzXzV4NSA9IHRvcmNoLm9uZXMoKDEsIDEsIDUsIDUpLCBkdHlwZT10b3JjaC5mbG9hdDMyKSAvIDI1LjAKICAgICAgICBzZWxmLnJlZ2lzdGVyX2J1ZmZlcigibWVhbl81eDUiLCBvbmVzXzV4NSkKCiAgICBkZWYgZm9yd2FyZChzZWxmLCB4OiB0b3JjaC5UZW5zb3IpIC0+IHRvcmNoLlRlbnNvcjoKICAgICAgICAiIiJBcmdzOgoKICAgICAgICAgICAgeDogW0IsIDEsIDQ4LCA0OF0gaW1hZ2VzIG5vcm1hbGl6ZWQgdG8gWzAsIDFdLgoKICAgICAgICBSZXR1cm5zOgogICAgICAgICAgICBmZWF0dXJlczogW0IsIDIzMDQsIDMyXSBjb250aW51b3VzIHJlbGF0aW9uYWwgcGl4ZWwgZmVhdHVyZXMuCiAgICAgICAgIiIiCiAgICAgICAgQiwgQywgSCwgVyA9IHguc2hhcGUKICAgICAgICBhc3NlcnQgQyA9PSAxIGFuZCBIID09IHNlbGYuaW1nX3NpemUgYW5kIFcgPT0gc2VsZi5pbWdfc2l6ZQoKICAgICAgICAjIDEuIE5vcm1hbGl6ZWQgaW50ZW5zaXR5IEk6IFtCLCAxLCA0OCwgNDhdCiAgICAgICAgZmVhdF9JID0geAoKICAgICAgICAjIDIuIE5vcm1hbGl6ZWQgY29vcmRpbmF0ZXMgKHgsIHkpOiBbQiwgMiwgNDgsIDQ4XQogICAgICAgIGZlYXRfY29vcmRzID0gdG9yY2guY2F0KAogICAgICAgICAgICBbc2VsZi5ncmlkX3guZXhwYW5kKEIsIC0xLCAtMSwgLTEpLCBzZWxmLmdyaWRfeS5leHBhbmQoQiwgLTEsIC0xLCAtMSldLAogICAgICAgICAgICBkaW09MSwKICAgICAgICApCgogICAgICAgICMgMy4gNXg1IG5laWdoYm9yaG9vZCB1bmZvbGRpbmcgd2l0aCByZWZsZWN0aW9uIHBhZGRpbmcKICAgICAgICAjIFBhZCBieSAyIG9uIGFsbCBzaWRlcyB0byBrZWVwIGV4YWN0ICg0OCwgNDgpIGdyaWQKICAgICAgICB4X3BhZCA9IEYucGFkKHgsICgyLCAyLCAyLCAyKSwgbW9kZT0icmVmbGVjdCIpCiAgICAgICAgIyB1bmZvbGQ6IFtCLCAyNSwgNDgsIDQ4XQogICAgICAgIHBhdGNoZXMgPSBGLnVuZm9sZCh4X3BhZCwga2VybmVsX3NpemU9NSwgcGFkZGluZz0wLCBzdHJpZGU9MSkucmVzaGFwZSgKICAgICAgICAgICAgQiwgMjUsIEgsIFcKICAgICAgICApCgogICAgICAgICMgQ2VudGVyIHBpeGVsIGlzIGF0IGluZGV4IDEyIChyb3cgMiwgY29sIDIgaW4gNXg1KQogICAgICAgIGNlbnRlcl9waXhlbCA9IHBhdGNoZXNbOiwgMTI6MTMsIDosIDpdICAjIFtCLCAxLCBILCBXXQoKICAgICAgICAjIDI0IG5vbi1jZW50ZXIgbmVpZ2hib3IgcGl4ZWxzCiAgICAgICAgaW5kaWNlc18yNCA9IFtpIGZvciBpIGluIHJhbmdlKDI1KSBpZiBpICE9IDEyXQogICAgICAgIG5laWdoYm9yc18yNCA9IHBhdGNoZXNbOiwgaW5kaWNlc18yNCwgOiwgOl0gICMgW0IsIDI0LCBILCBXXQoKICAgICAgICAjIExvY2FsIHZhcmlhbmNlICYgc2lnbWEgb3ZlciA1eDUgd2luZG93CiAgICAgICAgbWVhbl9sb2NhbCA9IEYuY29udjJkKHhfcGFkLCBzZWxmLm1lYW5fNXg1KQogICAgICAgIG1lYW5fc3FfbG9jYWwgPSBGLmNvbnYyZCh4X3BhZCoqMiwgc2VsZi5tZWFuXzV4NSkKICAgICAgICB2YXJfbG9jYWwgPSBGLnJlbHUobWVhbl9zcV9sb2NhbCAtIG1lYW5fbG9jYWwqKjIpCiAgICAgICAgc2lnbWFfbG9jYWwgPSB0b3JjaC5zcXJ0KHZhcl9sb2NhbCArIChzZWxmLmVwcyoqMikpICAjIFtCLCAxLCBILCBXXQoKICAgICAgICAjIDI0IHNpZ21hLW5vcm1hbGl6ZWQgY2VudGVyLXJlbGF0aXZlIGRpZmZlcmVuY2VzOgogICAgICAgICMgZF9rID0gKElfbmVpZ2hib3IgLSBJX2NlbnRlcikgLyBzcXJ0KHZhcl9sb2NhbCArIGVwc14yKQogICAgICAgIGRpZmZzXzI0ID0gKG5laWdoYm9yc18yNCAtIGNlbnRlcl9waXhlbCkgLyBzaWdtYV9sb2NhbCAgIyBbQiwgMjQsIEgsIFddCgogICAgICAgICMgNC4gbG9nX3NpZ21hOiBsb2coc3FydCh2YXJfbG9jYWwgKyBlcHNeMikgKyBlcHMpCiAgICAgICAgbG9nX3NpZ21hID0gdG9yY2gubG9nKHNpZ21hX2xvY2FsICsgc2VsZi5lcHMpICAjIFtCLCAxLCBILCBXXQoKICAgICAgICAjIDUuIEdyYWRpZW50cyAmIExhcGxhY2lhbiB3aXRoIHJlZmxlY3Rpb24gcGFkZGluZwogICAgICAgIHhfcGFkMyA9IEYucGFkKHgsICgxLCAxLCAxLCAxKSwgbW9kZT0icmVmbGVjdCIpCiAgICAgICAgZ3ggPSBGLmNvbnYyZCh4X3BhZDMsIHNlbGYuc29iZWxfeCkKICAgICAgICBneSA9IEYuY29udjJkKHhfcGFkMywgc2VsZi5zb2JlbF95KQogICAgICAgIGdyYWRfbWFnID0gdG9yY2guc3FydChneCoqMiArIGd5KioyICsgKHNlbGYuZXBzKioyKSkKICAgICAgICBsYXAgPSBGLmNvbnYyZCh4X3BhZDMsIHNlbGYubGFwbGFjaWFuKQogICAgICAgICMgW0IsIDQsIEgsIFddCiAgICAgICAgZGVyaXZhdGl2ZXMgPSB0b3JjaC5jYXQoW2d4LCBneSwgZ3JhZF9tYWcsIGxhcF0sIGRpbT0xKQoKICAgICAgICAjIENvbmNhdGVuYXRlIGFsbCAzMiBjaGFubmVsczoKICAgICAgICAjIDEgKEkpICsgMiAoY29vcmRzKSArIDI0IChkaWZmcykgKyAxIChsb2dfc2lnbWEpICsgNCAoZGVyaXZhdGl2ZXMpID0gMzIKICAgICAgICBhbGxfZmVhdHMgPSB0b3JjaC5jYXQoCiAgICAgICAgICAgIFtmZWF0X0ksIGZlYXRfY29vcmRzLCBkaWZmc18yNCwgbG9nX3NpZ21hLCBkZXJpdmF0aXZlc10sIGRpbT0xCiAgICAgICAgKSAgIyBbQiwgMzIsIDQ4LCA0OF0KCiAgICAgICAgIyBSZXNoYXBlIHRvIFtCLCAyMzA0LCAzMl0KICAgICAgICAjIENoYW5uZWxzLWxhc3Q6IFtCLCA0OCwgNDgsIDMyXSAtPiBbQiwgMjMwNCwgMzJdCiAgICAgICAgYWxsX2ZlYXRzID0gYWxsX2ZlYXRzLnBlcm11dGUoMCwgMiwgMywgMSkucmVzaGFwZShCLCBzZWxmLm51bV9waXhlbHMsIDMyKQogICAgICAgIHJldHVybiBhbGxfZmVhdHMK', 'mpg_fer_v2_3/graph.py': 'IiIiRml4ZWQgOC1uZWlnaGJvciBwaXhlbCBncmFwaCB0b3BvbG9neSBhbmQgcmVsYXRpdmUgZWRnZSBmZWF0dXJlIGNvbnN0cnVjdGlvbi4iIiIKCmZyb20gX19mdXR1cmVfXyBpbXBvcnQgYW5ub3RhdGlvbnMKCmltcG9ydCB0b3JjaAppbXBvcnQgdG9yY2gubm4gYXMgbm4KCgpjbGFzcyBQaXhlbEdyYXBoVG9wb2xvZ3kobm4uTW9kdWxlKToKICAgICIiIkJ1aWxkcyBhbmQgY2FjaGVzIHRoZSBzdGF0aWMgOC1uZWlnaGJvciBkaXJlY3RlZCBncmFwaCBvdmVyIGEgNDh4NDggcGl4ZWwgZ3JpZC4KCiAgICBOZWlnaGJvcmhvb2QgbGF5b3V0IGZvciByZWNlaXZlciBpIGF0IChyX2ksIGNfaSk6CiAgICAgIDA6IE5XIChyLTEsIGMtMSkKICAgICAgMTogTiAgKHItMSwgYykKICAgICAgMjogTkUgKHItMSwgYysxKQogICAgICAzOiBXICAociwgICBjLTEpCiAgICAgIDQ6IEUgIChyLCAgIGMrMSkKICAgICAgNTogU1cgKHIrMSwgYy0xKQogICAgICA2OiBTICAocisxLCBjKQogICAgICA3OiBTRSAocisxLCBjKzEpCgogICAgRWRnZXMgYXJlIGRpcmVjdGVkIHJlY2VpdmVyLXNlbmRlciByZWxhdGlvbnM6IGogLT4gaSAoc2VuZGVyIGogY29udHJpYnV0aW5nIHRvIHJlY2VpdmVyIGkpLgogICAgRWRnZSBmZWF0dXJlcyAoNUQpOgogICAgICBbZHgsIGR5LCBkaXN0YW5jZSwgZGVsdGFfSSwgYWJzX2RlbHRhX0ldCiAgICAgIHdoZXJlIGR4ID0geF9qIC0geF9pLCBkeSA9IHlfaiAtIHlfaSwgZGVsdGFfSSA9IElfaiAtIElfaS4KICAgICIiIgoKICAgIGRlZiBfX2luaXRfXyhzZWxmLCBpbWdfc2l6ZTogaW50ID0gNDgpIC0+IE5vbmU6CiAgICAgICAgc3VwZXIoKS5fX2luaXRfXygpCiAgICAgICAgc2VsZi5pbWdfc2l6ZSA9IGltZ19zaXplCiAgICAgICAgc2VsZi5udW1fcGl4ZWxzID0gaW1nX3NpemUgKiBpbWdfc2l6ZQogICAgICAgIHNlbGYubnVtX25laWdoYm9ycyA9IDgKCiAgICAgICAgIyA4IG5laWdoYm9yIHJlbGF0aXZlIHJvdywgY29sIG9mZnNldHM6IChkciwgZGMpCiAgICAgICAgIyB3aGVyZSByX2ogPSByX2kgKyBkciwgY19qID0gY19pICsgZGMKICAgICAgICBzZWxmLm5laWdoYm9yX29mZnNldHMgPSBbCiAgICAgICAgICAgICgtMSwgLTEpLCAgIyAwOiBOVwogICAgICAgICAgICAoLTEsIDApLCAgIyAxOiBOCiAgICAgICAgICAgICgtMSwgMSksICAjIDI6IE5FCiAgICAgICAgICAgICgwLCAtMSksICAjIDM6IFcKICAgICAgICAgICAgKDAsIDEpLCAgIyA0OiBFCiAgICAgICAgICAgICgxLCAtMSksICAjIDU6IFNXCiAgICAgICAgICAgICgxLCAwKSwgICMgNjogUwogICAgICAgICAgICAoMSwgMSksICAjIDc6IFNFCiAgICAgICAgXQoKICAgICAgICBuZWlnaGJvcl9pZHggPSB0b3JjaC56ZXJvcygKICAgICAgICAgICAgKHNlbGYubnVtX3BpeGVscywgc2VsZi5udW1fbmVpZ2hib3JzKSwgZHR5cGU9dG9yY2gubG9uZwogICAgICAgICkKICAgICAgICBuZWlnaGJvcl9tYXNrID0gdG9yY2guemVyb3MoCiAgICAgICAgICAgIChzZWxmLm51bV9waXhlbHMsIHNlbGYubnVtX25laWdoYm9ycyksIGR0eXBlPXRvcmNoLmJvb2wKICAgICAgICApCiAgICAgICAgZ2VvbV9lZGdlcyA9IHRvcmNoLnplcm9zKAogICAgICAgICAgICAoc2VsZi5udW1fcGl4ZWxzLCBzZWxmLm51bV9uZWlnaGJvcnMsIDMpLCBkdHlwZT10b3JjaC5mbG9hdDMyCiAgICAgICAgKQoKICAgICAgICBjb29yZHMgPSB0b3JjaC5saW5zcGFjZSgtMS4wLCAxLjAsIGltZ19zaXplKQoKICAgICAgICBmb3Igcl9pIGluIHJhbmdlKGltZ19zaXplKToKICAgICAgICAgICAgZm9yIGNfaSBpbiByYW5nZShpbWdfc2l6ZSk6CiAgICAgICAgICAgICAgICBpID0gcl9pICogaW1nX3NpemUgKyBjX2kKICAgICAgICAgICAgICAgIHhfaSA9IGNvb3Jkc1tjX2ldLml0ZW0oKQogICAgICAgICAgICAgICAgeV9pID0gY29vcmRzW3JfaV0uaXRlbSgpCgogICAgICAgICAgICAgICAgZm9yIGssIChkciwgZGMpIGluIGVudW1lcmF0ZShzZWxmLm5laWdoYm9yX29mZnNldHMpOgogICAgICAgICAgICAgICAgICAgIHJfaiA9IHJfaSArIGRyCiAgICAgICAgICAgICAgICAgICAgY19qID0gY19pICsgZGMKCiAgICAgICAgICAgICAgICAgICAgaWYgMCA8PSByX2ogPCBpbWdfc2l6ZSBhbmQgMCA8PSBjX2ogPCBpbWdfc2l6ZToKICAgICAgICAgICAgICAgICAgICAgICAgaiA9IHJfaiAqIGltZ19zaXplICsgY19qCiAgICAgICAgICAgICAgICAgICAgICAgIG5laWdoYm9yX2lkeFtpLCBrXSA9IGoKICAgICAgICAgICAgICAgICAgICAgICAgbmVpZ2hib3JfbWFza1tpLCBrXSA9IFRydWUKCiAgICAgICAgICAgICAgICAgICAgICAgIHhfaiA9IGNvb3Jkc1tjX2pdLml0ZW0oKQogICAgICAgICAgICAgICAgICAgICAgICB5X2ogPSBjb29yZHNbcl9qXS5pdGVtKCkKCiAgICAgICAgICAgICAgICAgICAgICAgIGR4ID0geF9qIC0geF9pCiAgICAgICAgICAgICAgICAgICAgICAgIGR5ID0geV9qIC0geV9pCiAgICAgICAgICAgICAgICAgICAgICAgIGRpc3QgPSAoZHgqKjIgKyBkeSoqMikgKiogMC41CgogICAgICAgICAgICAgICAgICAgICAgICBnZW9tX2VkZ2VzW2ksIGssIDBdID0gZHgKICAgICAgICAgICAgICAgICAgICAgICAgZ2VvbV9lZGdlc1tpLCBrLCAxXSA9IGR5CiAgICAgICAgICAgICAgICAgICAgICAgIGdlb21fZWRnZXNbaSwgaywgMl0gPSBkaXN0CiAgICAgICAgICAgICAgICAgICAgZWxzZToKICAgICAgICAgICAgICAgICAgICAgICAgbmVpZ2hib3JfaWR4W2ksIGtdID0gKAogICAgICAgICAgICAgICAgICAgICAgICAgICAgaSAgIyBzZWxmLWxvb3AgcGFkIHRvIGF2b2lkIG91dC1vZi1ib3VuZHMgZ2F0aGVyCiAgICAgICAgICAgICAgICAgICAgICAgICkKICAgICAgICAgICAgICAgICAgICAgICAgbmVpZ2hib3JfbWFza1tpLCBrXSA9IEZhbHNlCiAgICAgICAgICAgICAgICAgICAgICAgIGdlb21fZWRnZXNbaSwga10gPSAwLjAKCiAgICAgICAgc2VsZi5yZWdpc3Rlcl9idWZmZXIoIm5laWdoYm9yX2lkeCIsIG5laWdoYm9yX2lkeCkgICMgWzIzMDQsIDhdCiAgICAgICAgc2VsZi5yZWdpc3Rlcl9idWZmZXIoIm5laWdoYm9yX21hc2siLCBuZWlnaGJvcl9tYXNrKSAgIyBbMjMwNCwgOF0KICAgICAgICBzZWxmLnJlZ2lzdGVyX2J1ZmZlcigiZ2VvbV9lZGdlcyIsIGdlb21fZWRnZXMpICAjIFsyMzA0LCA4LCAzXQoKICAgICAgICAjIFByZS1leHBhbmRlZCBpbmRpY2VzIGZvciBiYXRjaGVkIGdhdGhlcmluZyBbMjMwNCwgOCwgMV0KICAgICAgICBJX2dhdGhlcl9pZHggPSBuZWlnaGJvcl9pZHgudW5zcXVlZXplKC0xKQogICAgICAgIHNlbGYucmVnaXN0ZXJfYnVmZmVyKCJJX2dhdGhlcl9pZHgiLCBJX2dhdGhlcl9pZHgpCgogICAgZGVmIGNvbXB1dGVfZWRnZV9mZWF0dXJlcyhzZWxmLCBwaXhlbF9pbnRlbnNpdHk6IHRvcmNoLlRlbnNvcikgLT4gdG9yY2guVGVuc29yOgogICAgICAgICIiIkNvbXB1dGUgdGhlIDVEIGVkZ2UgZmVhdHVyZXMgZm9yIGRpcmVjdGVkIGVkZ2VzIGogLT4gaS4KCiAgICAgICAgQXJnczoKICAgICAgICAgICAgcGl4ZWxfaW50ZW5zaXR5OiBbQiwgMjMwNCwgMV0gbm9ybWFsaXplZCBwaXhlbCBpbnRlbnNpdGllcy4KCiAgICAgICAgUmV0dXJuczoKICAgICAgICAgICAgZWRnZV9mZWF0dXJlczogW0IsIDIzMDQsIDgsIDVdCiAgICAgICAgICAgICAgW2R4LCBkeSwgZGlzdGFuY2UsIGRlbHRhX0ksIGFic19kZWx0YV9JXQogICAgICAgICIiIgogICAgICAgIEIsIE4sIF8gPSBwaXhlbF9pbnRlbnNpdHkuc2hhcGUKICAgICAgICBhc3NlcnQgTiA9PSBzZWxmLm51bV9waXhlbHMKCiAgICAgICAgIyBFeHBhbmQgc3RhdGljIGdlb21ldHJ5OiBbMSwgMjMwNCwgOCwgM10gLT4gW0IsIDIzMDQsIDgsIDNdCiAgICAgICAgZ2VvbSA9IHNlbGYuZ2VvbV9lZGdlcy51bnNxdWVlemUoMCkuZXhwYW5kKEIsIC0xLCAtMSwgLTEpCgogICAgICAgICMgRWZmaWNpZW50IGZsYXR0ZW4tZ2F0aGVyIGFjcm9zcyBiYXRjaAogICAgICAgICMgcGl4ZWxfaW50ZW5zaXR5OiBbQiwgMjMwNCwgMV0KICAgICAgICAjIElfaiBmb3IgcmVjZWl2ZXIgaTogZ2F0aGVyIHNlbmRlciBuZWlnaGJvciBpbmRleAogICAgICAgIElfaiA9IHBpeGVsX2ludGVuc2l0eVs6LCBzZWxmLm5laWdoYm9yX2lkeCwgOl0gICMgW0IsIDIzMDQsIDgsIDFdCiAgICAgICAgSV9pID0gcGl4ZWxfaW50ZW5zaXR5LnVuc3F1ZWV6ZSgyKSAgIyBbQiwgMjMwNCwgMSwgMV0KCiAgICAgICAgZGVsdGFfSSA9IElfaiAtIElfaSAgIyBbQiwgMjMwNCwgOCwgMV0KICAgICAgICBhYnNfZGVsdGFfSSA9IHRvcmNoLmFicyhkZWx0YV9JKQoKICAgICAgICAjIEVkZ2UgZmVhdHVyZXM6IFtkeCwgZHksIGRpc3QsIGRlbHRhX0ksIGFic19kZWx0YV9JXSAtPiBbQiwgMjMwNCwgOCwgNV0KICAgICAgICBlZGdlX2ZlYXRzID0gdG9yY2guY2F0KFtnZW9tLCBkZWx0YV9JLCBhYnNfZGVsdGFfSV0sIGRpbT0tMSkKCiAgICAgICAgIyBaZXJvIG91dCBpbnZhbGlkIHBhZGRlZCBib3VuZGFyeSBlZGdlcwogICAgICAgIG1hc2sgPSBzZWxmLm5laWdoYm9yX21hc2sudW5zcXVlZXplKDApLnVuc3F1ZWV6ZSgtMSkgICMgWzEsIDIzMDQsIDgsIDFdCiAgICAgICAgZWRnZV9mZWF0cyA9IGVkZ2VfZmVhdHMgKiBtYXNrLmZsb2F0KCkKICAgICAgICByZXR1cm4gZWRnZV9mZWF0cwo=', 'mpg_fer_v2_3/kaggle.py': 'IiIiRmFpbC1jbG9zZWQgS2FnZ2xlIGRhdGEgYW5kIGV4cGxpY2l0IE1QRy1GRVIgdjIuMyByZXN1bWUgcmVzb2x1dGlvbi4iIiIKCmZyb20gX19mdXR1cmVfXyBpbXBvcnQgYW5ub3RhdGlvbnMKCmltcG9ydCBvcwpmcm9tIHBhdGhsaWIgaW1wb3J0IFBhdGgKaW1wb3J0IGpzb24KCmZyb20gLmNoZWNrcG9pbnQgaW1wb3J0IGZpbmRfbGF0ZXN0X3ZhbGlkX3NuYXBzaG90LCBzaGEyNTZfZmlsZQoKZnJvbSAuZGF0YSBpbXBvcnQgdmFsaWRhdGVfc3BsaXRfcGF0aHMKCgpEQVRBU0VUX1NMVUcgPSAiZG9kdXlxdXluaWkvZmVyMTMtc3BsaXQiCkFUVEFDSEVEX1JPT1RTID0gKAogICAgUGF0aCgiL2thZ2dsZS9pbnB1dC9kYXRhc2V0cy9kb2R1eXF1eW5paS9mZXIxMy1zcGxpdC9mZXIxMy1zcGxpdCIpLAogICAgUGF0aCgiL2thZ2dsZS9pbnB1dC9mZXIxMy1zcGxpdC9mZXIxMy1zcGxpdCIpLAogICAgUGF0aCgiL2thZ2dsZS9pbnB1dC9mZXIxMy1zcGxpdCIpLAopCgoKZGVmIF9jb21wbGV0ZV9zcGxpdF90cmlwbGV0KHJvb3Q6IFBhdGgpIC0+IHR1cGxlW1BhdGgsIFBhdGgsIFBhdGhdIHwgTm9uZToKICAgIHBhdGhzID0gKHJvb3QgLyAidHJhaW4uY3N2Iiwgcm9vdCAvICJ2YWwuY3N2Iiwgcm9vdCAvICJ0ZXN0LmNzdiIpCiAgICByZXR1cm4gcGF0aHMgaWYgYWxsKHBhdGguaXNfZmlsZSgpIGZvciBwYXRoIGluIHBhdGhzKSBlbHNlIE5vbmUKCgpkZWYgcmVzb2x2ZV9rYWdnbGVfc3BsaXRzKAogICAgZG93bmxvYWRfcm9vdDogc3RyIHwgUGF0aCA9ICIva2FnZ2xlL3dvcmtpbmcvZmVyMTMtc3BsaXQtZG93bmxvYWQiLAopIC0+IHR1cGxlW1BhdGgsIFBhdGgsIFBhdGhdOgogICAgIiIiVXNlIGFuIGF0dGFjaGVkIGRhdGFzZXQgb3IgZG93bmxvYWQgaXQgd2l0aCBLYWdnbGUgU2VjcmV0cy4KCiAgICBDcmVkZW50aWFscyBhcmUgcmVhZCBvbmx5IGZyb20gYGBLQUdHTEVfVVNFUk5BTUVgYCBhbmQgYGBLQUdHTEVfS0VZYGAKICAgIHNlY3JldHMuIFRoZSBrZXkgaXMgbmV2ZXIgcHJpbnRlZCBvciB3cml0dGVuIHRvIGBga2FnZ2xlLmpzb25gYC4KICAgICIiIgogICAgZm9yIHJvb3QgaW4gQVRUQUNIRURfUk9PVFM6CiAgICAgICAgdHJpcGxldCA9IF9jb21wbGV0ZV9zcGxpdF90cmlwbGV0KHJvb3QpCiAgICAgICAgaWYgdHJpcGxldCBpcyBub3QgTm9uZToKICAgICAgICAgICAgcmV0dXJuIHZhbGlkYXRlX3NwbGl0X3BhdGhzKCp0cmlwbGV0KSAgIyB0eXBlOiBpZ25vcmVbcmV0dXJuLXZhbHVlXQoKICAgIHRyeToKICAgICAgICBmcm9tIGthZ2dsZV9zZWNyZXRzIGltcG9ydCBVc2VyU2VjcmV0c0NsaWVudAoKICAgICAgICBzZWNyZXRzID0gVXNlclNlY3JldHNDbGllbnQoKQogICAgICAgIHVzZXJuYW1lID0gc2VjcmV0cy5nZXRfc2VjcmV0KCJLQUdHTEVfVVNFUk5BTUUiKQogICAgICAgIGtleSA9IHNlY3JldHMuZ2V0X3NlY3JldCgiS0FHR0xFX0tFWSIpCiAgICBleGNlcHQgRXhjZXB0aW9uIGFzIGV4YzoKICAgICAgICByYWlzZSBSdW50aW1lRXJyb3IoCiAgICAgICAgICAgICJGRVIxMyBzcGxpdCBpcyBub3QgYXR0YWNoZWQuIEFkZCBLYWdnbGUgU2VjcmV0cyBLQUdHTEVfVVNFUk5BTUUgIgogICAgICAgICAgICAiYW5kIEtBR0dMRV9LRVkgdG8gZG93bmxvYWQgZG9kdXlxdXluaWkvZmVyMTMtc3BsaXQuIgogICAgICAgICkgZnJvbSBleGMKICAgIGlmIG5vdCB1c2VybmFtZSBvciBub3Qga2V5OgogICAgICAgIHJhaXNlIFJ1bnRpbWVFcnJvcigKICAgICAgICAgICAgIkZFUjEzIHNwbGl0IGlzIG5vdCBhdHRhY2hlZCBhbmQgS2FnZ2xlIGNyZWRlbnRpYWxzIGFyZSBpbmNvbXBsZXRlIgogICAgICAgICkKCiAgICBvcy5lbnZpcm9uWyJLQUdHTEVfVVNFUk5BTUUiXSA9IHVzZXJuYW1lCiAgICBvcy5lbnZpcm9uWyJLQUdHTEVfS0VZIl0gPSBrZXkKICAgIHRhcmdldCA9IFBhdGgoZG93bmxvYWRfcm9vdCkKICAgIHRhcmdldC5ta2RpcihwYXJlbnRzPVRydWUsIGV4aXN0X29rPVRydWUpCiAgICB0cnk6CiAgICAgICAgZnJvbSBrYWdnbGUuYXBpLmthZ2dsZV9hcGlfZXh0ZW5kZWQgaW1wb3J0IEthZ2dsZUFwaQoKICAgICAgICBhcGkgPSBLYWdnbGVBcGkoKQogICAgICAgIGFwaS5hdXRoZW50aWNhdGUoKQogICAgICAgIGFwaS5kYXRhc2V0X2Rvd25sb2FkX2ZpbGVzKERBVEFTRVRfU0xVRywgcGF0aD10YXJnZXQsIHVuemlwPVRydWUsIHF1aWV0PUZhbHNlKQogICAgZXhjZXB0IEV4Y2VwdGlvbiBhcyBleGM6CiAgICAgICAgcmFpc2UgUnVudGltZUVycm9yKGYiRmFpbGVkIHRvIGRvd25sb2FkIEthZ2dsZSBkYXRhc2V0IHtEQVRBU0VUX1NMVUd9IikgZnJvbSBleGMKCiAgICBjYW5kaWRhdGVzID0gW3RhcmdldCwgdGFyZ2V0IC8gImZlcjEzLXNwbGl0Il0KICAgIGNhbmRpZGF0ZXMuZXh0ZW5kKHBhdGgucGFyZW50IGZvciBwYXRoIGluIHRhcmdldC5yZ2xvYigidHJhaW4uY3N2IikpCiAgICBmb3Igcm9vdCBpbiBjYW5kaWRhdGVzOgogICAgICAgIHRyaXBsZXQgPSBfY29tcGxldGVfc3BsaXRfdHJpcGxldChyb290KQogICAgICAgIGlmIHRyaXBsZXQgaXMgbm90IE5vbmU6CiAgICAgICAgICAgIHJldHVybiB2YWxpZGF0ZV9zcGxpdF9wYXRocygqdHJpcGxldCkgICMgdHlwZTogaWdub3JlW3JldHVybi12YWx1ZV0KICAgIHJhaXNlIFJ1bnRpbWVFcnJvcigKICAgICAgICBmIkRvd25sb2FkZWQge0RBVEFTRVRfU0xVR30sIGJ1dCB0cmFpbi5jc3YvdmFsLmNzdi90ZXN0LmNzdiB3ZXJlIG5vdCBmb3VuZCB0b2dldGhlciIKICAgICkKCgpkZWYgcmVzb2x2ZV9yZXN1bWVfYXJ0aWZhY3QoCiAgICBtb2RlOiBzdHIgPSAiYXV0byIsCiAgICBleHBsaWNpdF9wYXRoOiBzdHIgfCBQYXRoIHwgTm9uZSA9IE5vbmUsCiAgICBpbnB1dF9yb290OiBzdHIgfCBQYXRoID0gIi9rYWdnbGUvaW5wdXQiLAopIC0+IHR1cGxlW1BhdGggfCBOb25lLCBzdHIgfCBOb25lXToKICAgICIiIlJlc29sdmUgb25seSBhbiBleHBsaWNpdCBvciBjbGVhcmx5IG5hbWVkIHYyLjMgcmVzdW1lIGF0dGFjaG1lbnQuCgogICAgQXV0by1kaXNjb3ZlcnkgaXMgZGVsaWJlcmF0ZWx5IHJlc3RyaWN0ZWQgdG8gZGF0YXNldHMgd2hvc2UgZGlyZWN0b3J5IG5hbWUKICAgIHN0YXJ0cyB3aXRoIGBgbXBnLWZlci12Mi0zLXJlc3VtZWBgLiBJdCBjYW5ub3QgZGlzY292ZXIgdjEvdjIvdjIuMQogICAgY2hlY2twb2ludHMgb3IgYSBnZW5lcmljIGBgYmVzdF92YWxfYWNjLnB0YGAuCiAgICAiIiIKICAgIG5vcm1hbGl6ZWQgPSBtb2RlLmxvd2VyKCkKICAgIGlmIG5vcm1hbGl6ZWQgbm90IGluIHsiYXV0byIsICJmcmVzaCIsICJyZXF1aXJlZCJ9OgogICAgICAgIHJhaXNlIFZhbHVlRXJyb3IoIlJFU1VNRV9NT0RFIG11c3QgYmUgYXV0bywgZnJlc2gsIG9yIHJlcXVpcmVkIikKICAgIGlmIG5vcm1hbGl6ZWQgPT0gImZyZXNoIjoKICAgICAgICByZXR1cm4gTm9uZSwgTm9uZQogICAgaWYgZXhwbGljaXRfcGF0aCBpcyBub3QgTm9uZToKICAgICAgICBjYW5kaWRhdGVzID0gW1BhdGgoZXhwbGljaXRfcGF0aCldCiAgICBlbHNlOgogICAgICAgIHJvb3QgPSBQYXRoKGlucHV0X3Jvb3QpCiAgICAgICAgY2FuZGlkYXRlcyA9IHNvcnRlZCgKICAgICAgICAgICAgcGF0aAogICAgICAgICAgICBmb3IgcGF0aCBpbiByb290LnJnbG9iKCJyZXN1bWVfbGF0ZXN0LnB0IikKICAgICAgICAgICAgaWYgYW55KAogICAgICAgICAgICAgICAgcGFydC5zdGFydHN3aXRoKCJtcGctZmVyLXYyLTMtcmVzdW1lIikKICAgICAgICAgICAgICAgIGZvciBwYXJ0IGluIHBhdGgucmVsYXRpdmVfdG8ocm9vdCkucGFydHNbOi0xXQogICAgICAgICAgICApCiAgICAgICAgKSBpZiByb290LmV4aXN0cygpIGVsc2UgW10KICAgIGlmIG5vdCBjYW5kaWRhdGVzOgogICAgICAgIGlmIG5vcm1hbGl6ZWQgPT0gInJlcXVpcmVkIjoKICAgICAgICAgICAgcmFpc2UgRmlsZU5vdEZvdW5kRXJyb3IoIk5vIGV4cGxpY2l0IE1QRy1GRVIgdjIuMyByZXN1bWUgYXJ0aWZhY3QgYXR0YWNoZWQiKQogICAgICAgIHJldHVybiBOb25lLCBOb25lCiAgICBpZiBsZW4oY2FuZGlkYXRlcykgIT0gMToKICAgICAgICByYWlzZSBSdW50aW1lRXJyb3IoCiAgICAgICAgICAgIGYiRXhwZWN0ZWQgZXhhY3RseSBvbmUgdjIuMyByZXN1bWUgYXJ0aWZhY3QsIGZvdW5kIHtsZW4oY2FuZGlkYXRlcyl9IgogICAgICAgICkKICAgIGNoZWNrcG9pbnQgPSBjYW5kaWRhdGVzWzBdCiAgICBpZiBjaGVja3BvaW50Lm5hbWUgIT0gInJlc3VtZV9sYXRlc3QucHQiIG9yIG5vdCBjaGVja3BvaW50LmlzX2ZpbGUoKToKICAgICAgICByYWlzZSBSdW50aW1lRXJyb3IoZiJJbnZhbGlkIHYyLjMgcmVzdW1lIGNoZWNrcG9pbnQgcGF0aDoge2NoZWNrcG9pbnR9IikKICAgIG1ldGFkYXRhX3BhdGggPSBjaGVja3BvaW50LndpdGhfbmFtZSgicmVzdW1lX2xhdGVzdC5qc29uIikKICAgIGlmIG5vdCBtZXRhZGF0YV9wYXRoLmlzX2ZpbGUoKToKICAgICAgICByYWlzZSBSdW50aW1lRXJyb3IoIlJlc3VtZSBtZXRhZGF0YSByZXN1bWVfbGF0ZXN0Lmpzb24gaXMgcmVxdWlyZWQiKQogICAgbWV0YWRhdGEgPSBqc29uLmxvYWRzKG1ldGFkYXRhX3BhdGgucmVhZF90ZXh0KGVuY29kaW5nPSJ1dGYtOCIpKQogICAgZXhwZWN0ZWQgPSBtZXRhZGF0YS5nZXQoImNoZWNrcG9pbnRfc2hhMjU2IikKICAgIGFjdHVhbCA9IHNoYTI1Nl9maWxlKGNoZWNrcG9pbnQpCiAgICBpZiBub3QgZXhwZWN0ZWQgb3IgYWN0dWFsICE9IGV4cGVjdGVkOgogICAgICAgIGZhbGxiYWNrID0gZmluZF9sYXRlc3RfdmFsaWRfc25hcHNob3QoCiAgICAgICAgICAgIGNoZWNrcG9pbnQucGFyZW50LAogICAgICAgICAgICBleHBlY3RlZF9pZGVudGl0eT1tZXRhZGF0YSwKICAgICAgICApCiAgICAgICAgZGV0YWlsID0gKAogICAgICAgICAgICAibm9uZSIKICAgICAgICAgICAgaWYgZmFsbGJhY2sgaXMgTm9uZQogICAgICAgICAgICBlbHNlIGYie2ZhbGxiYWNrWzBdfSAoc2hhMjU2PXtmYWxsYmFja1sxXX0pIgogICAgICAgICkKICAgICAgICByYWlzZSBSdW50aW1lRXJyb3IoCiAgICAgICAgICAgICJBdHRhY2hlZCByZXN1bWUgY2hlY2twb2ludCBTSEEtMjU2IG1pc21hdGNoOyByZWZ1c2luZyBhdXRvbWF0aWMgIgogICAgICAgICAgICBmImZhbGxiYWNrLiBNb3N0IHJlY2VudCB2YWxpZCBpbW11dGFibGUgc25hcHNob3Q6IHtkZXRhaWx9IgogICAgICAgICkKICAgIHJldHVybiBjaGVja3BvaW50LCBhY3R1YWwK', 'mpg_fer_v2_3/losses.py': 'IiIiRGlmZmVyZW50aWFibGUgb2JqZWN0aXZlcyBpbnRyb2R1Y2VkIGJ5IE1QRy1GRVIgdjIuMi4iIiIKCmZyb20gX19mdXR1cmVfXyBpbXBvcnQgYW5ub3RhdGlvbnMKCmltcG9ydCBtYXRoCmltcG9ydCB0b3JjaAppbXBvcnQgdG9yY2gubm4uZnVuY3Rpb25hbCBhcyBGCgoKZGVmIG1vdGlmX211dHVhbF9pbmZvcm1hdGlvbl9sb3NzKAogICAgYXNzaWdubWVudHM6IHRvcmNoLlRlbnNvciwgYmV0YTogZmxvYXQgPSAxLjAsIGVwczogZmxvYXQgPSAxZS04CikgLT4gdHVwbGVbdG9yY2guVGVuc29yLCBkaWN0W3N0ciwgdG9yY2guVGVuc29yXV06CiAgICAiIiJSZXR1cm4gSChNfEgpIC0gYmV0YSpIKE0pLCBub3JtYWxpemVkIGJ5IGxvZyhudW1iZXIgb2YgbW90aWZzKS4iIiIKICAgIGlmIGFzc2lnbm1lbnRzLm5kaW0gIT0gMyBvciBhc3NpZ25tZW50cy5zaGFwZVstMV0gPCAyOgogICAgICAgIHJhaXNlIFZhbHVlRXJyb3IoImFzc2lnbm1lbnRzIG11c3QgaGF2ZSBzaGFwZSBbYmF0Y2gsIG5vZGVzLCBtb3RpZnM+PTJdIikKICAgIHByb2JhYmlsaXRpZXMgPSBhc3NpZ25tZW50cy5jbGFtcF9taW4oZXBzKQogICAgbG9jYWxfZW50cm9weV9wZXJfbm9kZSA9IC0ocHJvYmFiaWxpdGllcyAqIHByb2JhYmlsaXRpZXMubG9nKCkpLnN1bShkaW09LTEpCiAgICBoX2xvY2FsX3JhdyA9IGxvY2FsX2VudHJvcHlfcGVyX25vZGUubWVhbigpCiAgICB1c2FnZSA9IGFzc2lnbm1lbnRzLm1lYW4oZGltPSgwLCAxKSkuY2xhbXBfbWluKGVwcykKICAgIGhfZ2xvYmFsX3JhdyA9IC0odXNhZ2UgKiB1c2FnZS5sb2coKSkuc3VtKCkKICAgIG5vcm1hbGl6ZXIgPSBtYXRoLmxvZyhhc3NpZ25tZW50cy5zaGFwZVstMV0pCiAgICBoX2xvY2FsX25vcm1hbGl6ZWQgPSBoX2xvY2FsX3JhdyAvIG5vcm1hbGl6ZXIKICAgIGhfZ2xvYmFsX25vcm1hbGl6ZWQgPSBoX2dsb2JhbF9yYXcgLyBub3JtYWxpemVyCiAgICBsb3NzID0gaF9sb2NhbF9ub3JtYWxpemVkIC0gYmV0YSAqIGhfZ2xvYmFsX25vcm1hbGl6ZWQKICAgIHJldHVybiBsb3NzLCB7CiAgICAgICAgIkhfbG9jYWxfcmF3IjogaF9sb2NhbF9yYXcsCiAgICAgICAgIkhfbG9jYWxfbm9ybWFsaXplZCI6IGhfbG9jYWxfbm9ybWFsaXplZCwKICAgICAgICAiSF9nbG9iYWxfcmF3IjogaF9nbG9iYWxfcmF3LAogICAgICAgICJIX2dsb2JhbF9ub3JtYWxpemVkIjogaF9nbG9iYWxfbm9ybWFsaXplZCwKICAgICAgICAiTF9NSSI6IGxvc3MsCiAgICAgICAgInByb3RvdHlwZV91dGlsaXphdGlvbiI6IHVzYWdlLAogICAgfQoKCmRlZiBzeW1tZXRyaWNfanNfZGl2ZXJnZW5jZSgKICAgIGxvZ2l0c19hOiB0b3JjaC5UZW5zb3IsIGxvZ2l0c19iOiB0b3JjaC5UZW5zb3IsIGVwczogZmxvYXQgPSAxZS04CikgLT4gdG9yY2guVGVuc29yOgogICAgIiIiTWVhbiBKZW5zZW4tU2hhbm5vbiBkaXZlcmdlbmNlIGJldHdlZW4gdHdvIGNsYXNzaWZpZXIgZGlzdHJpYnV0aW9ucy4iIiIKICAgIHAgPSBGLnNvZnRtYXgobG9naXRzX2EsIGRpbT0tMSkKICAgIHEgPSBGLnNvZnRtYXgobG9naXRzX2IsIGRpbT0tMSkKICAgIG0gPSAwLjUgKiAocCArIHEpCiAgICBsb2dfbSA9IG0uY2xhbXBfbWluKGVwcykubG9nKCkKICAgIGtsX3BtID0gKHAgKiAoRi5sb2dfc29mdG1heChsb2dpdHNfYSwgZGltPS0xKSAtIGxvZ19tKSkuc3VtKGRpbT0tMSkKICAgIGtsX3FtID0gKHEgKiAoRi5sb2dfc29mdG1heChsb2dpdHNfYiwgZGltPS0xKSAtIGxvZ19tKSkuc3VtKGRpbT0tMSkKICAgIHJldHVybiAwLjUgKiAoa2xfcG0gKyBrbF9xbSkubWVhbigpCgoKZGVmIHN1cGVydmlzZWRfY29udHJhc3RpdmVfbG9zcygKICAgIGVtYmVkZGluZ3M6IHRvcmNoLlRlbnNvciwKICAgIGxhYmVsczogdG9yY2guVGVuc29yLAogICAgdGVtcGVyYXR1cmU6IGZsb2F0ID0gMC4xMCwKKSAtPiB0dXBsZVt0b3JjaC5UZW5zb3IsIGRpY3Rbc3RyLCB0b3JjaC5UZW5zb3JdXToKICAgICIiIlN0YW5kYXJkIGluLWJhdGNoIFN1cENvbiB3aXRoIHNhZmUgZXhjbHVzaW9uIG9mIGludmFsaWQgYW5jaG9ycy4iIiIKICAgIGlmIGVtYmVkZGluZ3MubmRpbSAhPSAyOgogICAgICAgIHJhaXNlIFZhbHVlRXJyb3IoImVtYmVkZGluZ3MgbXVzdCBoYXZlIHNoYXBlIFtiYXRjaCwgZGltZW5zaW9uXSIpCiAgICBpZiBsYWJlbHMubmRpbSAhPSAxIG9yIGxhYmVscy5zaGFwZVswXSAhPSBlbWJlZGRpbmdzLnNoYXBlWzBdOgogICAgICAgIHJhaXNlIFZhbHVlRXJyb3IoImxhYmVscyBtdXN0IGhhdmUgc2hhcGUgW2JhdGNoXSIpCiAgICBpZiB0ZW1wZXJhdHVyZSA8PSAwLjA6CiAgICAgICAgcmFpc2UgVmFsdWVFcnJvcigidGVtcGVyYXR1cmUgbXVzdCBiZSBwb3NpdGl2ZSIpCgogICAgbm9ybWFsaXplZCA9IEYubm9ybWFsaXplKGVtYmVkZGluZ3MsIGRpbT0tMSkKICAgIGJhdGNoID0gZW1iZWRkaW5ncy5zaGFwZVswXQogICAgemVybyA9IGVtYmVkZGluZ3Muc3VtKCkgKiAwLjAKICAgIGlmIGJhdGNoIDwgMjoKICAgICAgICByZXR1cm4gemVybywgewogICAgICAgICAgICAidmFsaWRfc3VwY29uX2FuY2hvcl9mcmFjdGlvbiI6IHplcm8uZGV0YWNoKCksCiAgICAgICAgICAgICJtZWFuX3Bvc2l0aXZlX2NvdW50IjogemVyby5kZXRhY2goKSwKICAgICAgICB9CgogICAgc2VsZl9tYXNrID0gdG9yY2guZXllKGJhdGNoLCBkdHlwZT10b3JjaC5ib29sLCBkZXZpY2U9ZW1iZWRkaW5ncy5kZXZpY2UpCiAgICBwb3NpdGl2ZV9tYXNrID0gbGFiZWxzWzosIE5vbmVdLmVxKGxhYmVsc1tOb25lLCA6XSkgJiB+c2VsZl9tYXNrCiAgICBwb3NpdGl2ZV9jb3VudCA9IHBvc2l0aXZlX21hc2suc3VtKGRpbT0xKQogICAgdmFsaWQgPSBwb3NpdGl2ZV9jb3VudCA+IDAKCiAgICBzaW1pbGFyaXR5ID0gbm9ybWFsaXplZCBAIG5vcm1hbGl6ZWQudCgpIC8gdGVtcGVyYXR1cmUKICAgIHNpbWlsYXJpdHkgPSBzaW1pbGFyaXR5IC0gc2ltaWxhcml0eS5tYXgoZGltPTEsIGtlZXBkaW09VHJ1ZSkudmFsdWVzLmRldGFjaCgpCiAgICBkZW5vbWluYXRvcl9sb2dpdHMgPSBzaW1pbGFyaXR5Lm1hc2tlZF9maWxsKHNlbGZfbWFzaywgLXRvcmNoLmluZikKICAgIGxvZ19wcm9iID0gc2ltaWxhcml0eSAtIHRvcmNoLmxvZ3N1bWV4cChkZW5vbWluYXRvcl9sb2dpdHMsIGRpbT0xLCBrZWVwZGltPVRydWUpCiAgICBwZXJfYW5jaG9yID0gLSgKICAgICAgICBsb2dfcHJvYi5tYXNrZWRfZmlsbCh+cG9zaXRpdmVfbWFzaywgMC4wKS5zdW0oZGltPTEpCiAgICAgICAgLyBwb3NpdGl2ZV9jb3VudC5jbGFtcF9taW4oMSkKICAgICkKICAgIGxvc3MgPSBwZXJfYW5jaG9yW3ZhbGlkXS5tZWFuKCkgaWYgYm9vbCh2YWxpZC5hbnkoKSkgZWxzZSB6ZXJvCiAgICB2YWxpZF9mcmFjdGlvbiA9IHZhbGlkLnRvKGVtYmVkZGluZ3MuZHR5cGUpLm1lYW4oKQogICAgbWVhbl9wb3NpdGl2ZV9jb3VudCA9ICgKICAgICAgICBwb3NpdGl2ZV9jb3VudFt2YWxpZF0udG8oZW1iZWRkaW5ncy5kdHlwZSkubWVhbigpCiAgICAgICAgaWYgYm9vbCh2YWxpZC5hbnkoKSkKICAgICAgICBlbHNlIHplcm8uZGV0YWNoKCkKICAgICkKICAgIHJldHVybiBsb3NzLCB7CiAgICAgICAgInZhbGlkX3N1cGNvbl9hbmNob3JfZnJhY3Rpb24iOiB2YWxpZF9mcmFjdGlvbiwKICAgICAgICAibWVhbl9wb3NpdGl2ZV9jb3VudCI6IG1lYW5fcG9zaXRpdmVfY291bnQsCiAgICB9Cg==', 'mpg_fer_v2_3/model.py': '...', 'mpg_fer_v2_3/motif.py': 'IiIiRGlmZmVyZW50aWFibGUgbXVsdGlzY2FsZSBzcGF0aWFsIG1vdGlmIGNvbXBvc2l0aW9uIGZvciBNUEctRkVSIHYyLjIuIiIiCgpmcm9tIF9fZnV0dXJlX18gaW1wb3J0IGFubm90YXRpb25zCgppbXBvcnQgbWF0aAppbXBvcnQgdG9yY2gKaW1wb3J0IHRvcmNoLm5uIGFzIG5uCmltcG9ydCB0b3JjaC5ubi5mdW5jdGlvbmFsIGFzIEYKCmZyb20gLmxvc3NlcyBpbXBvcnQgbW90aWZfbXV0dWFsX2luZm9ybWF0aW9uX2xvc3MKCgpkZWYgc2NoZWR1bGVkX21vdGlmX3RlbXBlcmF0dXJlKAogICAgZXBvY2g6IGludCwKICAgIHRhdV9zdGFydDogZmxvYXQgPSAwLjcwLAogICAgdGF1X2ZpbmFsOiBmbG9hdCA9IDAuMzAsCiAgICBhbm5lYWxfZW5kX2Vwb2NoOiBpbnQgPSAzNSwKKSAtPiBmbG9hdDoKICAgICIiIkNvc2luZS1hbm5lYWwgbW90aWYgdGVtcGVyYXR1cmUsIHRoZW4gaG9sZCB0aGUgcmVnaXN0ZXJlZCBmbG9vci4iIiIKICAgIGlmIGVwb2NoIDwgMToKICAgICAgICByYWlzZSBWYWx1ZUVycm9yKCJlcG9jaCBtdXN0IGJlID49IDEiKQogICAgaWYgYW5uZWFsX2VuZF9lcG9jaCA8IDI6CiAgICAgICAgcmFpc2UgVmFsdWVFcnJvcigiYW5uZWFsX2VuZF9lcG9jaCBtdXN0IGJlID49IDIiKQogICAgaWYgbm90IDAuMCA8IHRhdV9maW5hbCA8PSB0YXVfc3RhcnQ6CiAgICAgICAgcmFpc2UgVmFsdWVFcnJvcigidGVtcGVyYXR1cmUgYm91bmRzIG11c3Qgc2F0aXNmeSAwIDwgZmluYWwgPD0gc3RhcnQiKQogICAgcHJvZ3Jlc3MgPSBtaW4obWF4KChlcG9jaCAtIDEpIC8gKGFubmVhbF9lbmRfZXBvY2ggLSAxKSwgMC4wKSwgMS4wKQogICAgcmV0dXJuIHRhdV9maW5hbCArIDAuNSAqICh0YXVfc3RhcnQgLSB0YXVfZmluYWwpICogKAogICAgICAgIDEuMCArIG1hdGguY29zKG1hdGgucGkgKiBwcm9ncmVzcykKICAgICkKCgpkZWYgX3JlZmxlY3RfaW5kZXgoaW5kZXg6IGludCwgc2l6ZTogaW50KSAtPiBpbnQ6CiAgICAiIiJNYXAgYW4gYXJiaXRyYXJ5IGludGVnZXIgdG8gWzAsc2l6ZSkgdXNpbmcgcmVmbGVjdGlvbiB3aXRob3V0IGVkZ2UgcmVwZWF0LiIiIgogICAgaWYgc2l6ZSA8IDI6CiAgICAgICAgcmV0dXJuIDAKICAgIHBlcmlvZCA9IDIgKiAoc2l6ZSAtIDEpCiAgICB2YWx1ZSA9IGluZGV4ICUgcGVyaW9kCiAgICByZXR1cm4gdmFsdWUgaWYgdmFsdWUgPCBzaXplIGVsc2UgcGVyaW9kIC0gdmFsdWUKCgpkZWYgYnVpbGRfYWxpZ25lZF9zdXBwb3J0X2luZGljZXMoCiAgICBpbWdfc2l6ZTogaW50ID0gNDgsCiAgICBhbmNob3Jfc2l6ZTogaW50ID0gMTIsCiAgICBzdHJpZGU6IGludCA9IDYsCiAgICBzY2FsZXM6IHR1cGxlW2ludCwgLi4uXSA9ICg4LCAxMiwgMTYpLAopIC0+IHR1cGxlW2RpY3RbaW50LCB0b3JjaC5UZW5zb3JdLCB0b3JjaC5UZW5zb3JdOgogICAgIiIiQnVpbGQgcmVmbGVjdGlvbi1tYXBwZWQgc3VwcG9ydHMgc2hhcmluZyB0aGUgMTJ4MTIgYW5jaG9yIGNlbnRlcnMuIiIiCiAgICBzdGFydHMgPSBsaXN0KHJhbmdlKDAsIGltZ19zaXplIC0gYW5jaG9yX3NpemUgKyAxLCBzdHJpZGUpKQogICAgaWYgbGVuKHN0YXJ0cykgIT0gNzoKICAgICAgICByYWlzZSBWYWx1ZUVycm9yKCJNUEctRkVSIHYyLjIgcmVxdWlyZXMgYSA3eDcgYW5jaG9yIGdyaWQiKQogICAgY2VudGVycyA9IFtdCiAgICBzdXBwb3J0czogZGljdFtpbnQsIGxpc3RbbGlzdFtpbnRdXV0gPSB7c2NhbGU6IFtdIGZvciBzY2FsZSBpbiBzY2FsZXN9CiAgICBmb3IgdG9wIGluIHN0YXJ0czoKICAgICAgICBmb3IgbGVmdCBpbiBzdGFydHM6CiAgICAgICAgICAgIGNlbnRlcl95ID0gdG9wICsgKGFuY2hvcl9zaXplIC0gMSkgLyAyLjAKICAgICAgICAgICAgY2VudGVyX3ggPSBsZWZ0ICsgKGFuY2hvcl9zaXplIC0gMSkgLyAyLjAKICAgICAgICAgICAgY2VudGVycy5hcHBlbmQoKGNlbnRlcl94LCBjZW50ZXJfeSkpCiAgICAgICAgICAgIGZvciBzY2FsZSBpbiBzY2FsZXM6CiAgICAgICAgICAgICAgICBzdGFydF95ID0gaW50KHJvdW5kKGNlbnRlcl95IC0gKHNjYWxlIC0gMSkgLyAyLjApKQogICAgICAgICAgICAgICAgc3RhcnRfeCA9IGludChyb3VuZChjZW50ZXJfeCAtIChzY2FsZSAtIDEpIC8gMi4wKSkKICAgICAgICAgICAgICAgIGluZGljZXMgPSBbCiAgICAgICAgICAgICAgICAgICAgX3JlZmxlY3RfaW5kZXgocm93LCBpbWdfc2l6ZSkgKiBpbWdfc2l6ZQogICAgICAgICAgICAgICAgICAgICsgX3JlZmxlY3RfaW5kZXgoY29sdW1uLCBpbWdfc2l6ZSkKICAgICAgICAgICAgICAgICAgICBmb3Igcm93IGluIHJhbmdlKHN0YXJ0X3ksIHN0YXJ0X3kgKyBzY2FsZSkKICAgICAgICAgICAgICAgICAgICBmb3IgY29sdW1uIGluIHJhbmdlKHN0YXJ0X3gsIHN0YXJ0X3ggKyBzY2FsZSkKICAgICAgICAgICAgICAgIF0KICAgICAgICAgICAgICAgIHN1cHBvcnRzW3NjYWxlXS5hcHBlbmQoaW5kaWNlcykKICAgIHJldHVybiAoCiAgICAgICAge3NjYWxlOiB0b3JjaC50ZW5zb3IodmFsdWVzLCBkdHlwZT10b3JjaC5sb25nKSBmb3Igc2NhbGUsIHZhbHVlcyBpbiBzdXBwb3J0cy5pdGVtcygpfSwKICAgICAgICB0b3JjaC50ZW5zb3IoY2VudGVycywgZHR5cGU9dG9yY2guZmxvYXQzMiksCiAgICApCgoKZGVmIGFsaWduZWRfbG9naWNhbF9zdGFydHMoCiAgICBpbWdfc2l6ZTogaW50ID0gNDgsCiAgICBhbmNob3Jfc2l6ZTogaW50ID0gMTIsCiAgICBzdHJpZGU6IGludCA9IDYsCiAgICBzY2FsZXM6IHR1cGxlW2ludCwgLi4uXSA9ICg4LCAxMiwgMTYpLAopIC0+IGRpY3RbaW50LCB0b3JjaC5UZW5zb3JdOgogICAgIiIiUmV0dXJuIHByZS1yZWZsZWN0aW9uIFt0b3AsbGVmdF0gc3RhcnRzIGZvciBhbGlnbm1lbnQgYXVkaXRzLiIiIgogICAgc3RhcnRzID0gbGlzdChyYW5nZSgwLCBpbWdfc2l6ZSAtIGFuY2hvcl9zaXplICsgMSwgc3RyaWRlKSkKICAgIHJlc3VsdDogZGljdFtpbnQsIGxpc3RbdHVwbGVbaW50LCBpbnRdXV0gPSB7c2NhbGU6IFtdIGZvciBzY2FsZSBpbiBzY2FsZXN9CiAgICBmb3IgdG9wIGluIHN0YXJ0czoKICAgICAgICBmb3IgbGVmdCBpbiBzdGFydHM6CiAgICAgICAgICAgIGNlbnRlcl95ID0gdG9wICsgKGFuY2hvcl9zaXplIC0gMSkgLyAyLjAKICAgICAgICAgICAgY2VudGVyX3ggPSBsZWZ0ICsgKGFuY2hvcl9zaXplIC0gMSkgLyAyLjAKICAgICAgICAgICAgZm9yIHNjYWxlIGluIHNjYWxlczoKICAgICAgICAgICAgICAgIHJlc3VsdFtzY2FsZV0uYXBwZW5kKAogICAgICAgICAgICAgICAgICAgICgKICAgICAgICAgICAgICAgICAgICAgICAgaW50KHJvdW5kKGNlbnRlcl95IC0gKHNjYWxlIC0gMSkgLyAyLjApKSwKICAgICAgICAgICAgICAgICAgICAgICAgaW50KHJvdW5kKGNlbnRlcl94IC0gKHNjYWxlIC0gMSkgLyAyLjApKSwKICAgICAgICAgICAgICAgICAgICApCiAgICAgICAgICAgICAgICApCiAgICByZXR1cm4ge3NjYWxlOiB0b3JjaC50ZW5zb3IodmFsdWVzLCBkdHlwZT10b3JjaC5sb25nKSBmb3Igc2NhbGUsIHZhbHVlcyBpbiByZXN1bHQuaXRlbXMoKX0KCgpjbGFzcyBTcGF0aWFsTW90aWZDb21wb3Nlcihubi5Nb2R1bGUpOgogICAgIiIiU29mdCBUWVBFIGFzc2lnbm1lbnQgYW5kIGFsaWduZWQgOC8xMi8xNiBvY2N1cnJlbmNlIGZ1c2lvbiBpbnRvIDQ5IG5vZGVzLiIiIgoKICAgIGRlZiBfX2luaXRfXygKICAgICAgICBzZWxmLAogICAgICAgIGRfcGl4ZWw6IGludCA9IDk2LAogICAgICAgIG51bV9tb3RpZnM6IGludCA9IDQ4LAogICAgICAgIG1vdGlmX3dpbmRvd19zaXplczogdHVwbGVbaW50LCBpbnQsIGludF0gPSAoOCwgMTIsIDE2KSwKICAgICAgICBtb3RpZl9zdHJpZGU6IGludCA9IDYsCiAgICAgICAgaW1nX3NpemU6IGludCA9IDQ4LAogICAgICAgIGRfdHlwZTogaW50ID0gMzIsCiAgICAgICAgZF9tb3RpZjogaW50ID0gMTkyLAogICAgICAgIHRhdV9zdGFydDogZmxvYXQgPSAwLjcwLAogICAgICAgIHRhdV9maW5hbDogZmxvYXQgPSAwLjMwLAogICAgICAgIHRhdV9hbm5lYWxfZW5kX2Vwb2NoOiBpbnQgPSAzNSwKICAgICAgICBtaV9iZXRhOiBmbG9hdCA9IDEuMCwKICAgICAgICBlcHM6IGZsb2F0ID0gMWUtNiwKICAgICkgLT4gTm9uZToKICAgICAgICBzdXBlcigpLl9faW5pdF9fKCkKICAgICAgICBzY2hlZHVsZWRfbW90aWZfdGVtcGVyYXR1cmUoCiAgICAgICAgICAgIDEsIHRhdV9zdGFydD10YXVfc3RhcnQsIHRhdV9maW5hbD10YXVfZmluYWwsCiAgICAgICAgICAgIGFubmVhbF9lbmRfZXBvY2g9dGF1X2FubmVhbF9lbmRfZXBvY2gsCiAgICAgICAgKQogICAgICAgIHNlbGYuZF9waXhlbCA9IGRfcGl4ZWwKICAgICAgICBzZWxmLm51bV9tb3RpZnMgPSBudW1fbW90aWZzCiAgICAgICAgc2VsZi53aW5kb3dfc2l6ZXMgPSB0dXBsZShtb3RpZl93aW5kb3dfc2l6ZXMpCiAgICAgICAgc2VsZi5zdHJpZGUgPSBtb3RpZl9zdHJpZGUKICAgICAgICBzZWxmLmltZ19zaXplID0gaW1nX3NpemUKICAgICAgICBzZWxmLmRfdHlwZSA9IGRfdHlwZQogICAgICAgIHNlbGYuZF9tb3RpZiA9IGRfbW90aWYKICAgICAgICBzZWxmLnRhdV9zdGFydCA9IGZsb2F0KHRhdV9zdGFydCkKICAgICAgICBzZWxmLnRhdV9maW5hbCA9IGZsb2F0KHRhdV9maW5hbCkKICAgICAgICBzZWxmLnRhdV9hbm5lYWxfZW5kX2Vwb2NoID0gaW50KHRhdV9hbm5lYWxfZW5kX2Vwb2NoKQogICAgICAgIHNlbGYubWlfYmV0YSA9IGZsb2F0KG1pX2JldGEpCiAgICAgICAgc2VsZi5lcHMgPSBlcHMKCiAgICAgICAgc2VsZi5wcm90b3R5cGVzID0gbm4uUGFyYW1ldGVyKHRvcmNoLnJhbmRuKG51bV9tb3RpZnMsIGRfcGl4ZWwpIC8gbWF0aC5zcXJ0KGRfcGl4ZWwpKQogICAgICAgIHNlbGYuYXNzaWdubWVudF9xdWVyeSA9IG5uLkxpbmVhcihkX3BpeGVsLCBkX3BpeGVsLCBiaWFzPUZhbHNlKQogICAgICAgIHNlbGYucHJvdG90eXBlX2tleSA9IG5uLkxpbmVhcihkX3BpeGVsLCBkX3BpeGVsLCBiaWFzPUZhbHNlKQogICAgICAgIHNlbGYucmVnaXN0ZXJfYnVmZmVyKAogICAgICAgICAgICAiY3VycmVudF90YXUiLCB0b3JjaC50ZW5zb3Ioc2VsZi50YXVfc3RhcnQsIGR0eXBlPXRvcmNoLmZsb2F0MzIpCiAgICAgICAgKQoKICAgICAgICBzZWxmLnNjYWxlX3NhbGllbmN5ID0gbm4uTW9kdWxlRGljdCgKICAgICAgICAgICAge3N0cihzY2FsZSk6IG5uLkxpbmVhcihkX3BpeGVsLCAxKSBmb3Igc2NhbGUgaW4gc2VsZi53aW5kb3dfc2l6ZXN9CiAgICAgICAgKQogICAgICAgIHNlbGYuc2NhbGVfY29uZmlkZW5jZSA9IG5uLlBhcmFtZXRlcih0b3JjaC5vbmVzKGxlbihzZWxmLndpbmRvd19zaXplcykpKQogICAgICAgIHNlbGYudHlwZV9wcm9qID0gbm4uU2VxdWVudGlhbCgKICAgICAgICAgICAgbm4uTGluZWFyKG51bV9tb3RpZnMsIGRfdHlwZSksIG5uLkxheWVyTm9ybShkX3R5cGUpLCBubi5HRUxVKCkKICAgICAgICApCiAgICAgICAgc2VsZi5vY2N1cnJlbmNlX3Byb2ogPSBubi5TZXF1ZW50aWFsKAogICAgICAgICAgICBubi5MaW5lYXIoZF9waXhlbCArIGRfdHlwZSArIDUsIGRfbW90aWYpLAogICAgICAgICAgICBubi5MYXllck5vcm0oZF9tb3RpZiksCiAgICAgICAgICAgIG5uLkdFTFUoKSwKICAgICAgICApCiAgICAgICAgc2VsZi5zY2FsZV9nYXRlID0gbm4uTGluZWFyKGRfbW90aWYsIDEpCgogICAgICAgIGNvb3JkcyA9IHRvcmNoLmxpbnNwYWNlKC0xLjAsIDEuMCwgaW1nX3NpemUpCiAgICAgICAgZ3JpZF95LCBncmlkX3ggPSB0b3JjaC5tZXNoZ3JpZChjb29yZHMsIGNvb3JkcywgaW5kZXhpbmc9ImlqIikKICAgICAgICBzZWxmLnJlZ2lzdGVyX2J1ZmZlcigiZ3JpZF94IiwgZ3JpZF94LnJlc2hhcGUoLTEpKQogICAgICAgIHNlbGYucmVnaXN0ZXJfYnVmZmVyKCJncmlkX3kiLCBncmlkX3kucmVzaGFwZSgtMSkpCiAgICAgICAgc3VwcG9ydHMsIGNlbnRlcnMgPSBidWlsZF9hbGlnbmVkX3N1cHBvcnRfaW5kaWNlcygKICAgICAgICAgICAgaW1nX3NpemU9aW1nX3NpemUsCiAgICAgICAgICAgIGFuY2hvcl9zaXplPTEyLAogICAgICAgICAgICBzdHJpZGU9bW90aWZfc3RyaWRlLAogICAgICAgICAgICBzY2FsZXM9c2VsZi53aW5kb3dfc2l6ZXMsCiAgICAgICAgKQogICAgICAgIHNlbGYucmVnaXN0ZXJfYnVmZmVyKCJhbmNob3JfY2VudGVyc19waXhlbCIsIGNlbnRlcnMpCiAgICAgICAgZm9yIHNjYWxlLCBpbmRpY2VzIGluIHN1cHBvcnRzLml0ZW1zKCk6CiAgICAgICAgICAgIHNlbGYucmVnaXN0ZXJfYnVmZmVyKGYic3VwcG9ydF9pZHhfe3NjYWxlfSIsIGluZGljZXMpCgogICAgQHByb3BlcnR5CiAgICBkZWYgdGVtcGVyYXR1cmUoc2VsZikgLT4gdG9yY2guVGVuc29yOgogICAgICAgIHJldHVybiBzZWxmLmN1cnJlbnRfdGF1CgogICAgQHRvcmNoLm5vX2dyYWQoKQogICAgZGVmIHNldF9lcG9jaF90ZW1wZXJhdHVyZShzZWxmLCBlcG9jaDogaW50KSAtPiBmbG9hdDoKICAgICAgICB2YWx1ZSA9IHNjaGVkdWxlZF9tb3RpZl90ZW1wZXJhdHVyZSgKICAgICAgICAgICAgZXBvY2gsCiAgICAgICAgICAgIHRhdV9zdGFydD1zZWxmLnRhdV9zdGFydCwKICAgICAgICAgICAgdGF1X2ZpbmFsPXNlbGYudGF1X2ZpbmFsLAogICAgICAgICAgICBhbm5lYWxfZW5kX2Vwb2NoPXNlbGYudGF1X2FubmVhbF9lbmRfZXBvY2gsCiAgICAgICAgKQogICAgICAgIHNlbGYuY3VycmVudF90YXUuZmlsbF8odmFsdWUpCiAgICAgICAgcmV0dXJuIHZhbHVlCgogICAgZGVmIF9wb29sX3NjYWxlKAogICAgICAgIHNlbGYsIGhfcGl4ZWw6IHRvcmNoLlRlbnNvciwgYXNzaWdubWVudHM6IHRvcmNoLlRlbnNvciwKICAgICAgICBjb25maWRlbmNlOiB0b3JjaC5UZW5zb3IsIHNjYWxlOiBpbnQsCiAgICApIC0+IHR1cGxlW3RvcmNoLlRlbnNvciwgdG9yY2guVGVuc29yLCB0b3JjaC5UZW5zb3IsIHRvcmNoLlRlbnNvcl06CiAgICAgICAgYmF0Y2ggPSBoX3BpeGVsLnNoYXBlWzBdCiAgICAgICAgaW5kaWNlcyA9IGdldGF0dHIoc2VsZiwgZiJzdXBwb3J0X2lkeF97c2NhbGV9IikKICAgICAgICBvY2N1cnJlbmNlcywgc3VwcG9ydF9zaXplID0gaW5kaWNlcy5zaGFwZQogICAgICAgIGdhdGhlcl9pZHggPSBpbmRpY2VzLnVuc3F1ZWV6ZSgwKS5leHBhbmQoYmF0Y2gsIC0xLCAtMSkKICAgICAgICBoX3N1cHBvcnQgPSBoX3BpeGVsWzosIGluZGljZXMsIDpdCiAgICAgICAgYV9zdXBwb3J0ID0gYXNzaWdubWVudHNbOiwgaW5kaWNlcywgOl0KICAgICAgICBjb25mX3N1cHBvcnQgPSBjb25maWRlbmNlWzosIGluZGljZXMsIDpdCiAgICAgICAgeF9zdXBwb3J0ID0gc2VsZi5ncmlkX3hbaW5kaWNlc10udmlldygxLCBvY2N1cnJlbmNlcywgc3VwcG9ydF9zaXplLCAxKS5leHBhbmQoYmF0Y2gsIC0xLCAtMSwgLTEpCiAgICAgICAgeV9zdXBwb3J0ID0gc2VsZi5ncmlkX3lbaW5kaWNlc10udmlldygxLCBvY2N1cnJlbmNlcywgc3VwcG9ydF9zaXplLCAxKS5leHBhbmQoYmF0Y2gsIC0xLCAtMSwgLTEpCgogICAgICAgIHNhbGllbmN5ID0gc2VsZi5zY2FsZV9zYWxpZW5jeVtzdHIoc2NhbGUpXShoX3N1cHBvcnQpCiAgICAgICAgc2NhbGVfcG9zaXRpb24gPSBzZWxmLndpbmRvd19zaXplcy5pbmRleChzY2FsZSkKICAgICAgICB3ZWlnaHRzID0gRi5zb2Z0bWF4KHNhbGllbmN5ICsgc2VsZi5zY2FsZV9jb25maWRlbmNlW3NjYWxlX3Bvc2l0aW9uXSAqIGNvbmZfc3VwcG9ydCwgZGltPTIpCiAgICAgICAgd2hhdCA9ICh3ZWlnaHRzICogaF9zdXBwb3J0KS5zdW0oZGltPTIpCiAgICAgICAgdHlwZV9kaXN0cmlidXRpb24gPSAod2VpZ2h0cyAqIGFfc3VwcG9ydCkuc3VtKGRpbT0yKQogICAgICAgIG1vdGlmX3R5cGUgPSBzZWxmLnR5cGVfcHJvaih0eXBlX2Rpc3RyaWJ1dGlvbikKICAgICAgICBjeCA9ICh3ZWlnaHRzICogeF9zdXBwb3J0KS5zdW0oZGltPTIpCiAgICAgICAgY3kgPSAod2VpZ2h0cyAqIHlfc3VwcG9ydCkuc3VtKGRpbT0yKQogICAgICAgIHN4ID0gdG9yY2guc3FydCgod2VpZ2h0cyAqICh4X3N1cHBvcnQgLSBjeC51bnNxdWVlemUoMikpLnNxdWFyZSgpKS5zdW0oZGltPTIpICsgc2VsZi5lcHMpCiAgICAgICAgc3kgPSB0b3JjaC5zcXJ0KCh3ZWlnaHRzICogKHlfc3VwcG9ydCAtIGN5LnVuc3F1ZWV6ZSgyKSkuc3F1YXJlKCkpLnN1bShkaW09MikgKyBzZWxmLmVwcykKICAgICAgICBtYXNzID0gKHdlaWdodHMgKiBjb25mX3N1cHBvcnQpLnN1bShkaW09MikKICAgICAgICB3aGVyZSA9IHRvcmNoLmNhdChbY3gsIGN5LCBzeCwgc3ksIG1hc3NdLCBkaW09LTEpCiAgICAgICAgY2FuZGlkYXRlID0gc2VsZi5vY2N1cnJlbmNlX3Byb2oodG9yY2guY2F0KFt3aGF0LCBtb3RpZl90eXBlLCB3aGVyZV0sIGRpbT0tMSkpCiAgICAgICAgY2VudGVycyA9IHRvcmNoLmNhdChbY3gsIGN5XSwgZGltPS0xKQogICAgICAgIHJldHVybiBjYW5kaWRhdGUsIGNlbnRlcnMsIHdlaWdodHMuc3F1ZWV6ZSgtMSksIHR5cGVfZGlzdHJpYnV0aW9uCgogICAgZGVmIGZvcndhcmQoc2VsZiwgaF9waXhlbDogdG9yY2guVGVuc29yKSAtPiB0dXBsZVt0b3JjaC5UZW5zb3IsIHRvcmNoLlRlbnNvciwgZGljdFtzdHIsIHRvcmNoLlRlbnNvcl1dOgogICAgICAgIGJhdGNoLCBub2RlcywgZGltZW5zaW9uID0gaF9waXhlbC5zaGFwZQogICAgICAgIGlmIG5vZGVzICE9IHNlbGYuaW1nX3NpemUgKiBzZWxmLmltZ19zaXplIG9yIGRpbWVuc2lvbiAhPSBzZWxmLmRfcGl4ZWw6CiAgICAgICAgICAgIHJhaXNlIFZhbHVlRXJyb3IoIlVuZXhwZWN0ZWQgcGl4ZWwgZW1iZWRkaW5nIHNoYXBlIikKCiAgICAgICAgcXVlcmllcyA9IEYubm9ybWFsaXplKHNlbGYuYXNzaWdubWVudF9xdWVyeShoX3BpeGVsKSwgZGltPS0xKQogICAgICAgIGtleXMgPSBGLm5vcm1hbGl6ZShzZWxmLnByb3RvdHlwZV9rZXkoc2VsZi5wcm90b3R5cGVzKSwgZGltPS0xKQogICAgICAgIGFzc2lnbm1lbnRzID0gRi5zb2Z0bWF4KHF1ZXJpZXMgQCBrZXlzLnQoKSAvIHNlbGYudGVtcGVyYXR1cmUsIGRpbT0tMSkKICAgICAgICBjb25maWRlbmNlID0gYXNzaWdubWVudHMubWF4KGRpbT0tMSwga2VlcGRpbT1UcnVlKS52YWx1ZXMKCiAgICAgICAgY2FuZGlkYXRlcywgY2VudGVycywgd2VpZ2h0cywgdHlwZV9kaXN0cmlidXRpb25zID0gW10sIFtdLCB7fSwge30KICAgICAgICBmb3Igc2NhbGUgaW4gc2VsZi53aW5kb3dfc2l6ZXM6CiAgICAgICAgICAgIGNhbmRpZGF0ZSwgY2VudGVyLCBzY2FsZV93ZWlnaHRzLCBzY2FsZV90eXBlcyA9IHNlbGYuX3Bvb2xfc2NhbGUoCiAgICAgICAgICAgICAgICBoX3BpeGVsLCBhc3NpZ25tZW50cywgY29uZmlkZW5jZSwgc2NhbGUKICAgICAgICAgICAgKQogICAgICAgICAgICBjYW5kaWRhdGVzLmFwcGVuZChjYW5kaWRhdGUpCiAgICAgICAgICAgIGNlbnRlcnMuYXBwZW5kKGNlbnRlcikKICAgICAgICAgICAgd2VpZ2h0c1tzdHIoc2NhbGUpXSA9IHNjYWxlX3dlaWdodHMKICAgICAgICAgICAgdHlwZV9kaXN0cmlidXRpb25zW3N0cihzY2FsZSldID0gc2NhbGVfdHlwZXMKICAgICAgICBjYW5kaWRhdGVfc3RhY2sgPSB0b3JjaC5zdGFjayhjYW5kaWRhdGVzLCBkaW09MikgICMgW0IsNDksMywxOTJdCiAgICAgICAgY2VudGVyX3N0YWNrID0gdG9yY2guc3RhY2soY2VudGVycywgZGltPTIpICAjIFtCLDQ5LDMsMl0KICAgICAgICBhbHBoYSA9IEYuc29mdG1heChzZWxmLnNjYWxlX2dhdGUoY2FuZGlkYXRlX3N0YWNrKS5zcXVlZXplKC0xKSwgZGltPS0xKQogICAgICAgIGhfbW90aWYgPSAoYWxwaGEudW5zcXVlZXplKC0xKSAqIGNhbmRpZGF0ZV9zdGFjaykuc3VtKGRpbT0yKQogICAgICAgIGZ1c2VkX2NlbnRlcnMgPSAoYWxwaGEudW5zcXVlZXplKC0xKSAqIGNlbnRlcl9zdGFjaykuc3VtKGRpbT0yKQoKICAgICAgICBwX25vcm0gPSBGLm5vcm1hbGl6ZShzZWxmLnByb3RvdHlwZXMsIGRpbT0tMSkKICAgICAgICBwcm90b3R5cGVfY29zaW5lID0gcF9ub3JtIEAgcF9ub3JtLnQoKQogICAgICAgIG9mZmRpYWdfbWFzayA9IH50b3JjaC5leWUoc2VsZi5udW1fbW90aWZzLCBkdHlwZT10b3JjaC5ib29sLCBkZXZpY2U9aF9waXhlbC5kZXZpY2UpCiAgICAgICAgb2ZmZGlhZyA9IHByb3RvdHlwZV9jb3NpbmVbb2ZmZGlhZ19tYXNrXQogICAgICAgIGRpdmVyc2l0eSA9IEYucmVsdShvZmZkaWFnKS5zcXVhcmUoKS5tZWFuKCkKICAgICAgICBtaV9sb3NzLCBtaSA9IG1vdGlmX211dHVhbF9pbmZvcm1hdGlvbl9sb3NzKGFzc2lnbm1lbnRzLCBiZXRhPXNlbGYubWlfYmV0YSkKICAgICAgICB1dGlsaXphdGlvbiA9IG1pWyJwcm90b3R5cGVfdXRpbGl6YXRpb24iXQogICAgICAgIHRvcDIgPSBhc3NpZ25tZW50cy50b3BrKDIsIGRpbT0tMSkudmFsdWVzCiAgICAgICAgbWVhbl9lbnRyb3B5ID0gbWlbIkhfbG9jYWxfcmF3Il0KICAgICAgICBkaWFnbm9zdGljcyA9IHsKICAgICAgICAgICAgImxvc3NfZGl2ZXJzaXR5IjogZGl2ZXJzaXR5LAogICAgICAgICAgICAibG9zc19taSI6IG1pX2xvc3MsCiAgICAgICAgICAgICJ0YXUiOiBzZWxmLnRlbXBlcmF0dXJlLAogICAgICAgICAgICAiSF9sb2NhbF9yYXciOiBtaVsiSF9sb2NhbF9yYXciXSwKICAgICAgICAgICAgIkhfbG9jYWxfbm9ybWFsaXplZCI6IG1pWyJIX2xvY2FsX25vcm1hbGl6ZWQiXSwKICAgICAgICAgICAgIkhfZ2xvYmFsX3JhdyI6IG1pWyJIX2dsb2JhbF9yYXciXSwKICAgICAgICAgICAgIkhfZ2xvYmFsX25vcm1hbGl6ZWQiOiBtaVsiSF9nbG9iYWxfbm9ybWFsaXplZCJdLAogICAgICAgICAgICAiTF9NSSI6IG1pWyJMX01JIl0sCiAgICAgICAgICAgICJtZWFuX2VudHJvcHkiOiBtZWFuX2VudHJvcHksCiAgICAgICAgICAgICJlZmZlY3RpdmVfbW90aWZfY291bnQiOiBtZWFuX2VudHJvcHkuZXhwKCksCiAgICAgICAgICAgICJtaW5fdXRpbGl6YXRpb24iOiB1dGlsaXphdGlvbi5taW4oKSwKICAgICAgICAgICAgIm1heF91dGlsaXphdGlvbiI6IHV0aWxpemF0aW9uLm1heCgpLAogICAgICAgICAgICAic3RkX3V0aWxpemF0aW9uIjogdXRpbGl6YXRpb24uc3RkKHVuYmlhc2VkPUZhbHNlKSwKICAgICAgICAgICAgIm1lYW5fdG9wMV9wcm9iYWJpbGl0eSI6IHRvcDJbLi4uLCAwXS5tZWFuKCksCiAgICAgICAgICAgICJtZWFuX3RvcDJfcHJvYmFiaWxpdHkiOiB0b3AyWy4uLiwgMV0ubWVhbigpLAogICAgICAgICAgICAibWVhbl90b3AxX3RvcDJfbWFyZ2luIjogKHRvcDJbLi4uLCAwXSAtIHRvcDJbLi4uLCAxXSkubWVhbigpLAogICAgICAgICAgICAibWVhbl9vZmZkaWFnX3Byb3RvdHlwZV9jb3NpbmUiOiBvZmZkaWFnLm1lYW4oKSwKICAgICAgICAgICAgImxlYXJuZWRfY2VudGVyc194IjogZnVzZWRfY2VudGVyc1suLi4sIDBdLAogICAgICAgICAgICAibGVhcm5lZF9jZW50ZXJzX3kiOiBmdXNlZF9jZW50ZXJzWy4uLiwgMV0sCiAgICAgICAgICAgICJzY2FsZV93ZWlnaHRzIjogYWxwaGEsCiAgICAgICAgICAgICJtZWFuX2FscGhhXzgiOiBhbHBoYVsuLi4sIDBdLm1lYW4oKSwKICAgICAgICAgICAgIm1lYW5fYWxwaGFfMTIiOiBhbHBoYVsuLi4sIDFdLm1lYW4oKSwKICAgICAgICAgICAgIm1lYW5fYWxwaGFfMTYiOiBhbHBoYVsuLi4sIDJdLm1lYW4oKSwKICAgICAgICAgICAgInN0ZF9hbHBoYV84IjogYWxwaGFbLi4uLCAwXS5zdGQodW5iaWFzZWQ9RmFsc2UpLAogICAgICAgICAgICAic3RkX2FscGhhXzEyIjogYWxwaGFbLi4uLCAxXS5zdGQodW5iaWFzZWQ9RmFsc2UpLAogICAgICAgICAgICAic3RkX2FscGhhXzE2IjogYWxwaGFbLi4uLCAyXS5zdGQodW5iaWFzZWQ9RmFsc2UpLAogICAgICAgICAgICAib2NjdXJyZW5jZV93ZWlnaHRzXzgiOiB3ZWlnaHRzWyI4Il0sCiAgICAgICAgICAgICJvY2N1cnJlbmNlX3dlaWdodHNfMTIiOiB3ZWlnaHRzWyIxMiJdLAogICAgICAgICAgICAib2NjdXJyZW5jZV93ZWlnaHRzXzE2Ijogd2VpZ2h0c1siMTYiXSwKICAgICAgICAgICAgInR5cGVfZGlzdHJpYnV0aW9ucyI6IHN1bSgKICAgICAgICAgICAgICAgIGFscGhhWy4uLiwgaW5kZXhdLnVuc3F1ZWV6ZSgtMSkgKiB0eXBlX2Rpc3RyaWJ1dGlvbnNbc3RyKHNjYWxlKV0KICAgICAgICAgICAgICAgIGZvciBpbmRleCwgc2NhbGUgaW4gZW51bWVyYXRlKHNlbGYud2luZG93X3NpemVzKQogICAgICAgICAgICApLAogICAgICAgIH0KICAgICAgICByZXR1cm4gaF9tb3RpZiwgYXNzaWdubWVudHMsIGRpYWdub3N0aWNzCg==', 'mpg_fer_v2_3/train.py': '...', 'mpg_fer_v2_3/utils.py': 'IiIiVXRpbGl0eSBmdW5jdGlvbnMgZm9yIHRyYWluaW5nLCBsb2dnaW5nLCBjaGVja3BvaW50aW5nLCBhbmQgcmVwcm9kdWNpYmlsaXR5LiIiIgoKZnJvbSBfX2Z1dHVyZV9fIGltcG9ydCBhbm5vdGF0aW9ucwoKaW1wb3J0IGpzb24KaW1wb3J0IG9zCmZyb20gcGF0aGxpYiBpbXBvcnQgUGF0aAppbXBvcnQgcmFuZG9tCgppbXBvcnQgbnVtcHkgYXMgbnAKaW1wb3J0IHRvcmNoCmltcG9ydCB0b3JjaC5ubiBhcyBubgoKCmRlZiBzZXRfc2VlZChzZWVkOiBpbnQgPSA0MikgLT4gTm9uZToKICAgICIiIkVuc3VyZSBmdWxsIGRldGVybWluaXNtIGFjcm9zcyBQeXRob24sIE51bVB5LCBhbmQgUHlUb3JjaC4iIiIKICAgIHJhbmRvbS5zZWVkKHNlZWQpCiAgICBucC5yYW5kb20uc2VlZChzZWVkKQogICAgdG9yY2gubWFudWFsX3NlZWQoc2VlZCkKICAgIGlmIHRvcmNoLmN1ZGEuaXNfYXZhaWxhYmxlKCk6CiAgICAgICAgdG9yY2guY3VkYS5tYW51YWxfc2VlZChzZWVkKQogICAgICAgIHRvcmNoLmN1ZGEubWFudWFsX3NlZWRfYWxsKHNlZWQpCiAgICAgICAgdG9yY2guYmFja2VuZHMuY3Vkbm4uZGV0ZXJtaW5pc3RpYyA9IFRydWUKICAgICAgICB0b3JjaC5iYWNrZW5kcy5jdWRubi5iZW5jaG1hcmsgPSBGYWxzZQoKCmRlZiBzYXZlX2NoZWNrcG9pbnQoCiAgICBjaGVja3BvaW50X3BhdGg6IHN0ciB8IFBhdGgsCiAgICBtb2RlbDogbm4uTW9kdWxlLAogICAgb3B0aW1pemVyOiB0b3JjaC5vcHRpbS5PcHRpbWl6ZXIsCiAgICBlcG9jaDogaW50LAogICAgdmFsX21ldHJpY3M6IGRpY3QsCiAgICBjb25maWdfZGljdDogZGljdCwKICAgIGV4dHJhX3N0YXRlOiBkaWN0IHwgTm9uZSA9IE5vbmUsCikgLT4gTm9uZToKICAgICIiIlNhdmUgYW4gYXRvbWljLCByZXByb2R1Y2libGUgZXhwZXJpbWVudCBjaGVja3BvaW50LiIiIgogICAgcCA9IFBhdGgoY2hlY2twb2ludF9wYXRoKQogICAgcC5wYXJlbnQubWtkaXIocGFyZW50cz1UcnVlLCBleGlzdF9vaz1UcnVlKQoKICAgIHN0YXRlID0gewogICAgICAgICJlcG9jaCI6IGVwb2NoLAogICAgICAgICJtb2RlbF9zdGF0ZV9kaWN0IjogbW9kZWwuc3RhdGVfZGljdCgpLAogICAgICAgICJvcHRpbWl6ZXJfc3RhdGVfZGljdCI6IG9wdGltaXplci5zdGF0ZV9kaWN0KCksCiAgICAgICAgInZhbF9tZXRyaWNzIjogdmFsX21ldHJpY3MsCiAgICAgICAgImNvbmZpZyI6IGNvbmZpZ19kaWN0LAogICAgfQogICAgaWYgZXh0cmFfc3RhdGUgaXMgbm90IE5vbmU6CiAgICAgICAgc3RhdGUudXBkYXRlKGV4dHJhX3N0YXRlKQoKICAgIHRvcmNoLnNhdmUoc3RhdGUsIHApCgoKZGVmIGxvYWRfY2hlY2twb2ludCgKICAgIGNoZWNrcG9pbnRfcGF0aDogc3RyIHwgUGF0aCwKICAgIG1vZGVsOiBubi5Nb2R1bGUsCiAgICBvcHRpbWl6ZXI6IHRvcmNoLm9wdGltLk9wdGltaXplciB8IE5vbmUgPSBOb25lLAopIC0+IGRpY3Q6CiAgICAiIiJMb2FkIG1vZGVsIGFuZCBvcHRpbWl6ZXIgc3RhdGUgZnJvbSBjaGVja3BvaW50LiIiIgogICAgcCA9IFBhdGgoY2hlY2twb2ludF9wYXRoKQogICAgaWYgbm90IHAuaXNfZmlsZSgpOgogICAgICAgIHJhaXNlIEZpbGVOb3RGb3VuZEVycm9yKGYiQ2hlY2twb2ludCBub3QgZm91bmQ6IHtwfSIpCiAgICBjaGVja3BvaW50ID0gdG9yY2gubG9hZChwLCBtYXBfbG9jYXRpb249ImNwdSIpCiAgICBtb2RlbC5sb2FkX3N0YXRlX2RpY3QoY2hlY2twb2ludFsibW9kZWxfc3RhdGVfZGljdCJdKQogICAgaWYgb3B0aW1pemVyIGlzIG5vdCBOb25lIGFuZCAib3B0aW1pemVyX3N0YXRlX2RpY3QiIGluIGNoZWNrcG9pbnQ6CiAgICAgICAgb3B0aW1pemVyLmxvYWRfc3RhdGVfZGljdChjaGVja3BvaW50WyJvcHRpbWl6ZXJfc3RhdGVfZGljdCJdKQogICAgcmV0dXJuIGNoZWNrcG9pbnQK'}
EXPECTED_SOURCE_TREE_SHA256 = '774fba4ba766c9353c7491516d961d597f164d3a703894f9b55044f5bb48c99c'
SOURCE_ROOT = Path("/kaggle/working/source_root")
SOURCE_ROOT.mkdir(parents=True, exist_ok=True)

digest = hashlib.sha256()
for name, b64payload in sorted(EMBEDDED_SOURCES.items()):
    payload = base64.b64decode(b64payload)
    digest.update(name.encode("utf-8"))
    digest.update(payload)
    target = SOURCE_ROOT / name
    target.parent.mkdir(parents=True, exist_ok=True)
    target.write_bytes(payload)

actual_sha = digest.hexdigest()
if actual_sha != EXPECTED_SOURCE_TREE_SHA256:
    raise RuntimeError(f"Source SHA mismatch: expected {EXPECTED_SOURCE_TREE_SHA256}, got {actual_sha}")

sys.path.insert(0, str(SOURCE_ROOT))
print("Source unpacked and verified successfully:", actual_sha)


In [ ]:
from pathlib import Path
from mpg_fer_cumulative_ablation7.model import CumulativeAblationMode, CumulativeAblationMPGFER
from mpg_fer_cumulative_ablation7.protocol import validate_all_splits, CANONICAL_DATASET_HASHES
from mpg_fer_cumulative_ablation7.train import run_cumulative_training_job

INPUT_ROOT = Path("/kaggle/input")
def find_unique_csv(filename):
    matches = sorted(INPUT_ROOT.rglob(filename))
    if len(matches) != 1:
        raise RuntimeError(f"Expected exactly 1 {filename}, found {len(matches)}: {matches}")
    return matches[0]

train_csv = find_unique_csv("train.csv")
val_csv = find_unique_csv("val.csv")
test_csv = find_unique_csv("test.csv")

print("Found dataset CSVs:")
print("  train:", train_csv)
print("  val:  ", val_csv)
print("  test: ", test_csv)

# Fail closed dataset identity verification
audit = validate_all_splits(train_csv, val_csv, test_csv)
print("Dataset validation audit PASS:", audit)


In [ ]:
# Execute end-to-end training and evaluation
result = run_cumulative_training_job(
    mode=CumulativeAblationMode(CUMULATIVE_MODE),
    run_id=RUN_ID,
    train_csv=train_csv,
    val_csv=val_csv,
    test_csv=test_csv,
    output_dir=OUTPUT_DIR,
)
print("Job completed successfully!")
print("Selected Best Epoch:", result["best_epoch"])
print("Checkpoint SHA256:", result["checkpoint_sha256"])
print("Public TTA Accuracy:", result["public_metrics"]["tta"]["accuracy"])
print("Private TTA Accuracy:", result["private_metrics"]["tta"]["accuracy"])


In [ ]:
import shutil
import json

# Write final result json at root of output
res_path = Path("/kaggle/working") / f"{CUMULATIVE_MODE}_RESULT.json"
with open(res_path, "w", encoding="utf-8") as f:
    json.dump(result, f, indent=2)
print("Wrote summary result to", res_path)
